# Full Planner — Best Strategy Simulation Notebook

This notebook is dedicated to simulating, evaluating, and discovering the **best overall strategy** for SpaceInvasion Classic across full-empire progression (economy scaling, energy optimization, research unlocks, fleet composition, raiding, colonization, and action scheduling).

**Inputs** (read-only; the notebook changes nothing in the repository):

- `state/spaceinvasion.db` — the bot's telemetry, always read through a temporary copy so the live database is never opened;
- `spaceinvasion.core.game_data` — the wiki catalog of costs, build times, production and requirements;
- `notebooks/metadata/*.json` — static descriptions and planner settings that are not part of the catalog.

**Layout**: Sections 0–1 set up packages, imports, helpers and static metadata. Sections 2–5 measure the attacks. Section 6 loads every colony, adds generic planets up to the planet limit, and builds the game-parameter tables for them. Section 7 builds the technology requirement graph, Section 8 tabulates the planet sizes and states, and Section 9 collects the highest research levels known in the universe. Section 10 prints everything that Sections 1 to 9 generated. Section 11 then searches, with a genetic algorithm, for the empire configuration that builds the most Imperial starbases per month.

## 0. Package Installation

Installs the third-party packages this notebook imports, directly or through the `spaceinvasion` package: `numpy`, `pandas` and `beautifulsoup4`. The version ranges are the ones in `requirements.txt`. When the packages are already installed, the cell changes nothing.

In [48]:
!pip install -q "numpy>=1.26,<3" "pandas>=2.2,<4" "beautifulsoup4>=4.12,<5"

## 0.1 Imports

Every import the notebook uses is in this cell. The loop before the `spaceinvasion` imports puts the repository root on `sys.path`, so the notebook also runs when it is opened from the `notebooks/` folder.

In [49]:
from collections import deque
from collections.abc import Iterable, Iterator
import contextlib
import copy
import dataclasses
import datetime
import json
import math
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
from typing import Any

import numpy as np
import pandas as pd

# Make the repository importable when the notebook is opened from `notebooks/`.
for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion.combat.fleet import DEFENSE_SLUG_BY_KEY, SHIP_SLUG_BY_KEY, _parse_coords_distance
from spaceinvasion.combat.ranking import POINTS_PER_RESOURCE_DIVISOR
from spaceinvasion.core import game_data
from spaceinvasion.core.logging_utils import redact_sensitive
from spaceinvasion.core.state import EnergyBalance, PlanetState, Resources
from spaceinvasion.db.storage import GameDatabase

## 1. Utils

Helpers shared by the sections below, grouped by purpose into one cell each (1.1–1.5 and 1.7). Cell 1.6 loads the static metadata from JSON files.

### 1.1 Workspace Paths & Shared Constants

Locates the repository and the database to read. `ACTIVE_DATA_ROOT` is the working directory when it contains its own `state/spaceinvasion.db` (isolated test fixtures); otherwise it is the repository root.

In [50]:
def resolve_repo_root() -> Path:
    """Return the directory that contains the `spaceinvasion` package, searching outward from the working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    for entry in sys.path:
        if entry and (Path(entry).resolve() / "spaceinvasion" / "__init__.py").is_file():
            return Path(entry).resolve()
    for candidate in Path(game_data.__file__).resolve().parents:
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    raise RuntimeError("Cannot locate the repository root: no directory contains spaceinvasion/__init__.py")


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer the working directory when it has its own `state/spaceinvasion.db`; otherwise use the repository."""
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "state" / "spaceinvasion.db").is_file() else repo_root


REPO_ROOT: Path = resolve_repo_root()
ACTIVE_DATA_ROOT: Path = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH: Path = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
METADATA_DIR: Path = REPO_ROOT / "notebooks" / "metadata"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

print(f"REPO_ROOT        = {REPO_ROOT}")
print(f"ACTIVE_DATA_ROOT = {ACTIVE_DATA_ROOT}")
print(f"METADATA_DIR     = {METADATA_DIR}")

REPO_ROOT        = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer
ACTIVE_DATA_ROOT = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer
METADATA_DIR     = /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata


### 1.2 Formatting, Statistics & DataFrame Helpers

Display options and three small helpers: `fmt_duration` turns seconds into text, `compute_mean_and_std` returns the mean with the sample and population standard deviations (`numpy`), and `to_float_columns` gives numeric columns a float type even when a table is empty.

In [51]:
pd.set_option("display.max_columns", 28)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 64)


def fmt_duration(seconds: float) -> str:
    """Format seconds as `45s`, `3m 07s`, `5h 35m`, or `12d 04h`."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    if hrs < 48:
        return f"{hrs}h {rem_m:02d}m"
    days, rem_h = divmod(hrs, 24)
    return f"{days}d {rem_h:02d}h"


def compute_mean_and_std(values: Any) -> tuple[float, float, float]:
    """Return `(mean, sample std with ddof=1, population std with ddof=0)`; non-finite values are ignored."""
    arr = np.asarray(values, dtype=np.float64).ravel()
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return 0.0, 0.0, 0.0
    mean_val = float(np.mean(arr))
    sample_std = float(np.std(arr, ddof=1)) if arr.size >= 2 else 0.0
    pop_std = float(np.std(arr, ddof=0))
    return mean_val, sample_std, pop_std


def to_float_columns(frame: pd.DataFrame, columns: tuple[str, ...]) -> pd.DataFrame:
    """Convert `columns` to float64 (missing or non-numeric values become 0.0), so empty tables are typed too."""
    for column in columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0.0).astype(np.float64)
    return frame

### 1.3 Game Catalog Helpers & Formulas

Helpers that read `game_data` in a fixed order (`research_catalog`, `unit_keys`), normalize costs, format requirements, and repair the build times that the wiki truncates at very high levels (`build_monotonic_base_times`). The game's time formulas are here too:

- buildings: `base_time / ((b7 + 1) * 2^b17 * speed)`
- research: `base_time / ((b8 + 1) * speed)`
- ships and defenses: `base_time / (max(1, b9) * 2^b17 * speed)`

where `b7` is the Development center, `b8` the Research lab, `b9` the Weapons factory, `b17` the Microsystem accelerator and `speed` the universe speed factor.

`warehouse_capacity` is the storage formula `100,000 * 2^level`, and `solar_cell_energy_per_unit` reads the catalog's Solar cell temperature table.

In [52]:
def research_catalog() -> list[tuple[str, dict[str, Any]]]:
    """Researches as `(key, info)` in numeric key order (`r1`, `r2`, ...); wiki entries without an id are left out."""
    return sorted(game_data.RESEARCH_BY_ID.items(), key=lambda entry: int(entry[0][1:]))


def unit_keys(slug_by_key: dict[str, str]) -> list[str]:
    """Ship or defense keys (`s1`, `d1`, ...) in numeric order."""
    return sorted(slug_by_key, key=lambda key: int(key[1:]))


def full_cost(raw_cost: Any) -> dict[str, Any]:
    """Copy a cost mapping, adding a zero for any of the four resources it leaves out."""
    cost = dict(raw_cost or {})
    for resource in RESOURCE_KEYS:
        cost.setdefault(resource, 0)
    return cost


def cost_points(cost: dict[str, Any]) -> float:
    """Ranking points earned by spending `cost`: one point per 1,000 resources."""
    return float(sum(int(cost[resource]) for resource in RESOURCE_KEYS)) / float(POINTS_PER_RESOURCE_DIVISOR)


def format_prerequisites(requirements: list[dict[str, Any]] | tuple[dict[str, Any], ...]) -> str:
    """Format tech-tree requirements as `r2 L1 (Energy technology), b7 L3 (Development center)`."""
    if not requirements:
        return "None"
    parts: list[str] = []
    for requirement in requirements:
        category = str(requirement.get("category") or "")
        slug = str(requirement.get("slug") or "")
        level = int(requirement.get("level") or 0)
        if category == "buildings":
            building_id = (game_data.BUILDINGS.get(slug) or {}).get("id")
            code = f"b{building_id}" if building_id is not None else slug
        elif category == "research":
            research_id = (game_data.RESEARCHES.get(slug) or {}).get("id")
            code = str(research_id) if research_id else slug
        else:
            code = slug or str(requirement.get("name") or "?")
        parts.append(f"{code} L{level} ({requirement.get('name', slug)})")
    return ", ".join(parts)


def build_monotonic_base_times(levels_map: dict[int, dict[str, Any]], max_level: int) -> dict[int, float]:
    """Return `{level: base_time_s}`, repairing levels where the wiki's scientific notation lost its exponent.

    At very high levels the scraped time drops to a bare mantissa (for example 1.37 s after 9e13 s). Such a
    level is rebuilt from the previous one, scaled by the same ratio as its Spice-equivalent cost.
    """
    times: dict[int, float] = {}
    previous_time = 0.0
    previous_cost = 0.0
    for level in range(1, max_level + 1):
        row = levels_map.get(level) or {}
        raw_time = float(row.get("base_time_s") or 0.0)
        cost = float(game_data.spice_equivalent(row.get("cost") or {}))
        if level > 1 and previous_time > 0.0 and raw_time < previous_time:
            ratio = (cost / previous_cost) if previous_cost > 0.0 else 1.51
            fixed_time = previous_time * max(1.05, ratio)
        else:
            fixed_time = max(0.0, raw_time)
        times[level] = fixed_time
        previous_time = fixed_time
        previous_cost = cost
    return times


def building_time_divisor(dev_center_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a building's base time: `(b7 + 1) * 2^b17 * speed`, never below 1."""
    return max(1.0, float(dev_center_level + 1) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor))


def research_time_divisor(research_lab_level: int, speedup_factor: float) -> float:
    """Divisor of a research's base time: `(b8 + 1) * speed`, never below 1."""
    return max(1.0, float(research_lab_level + 1) * float(speedup_factor))


def shipyard_time_divisor(weapons_factory_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a ship's or defense's base time: `max(1, b9) * 2^b17 * speed`, never below 1."""
    return max(
        1.0,
        float(max(1, weapons_factory_level)) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor),
    )


def warehouse_capacity(level: int) -> int:
    """Storage capacity of a warehouse at `level`: `100,000 * 2^level` (level 0 is every planet's base capacity)."""
    return int(100_000 * (2 ** max(0, int(level))))


def solar_cell_energy_per_unit(temperature_c: float) -> float:
    """Base energy of one orbital Solar cell (`s9`), interpolated from the catalog's temperature table."""
    energy_by_temperature = game_data.SHIPS[SHIP_SLUG_BY_KEY["s9"]]["solar_cell_energy_by_temp"]
    return float(game_data.interpolate_temp_value(energy_by_temperature, temperature_c))

### 1.4 Read-Only Database Access & JSON Parsing

`open_readonly_db_copy` is the only way this notebook touches SQLite. It copies the database (with its WAL sidecars) into a temporary directory, lets `GameDatabase` bring the copy to the current schema, and yields a query-only connection to the copy. The live file is never opened, locked or modified.

`read_latest_game_state` returns the bot's latest full game state, which the telemetry loader of Section 2 and the roster loader of Section 1.7 both read. `parse_json_object` and `parse_level_map` decode the JSON text columns.

In [53]:
def parse_json_object(raw: Any) -> dict[str, Any]:
    """Return `raw` as a dict when it is a dict or the JSON text of an object; otherwise `{}`."""
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
        except ValueError:
            return {}
        return dict(parsed) if isinstance(parsed, dict) else {}
    return {}


def parse_level_map(raw: Any) -> dict[str, int]:
    """Parse a `{key: level or count}` object into non-negative ints, dropping non-numeric values."""
    return {
        str(key): max(0, int(value or 0))
        for key, value in parse_json_object(raw).items()
        if isinstance(value, (int, float))
    }


def has_database(db_path: Path) -> bool:
    """True when `db_path` is a file large enough to hold a SQLite header."""
    return db_path.is_file() and db_path.stat().st_size >= 100


@contextlib.contextmanager
def open_readonly_db_copy(db_path: Path) -> Iterator[sqlite3.Connection]:
    """Yield a query-only connection to a temporary copy of `db_path`, migrated to the current schema.

    The live database is only copied (with its WAL sidecars), never opened, so the notebook cannot lock
    or change it. Opening the copy once through `GameDatabase` applies any pending forward-only
    migrations, which lets the notebook's SQL rely on the current schema.
    """
    with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
        tmp_db = Path(tmp_dir) / db_path.name
        shutil.copy2(db_path, tmp_db)
        tmp_db.chmod(0o600)
        for suffix in ("-wal", "-shm", "-journal"):
            sidecar = Path(f"{db_path}{suffix}")
            if sidecar.is_file():
                tmp_sidecar = Path(f"{tmp_db}{suffix}")
                shutil.copy2(sidecar, tmp_sidecar)
                tmp_sidecar.chmod(0o600)
        GameDatabase(tmp_db, backup_on_upgrade=False).close()
        with contextlib.closing(sqlite3.connect(tmp_db)) as conn:
            conn.row_factory = sqlite3.Row
            conn.execute("PRAGMA query_only = ON")
            yield conn


SQL_LATEST_GAME_STATE_META = """
SELECT value
FROM meta
WHERE key = 'latest_game_state'
LIMIT 1
"""


def read_latest_game_state(conn: sqlite3.Connection) -> dict[str, Any]:
    """Return the bot's most recent full game state (`meta['latest_game_state']`), or `{}` when it is absent."""
    row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
    return parse_json_object(row[0]) if row and row[0] else {}

### 1.5 Full-Data Printing

`print_section_data` prints every row, column and element of the objects it is given, under a header with the date and time of the printout. Section 10 uses it to print everything that Sections 1 to 9 generate, and every cell of Section 11 prints its own data with it. Floats of 1 or more are never switched to scientific notation, and all text passes through `redact_sensitive`.

In [54]:
def format_full_float(value: float) -> str:
    """Format a float for full printouts without scientific notation for large values.

    `|x| >= 1` keeps its shortest exact decimal text, capped at 6 decimals; `|x| < 1` keeps 6 significant digits.
    """
    val = float(value)
    if not math.isfinite(val):
        return str(val)
    if val == 0.0:
        return "0"
    magnitude = abs(val)
    if magnitude < 1.0:
        return f"{val:.6g}"
    if magnitude >= 1e16:
        return f"{val:.0f}"
    whole, _, frac = repr(val).partition(".")
    if len(frac) > 6:
        return f"{val:.6f}".rstrip("0").rstrip(".")
    return whole if frac in ("", "0") else f"{whole}.{frac}"


def _float_column_formatter(column: pd.Series) -> Any:
    """Return one formatter for a float column so that its decimal points line up."""
    texts = [format_full_float(v) for v in column.to_numpy(dtype=np.float64) if math.isfinite(v)]
    if any("e" in text for text in texts):
        return format_full_float
    decimals = max((len(text.partition(".")[2]) for text in texts), default=0)
    return lambda v: f"{float(v) + 0.0:.{decimals}f}"


def _json_default(obj: Any) -> Any:
    """Convert numpy values, dataclasses, and sets for `json.dumps`; fall back to `str`."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return dataclasses.asdict(obj)
    if isinstance(obj, (set, frozenset)):
        return sorted(obj, key=str)
    return str(obj)


def _compact_json(value: Any) -> str:
    """Single-line JSON text for one value (falls back to `repr` for unserializable objects)."""
    try:
        return json.dumps(value, default=_json_default)
    except (TypeError, ValueError):
        return repr(value)


def describe_data_object(obj: Any) -> str:
    """One-line type and size summary of a generated object."""
    if isinstance(obj, pd.DataFrame):
        return f"pandas.DataFrame, shape=({obj.shape[0]}, {obj.shape[1]})"
    if isinstance(obj, np.ndarray):
        return f"numpy.ndarray, shape={obj.shape}, dtype={obj.dtype}"
    if isinstance(obj, (dict, list, tuple, set, frozenset)):
        return f"{type(obj).__name__}, len={len(obj)}"
    return type(obj).__name__


def render_data_item(name: str, obj: Any) -> str:
    """Render one named object in full: every row, column, element, and cell text, with no `...` truncation."""
    head = f"[{name}] {describe_data_object(obj)}"
    if isinstance(obj, pd.DataFrame):
        if obj.empty:
            return f"{head}\n(no rows) columns: {list(obj.columns)}"
        formatters: dict[Any, Any] = {}
        if obj.columns.is_unique:
            formatters = {
                col: _float_column_formatter(obj[col])
                for col in obj.columns
                if pd.api.types.is_float_dtype(obj[col])
            }
        show_index = not isinstance(obj.index, pd.RangeIndex)
        if show_index and not isinstance(obj.index, pd.MultiIndex) and pd.api.types.is_float_dtype(obj.index):
            formatters["__index__"] = format_full_float
        table = obj.to_string(
            index=show_index,
            formatters=formatters or None,
            float_format=format_full_float,
        )
        return f"{head}\n{table}"
    if isinstance(obj, np.ndarray):
        body = np.array2string(
            obj,
            threshold=obj.size + 1,
            max_line_width=160,
            formatter={"float_kind": format_full_float},
        )
        return f"{head}\n{body}"
    if isinstance(obj, dict):
        if not obj:
            return f"{head}\n{{}}"
        if all(isinstance(v, pd.DataFrame) for v in obj.values()):
            note = "registry of DataFrames; each table is printed in full by its own section"
            rows = [f"  {key}: shape=({tbl.shape[0]}, {tbl.shape[1]})" for key, tbl in obj.items()]
            return f"{head} ({note})\n" + "\n".join(rows)
        rows = [f"  {json.dumps(str(key))}: {_compact_json(value)}" for key, value in obj.items()]
        return f"{head}\n{{\n" + ",\n".join(rows) + "\n}"
    if isinstance(obj, (list, tuple)):
        if all(v is None or isinstance(v, (str, int, float, bool)) for v in obj):
            return f"{head}\n{_compact_json(list(obj))}"
        return f"{head} (one JSON value per line)\n" + "\n".join(_compact_json(v) for v in obj)
    return f"{head} = {obj!r}"


def print_section_data(section_title: str, items: list[tuple[str, Any]]) -> None:
    """Print everything one section generated, in full, stamped with the date and time of the printout.

    Each item is `(name, obj)`. The whole text passes through `redact_sensitive` before printing, so a
    session token can never reach a notebook output.
    """
    printed_at = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
    rule = "=" * 100
    header = f"{rule}\n{section_title}\nPrinted: {printed_at} | {len(items)} item(s)\n{rule}"
    blocks = [render_data_item(name, obj) for name, obj in items]
    print(redact_sensitive(header + "\n" + ("\n" + "-" * 100 + "\n").join(blocks)))

### 1.6 Static Metadata (`notebooks/metadata/*.json`)

Static data is kept in JSON files instead of code cells:

| File | Content |
|---|---|
| `buildings.json` | category, nominal cost scale factor and primary effect of `b1`–`b21` |
| `technologies.json` | domain, effect, per-level bonus and scope of `r1`–`r15` |
| `ships.json` | role of each ship |
| `defenses.json` | role of each defense and whether it can dig in |
| `resources.json` | mine and warehouse building of each resource, Mining technology flag, notes |
| `planner_settings.json` | planner settings and fallback assumptions: table level caps, empire size, cold-start colony, flight-time fallback |

The per-entity files repeat each entity's name. Loading stops with an error when a file misses an entity, has an unknown one, lacks a field, or names an entity differently from `game_data`, so a description can never end up on the wrong entity.

`technology_multiplier` turns a research level into its multiplier, using the `per_level_bonus_pct` of `technologies.json`.

In [55]:
def load_metadata(file_name: str) -> dict[str, Any]:
    """Read one JSON document from `notebooks/metadata/`."""
    return json.loads((METADATA_DIR / file_name).read_text(encoding="utf-8"))


def load_keyed_metadata(
    file_name: str,
    section: str,
    expected_keys: Iterable[str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load `{key: entry}` metadata; it must cover exactly `expected_keys`, each entry with every required field."""
    entries = load_metadata(file_name)[section]
    expected = set(expected_keys)
    if set(entries) != expected:
        raise ValueError(
            f"{file_name}: missing {sorted(expected - set(entries))}, unknown {sorted(set(entries) - expected)}"
        )
    for key, entry in entries.items():
        absent = [field for field in required_fields if field not in entry]
        if absent:
            raise ValueError(f"{file_name}: {key} lacks {absent}")
    return entries


def load_catalog_metadata(
    file_name: str,
    section: str,
    catalog_names: dict[str, str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load per-entity metadata and check each entry's `name` against the game catalog (`{key: name}`)."""
    entries = load_keyed_metadata(file_name, section, catalog_names, ("name", *required_fields))
    for key, entry in entries.items():
        if entry["name"] != catalog_names[key]:
            raise ValueError(
                f"{file_name}: {key} is named {entry['name']!r}, but game_data calls it {catalog_names[key]!r}"
            )
    return entries


BUILDING_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "buildings.json",
    "buildings",
    {f"b{building_id}": str(info["name"]) for building_id, info in game_data.BUILDING_BY_ID.items()},
    ("category", "cost_scale_factor", "primary_effect"),
)
TECHNOLOGY_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "technologies.json",
    "technologies",
    {key: str(info["name"]) for key, info in research_catalog()},
    ("domain", "effect_metric", "per_level_bonus_pct", "applies_to", "strategic_effect"),
)
TECHNOLOGY_COST_SCALE_FACTOR: float = float(load_metadata("technologies.json")["cost_scale_factor"])
SHIP_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "ships.json",
    "ships",
    {key: str(game_data.SHIPS[slug]["name"]) for key, slug in SHIP_SLUG_BY_KEY.items()},
    ("role",),
)
DEFENSE_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "defenses.json",
    "defenses",
    {key: str(game_data.DEFENSES[slug]["name"]) for key, slug in DEFENSE_SLUG_BY_KEY.items()},
    ("role", "can_dig_in"),
)
RESOURCE_METADATA: dict[str, dict[str, Any]] = load_keyed_metadata(
    "resources.json",
    "resources",
    RESOURCE_KEYS,
    ("mine_building_key", "warehouse_building_key", "boosted_by_mining_tech", "notes"),
)
PLANNER_SETTINGS: dict[str, Any] = load_metadata("planner_settings.json")


def technology_multiplier(research_key: str, level: int) -> float:
    """Multiplier a research gives at `level`: `1 + per_level_bonus_pct / 100 * level` (1.0 for an unknown key)."""
    bonus_per_level = float(TECHNOLOGY_METADATA.get(research_key, {}).get("per_level_bonus_pct", 0.0)) / 100.0
    return 1.0 + bonus_per_level * float(level)


print(
    f"Static metadata loaded from {METADATA_DIR}: {len(BUILDING_METADATA)} buildings, "
    f"{len(TECHNOLOGY_METADATA)} technologies, {len(SHIP_METADATA)} ships, {len(DEFENSE_METADATA)} defenses, "
    f"{len(RESOURCE_METADATA)} resources, and the planner settings"
)

Static metadata loaded from /usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata: 21 buildings, 15 technologies, 15 ships, 11 defenses, 4 resources, and the planner settings


### 1.7 Planet Roster Helpers

`load_live_planet_states` reads every owned planet from the temporary database copy. It prefers the bot's latest full game state, which records the real warehouse capacities. Without one, it takes the newest `planet_snapshots` row of each planet and derives the capacities from the warehouse levels.

`build_padded_empire_planets` appends generic planets to the owned ones until the empire has the given number of normal planets. Asteroids are kept but do not count toward that limit. A generic planet (`empty_colony_slot`) is an empty colony slot: a planet with the given number of free fields, no resources, level-0 warehouses and nothing built. Its `planet_id` is negative (`-1`, `-2`, …) and its coordinates are `EMPTY:01`, `EMPTY:02`, ….

Sections 6 and 8 work on this roster.

In [56]:
SQL_LATEST_PLANETS_PER_ID = """
SELECT
    p.planet_id,
    p.coords,
    p.name,
    p.planet_type,
    p.temperature,
    p.fields_used,
    p.fields_max,
    p.pig_iron,
    p.metal,
    p.kryptonite,
    p.spice,
    p.energy_produced,
    p.energy_used,
    p.buildings_json,
    p.ships_json,
    p.defense_json,
    p.economy_rates_json
FROM planet_snapshots p
INNER JOIN (
    SELECT planet_id, MAX(id) AS max_id
    FROM planet_snapshots
    GROUP BY planet_id
) latest ON latest.max_id = p.id
ORDER BY p.planet_id ASC
"""


def snapshot_row_to_planet_fields(row: sqlite3.Row) -> dict[str, Any]:
    """Reshape a `planet_snapshots` row into the nested layout the saved game state uses for a planet."""
    return {
        "planet_id": row["planet_id"],
        "name": row["name"],
        "coords": row["coords"],
        "planet_type": row["planet_type"],
        "temperature": row["temperature"],
        "fields_used": row["fields_used"],
        "fields_max": row["fields_max"],
        "resources": {resource: row[resource] for resource in RESOURCE_KEYS},
        "energy": {"produced": row["energy_produced"], "used": row["energy_used"]},
        "buildings": row["buildings_json"],
        "ships": row["ships_json"],
        "defense": row["defense_json"],
        "economy_rates": row["economy_rates_json"],
    }


def planet_state_from_fields(fields: dict[str, Any]) -> PlanetState:
    """Build a typed `PlanetState` from one planet's saved fields.

    Gaps are filled from the building levels: used fields default to the sum of all levels, and a missing
    warehouse capacity to `warehouse_capacity(level)`. A missing size defaults to `fallback_planet_fields`.
    """
    buildings = parse_level_map(fields.get("buildings"))
    resources = parse_json_object(fields.get("resources"))
    storage = parse_json_object(fields.get("storage"))
    energy = parse_json_object(fields.get("energy"))
    fields_used = int(fields.get("fields_used") or sum(buildings.values()))
    fallback_fields = int(PLANNER_SETTINGS["empire"]["fallback_planet_fields"])
    return PlanetState(
        planet_id=int(fields.get("planet_id") or 0),
        name=str(fields.get("name") or "Colony"),
        coords=str(fields.get("coords") or "00:000:00"),
        planet_type=str(fields.get("planet_type") or "normal"),
        temperature=int(fields.get("temperature") or 0),
        fields_used=fields_used,
        fields_max=int(fields.get("fields_max") or max(fallback_fields, fields_used)),
        resources=Resources(**{resource: int(resources.get(resource) or 0) for resource in RESOURCE_KEYS}),
        storage=Resources(
            **{
                resource: int(
                    storage.get(resource)
                    or warehouse_capacity(buildings.get(RESOURCE_METADATA[resource]["warehouse_building_key"], 0))
                )
                for resource in RESOURCE_KEYS
            }
        ),
        energy=EnergyBalance(produced=int(energy.get("produced") or 0), used=int(energy.get("used") or 0)),
        buildings=buildings,
        ships=parse_level_map(fields.get("ships")),
        defense=parse_level_map(fields.get("defense")),
        economy_rates=parse_level_map(fields.get("economy_rates")),
    )


def load_live_planet_states(db_path: Path) -> list[PlanetState]:
    """Load every owned planet from a temporary copy of `db_path`.

    The bot's latest full game state is preferred because it records the real warehouse capacities. Without
    it, the newest `planet_snapshots` row of each planet is used and capacities come from the warehouse levels.
    """
    if not has_database(db_path):
        return []
    with open_readonly_db_copy(db_path) as conn:
        saved_planets = read_latest_game_state(conn).get("planets") or []
        if isinstance(saved_planets, list) and saved_planets:
            return [planet_state_from_fields(planet) for planet in saved_planets if isinstance(planet, dict)]
        return [
            planet_state_from_fields(snapshot_row_to_planet_fields(row))
            for row in conn.execute(SQL_LATEST_PLANETS_PER_ID)
        ]


def empty_colony_slot(slot_number: int, fields: int, temperature_c: int) -> PlanetState:
    """A generic planet that is not colonized yet: `fields` free fields, base warehouses, nothing built."""
    base_storage = warehouse_capacity(0)
    return PlanetState(
        planet_id=-slot_number,
        name=f"Empty Colony Slot {slot_number}",
        coords=f"EMPTY:{slot_number:02d}",
        planet_type="normal",
        temperature=int(temperature_c),
        fields_used=0,
        fields_max=int(fields),
        resources=Resources(0, 0, 0, 0),
        storage=Resources(base_storage, base_storage, base_storage, base_storage),
        energy=EnergyBalance(0, 0),
        buildings={f"b{building_id}": 0 for building_id in sorted(game_data.BUILDING_BY_ID)},
        ships={},
        defense={},
        economy_rates={},
    )


def build_padded_empire_planets(
    live_planets: list[PlanetState],
    *,
    max_planets: int,
    empty_planet_size: int,
    empty_temp_c: int,
) -> list[PlanetState]:
    """Return the owned planets followed by generic planets (empty colony slots).

    Slots are added until the empire has `max_planets` normal planets; asteroids and other planet types are
    kept but do not count toward that limit.
    """
    owned_normal_planets = sum(1 for planet in live_planets if planet.planet_type == "normal")
    return [
        *live_planets,
        *(
            empty_colony_slot(slot_number, empty_planet_size, empty_temp_c)
            for slot_number in range(1, max(0, int(max_planets) - owned_normal_planets) + 1)
        ),
    ]

## 2. Attack Telemetry from SQLite

Loads the reference colony and the attack history from `state/spaceinvasion.db`. The tables and their relationships are documented in [docs/database-mer.md](../docs/database-mer.md). Every query runs on the temporary copy opened by `open_readonly_db_copy`.

### 2.1 SQL Queries & Table Columns

| Query | Tables | Returns |
|---|---|---|
| `SQL_LATEST_COLONY_SNAPSHOT` | `planet_snapshots` | coordinates, building and research levels of the most recently snapshotted planet |
| `SQL_BATTLE_REPORTS_MER_JOIN` | `battle_reports`, `game_messages`, `spy_targets` | loot, losses and rubble of every battle with the target's status; `spy_targets` rows for own planets and merged targets are left out |
| `SQL_RAID_DISPATCH_ORIGIN_JOIN` | `raid_history`, `action_history` | every raid dispatch with the colony that launched it: the executed `ATTACK` action within 5 s of the dispatch, or the query parameter when there is none |
| `SQL_RECYCLE_HARVESTS` | `recycle_reports` | recycler harvests |
| `SQL_ACTIVE_BASH_BLOCKS` | `bash_blocks` | number of targets still bash-blocked at the given time |

The `*_COLUMNS` tuples list the columns of `attacks_df` and `recycles_df` and which of them are numeric.

In [57]:
SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""

SQL_BATTLE_REPORTS_MER_JOIN = """
SELECT
    br.id,
    br.ts,
    br.msg_uid,
    br.battle_id,
    br.msg_date_text,
    br.target_coords,
    br.outcome,
    br.rounds,
    br.pig_iron,
    br.metal,
    br.kryptonite,
    br.spice,
    br.loot_spice_eq,
    br.attacker_lost_spice,
    br.defender_lost_spice,
    br.debris_spice_eq,
    gm.ingested_ts AS msg_ingested_ts,
    gm.deleted_in_game,
    COALESCE(st.status_color, 'unknown') AS target_status_color,
    COALESCE(st.is_inactive, 0) AS target_is_inactive,
    COALESCE(st.is_spacia, 0) AS target_is_spacia,
    COALESCE(st.bash_hits, 0) AS target_bash_hits
FROM battle_reports br
LEFT JOIN game_messages gm ON gm.msg_uid = br.msg_uid
LEFT JOIN spy_targets st
       ON st.coords = br.target_coords
      AND st.is_own_planet = 0
      AND st.merged_into = ''
ORDER BY br.msg_date_text ASC, br.id ASC
"""

# The `BETWEEN` range (instead of `ABS(ah.ts - rh.attacked_ts) <= 5.0`) lets SQLite use `idx_action_history_ts`.
SQL_RAID_DISPATCH_ORIGIN_JOIN = """
SELECT
    rh.id,
    rh.target_coords,
    rh.attacked_ts,
    rh.is_exempt,
    rh.outcome,
    rh.loot_json,
    COALESCE(
        (
            SELECT ah.planet_coords
            FROM action_history ah
            WHERE ah.ts BETWEEN rh.attacked_ts - 5.0 AND rh.attacked_ts + 5.0
              AND ah.kind = 'ATTACK'
              AND ah.executed = 1
              AND ah.dry_run = 0
              AND ah.planet_coords IS NOT NULL
              AND ah.planet_coords != ''
            ORDER BY ABS(ah.ts - rh.attacked_ts) ASC
            LIMIT 1
        ),
        ?
    ) AS origin_coords
FROM raid_history rh
ORDER BY rh.attacked_ts ASC, rh.id ASC
"""

SQL_RECYCLE_HARVESTS = """
SELECT
    coords,
    recycler_count,
    harvested_pig_iron,
    harvested_metal,
    harvested_kryptonite,
    harvested_spice,
    harvested_spice_eq,
    ts
FROM recycle_reports
ORDER BY ts ASC, id ASC
"""

SQL_ACTIVE_BASH_BLOCKS = """
SELECT COUNT(DISTINCT target_coords)
FROM bash_blocks
WHERE blocked_until_ts > ?
"""

ATTACK_DF_COLUMNS: tuple[str, ...] = (
    "event_ts",
    "origin_coords",
    "target_coords",
    "target_class",
    "won",
    "outcome",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
    "time_source",
)
ATTACK_FLOAT_COLUMNS: tuple[str, ...] = (
    "event_ts",
    "pig_iron",
    "metal",
    "kryptonite",
    "spice",
    "spice_eq",
    "attacker_lost_spice",
    "net_spice_eq",
    "debris_spice_eq",
    "one_way_sec",
    "round_trip_sec",
)

RECYCLE_DF_COLUMNS: tuple[str, ...] = (
    "coords",
    "recycler_count",
    "harvested_pig_iron",
    "harvested_metal",
    "harvested_kryptonite",
    "harvested_spice",
    "harvested_spice_eq",
    "ts",
)
RECYCLE_FLOAT_COLUMNS: tuple[str, ...] = tuple(column for column in RECYCLE_DF_COLUMNS if column != "coords")

### 2.2 Telemetry Loader

`load_attack_telemetry` returns the reference colony and one record per attack. Dispatches and battle reports are timed differently: a raid dispatch is stored in epoch seconds, a battle report carries the server's wall-clock text without a time zone. The loader reconciles them in three steps:

1. `infer_server_tz_offset_sec` finds the whole-hour offset between the two clocks (the median over all report/dispatch pairs on the same target).
2. `attack_records_from_battles` pairs each battle report with its dispatch. The flight time is the observed gap between dispatch and impact (`observed_dispatch_pair`). A report without a dispatch gets the median observed seconds per square root of distance, applied to its own distance (`calibrated_distance`). When no pair was observed at all, the rough estimate `estimate_physics_one_way_seconds` is used (`physics_formula`).
3. `attack_records_from_raids` covers a database that has dispatches but no battle report yet: one record per dispatch, timed by the same rough estimate.

In [58]:
def estimate_physics_one_way_seconds(origin_coords: str, target_coords: str) -> float:
    """Rough OGame-style one-way flight time in seconds: `10 + 3500 / factor * sqrt(10 * distance / speed)`.

    Only a fallback for attacks without an observed flight time. `speed` and `factor` are the assumptions
    under `flight_time_fallback` in `planner_settings.json`; they are not measured from the game.
    """
    assumptions = PLANNER_SETTINGS["flight_time_fallback"]
    distance = max(1.0, float(_parse_coords_distance(origin_coords, target_coords)))
    speed = max(100.0, float(assumptions["min_ship_speed"]))
    factor = min(100.0, max(1.0, float(assumptions["fleet_speed_factor"])))
    return float(round(10.0 + (3500.0 / factor) * math.sqrt((10.0 * distance) / speed)))


def parse_server_clock(text: Any) -> float | None:
    """Read a battle report's `YYYY-MM-DD HH:MM:SS` server clock text as if it were UTC (`None` when malformed)."""
    try:
        parsed = datetime.datetime.strptime(str(text or "").strip(), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None
    return parsed.replace(tzinfo=datetime.timezone.utc).timestamp()


def infer_server_tz_offset_sec(
    battle_rows: list[dict[str, Any]],
    server_clock_ts: list[float | None],
    raid_rows: list[dict[str, Any]],
) -> float:
    """Estimate the offset, in seconds, to add to a battle report's server clock to get epoch time.

    Every report/dispatch pair on the same target votes for the whole-hour offset (at most 14 h) that puts
    the impact 30 s to 1 h after the dispatch. The median vote wins; without votes the offset is 0.
    """
    votes: list[float] = []
    for battle, clock_ts in zip(battle_rows, server_clock_ts):
        if clock_ts is None:
            continue
        for raid in raid_rows:
            if str(raid.get("target_coords") or "") != str(battle.get("target_coords") or ""):
                continue
            dispatch_ts = float(raid.get("attacked_ts") or 0.0)
            offset_hours = round((dispatch_ts - clock_ts) / 3600.0)
            if abs(offset_hours) <= 14 and 30.0 <= (clock_ts + offset_hours * 3600.0 - dispatch_ts) <= 3600.0:
                votes.append(offset_hours * 3600.0)
    return float(np.median(np.asarray(votes, dtype=np.float64))) if votes else 0.0


def attack_record(
    *,
    event_ts: float,
    origin_coords: str,
    target_coords: str,
    target_class: str,
    won: bool,
    outcome: str,
    loot: dict[str, float],
    spice_eq: float,
    attacker_lost_spice: float,
    debris_spice_eq: float,
    one_way_sec: float,
    time_source: str,
) -> dict[str, Any]:
    """Assemble one `attacks_df` row (`loot` holds the four resources; net yield and round trip are derived)."""
    return {
        "event_ts": event_ts,
        "origin_coords": origin_coords,
        "target_coords": target_coords,
        "target_class": target_class,
        "won": won,
        "outcome": outcome,
        **loot,
        "spice_eq": spice_eq,
        "attacker_lost_spice": attacker_lost_spice,
        "net_spice_eq": spice_eq - attacker_lost_spice,
        "debris_spice_eq": debris_spice_eq,
        "one_way_sec": float(one_way_sec),
        "round_trip_sec": float(2.0 * one_way_sec),
        "time_source": time_source,
    }


def attack_records_from_battles(
    battle_rows: list[dict[str, Any]],
    server_clock_ts: list[float | None],
    raid_rows: list[dict[str, Any]],
    default_origin: str,
    tz_offset_sec: float,
) -> list[dict[str, Any]]:
    """Build one record per battle report, timing each attack by pairing the report with its raid dispatch."""
    used_raids: set[int] = set()
    seconds_per_sqrt_distance: list[float] = []
    paired: list[tuple[dict[str, Any], float, float | None, str]] = []
    for battle, clock_ts in zip(battle_rows, server_clock_ts):
        target = str(battle.get("target_coords") or "")
        impact_ts = (clock_ts + tz_offset_sec) if clock_ts is not None else float(battle.get("ts") or 0.0)
        flight_sec: float | None = None
        origin = default_origin
        paired_raid: int | None = None
        for raid_index, raid in enumerate(raid_rows):
            if raid_index in used_raids or str(raid.get("target_coords") or "") != target:
                continue
            gap = impact_ts - float(raid.get("attacked_ts") or 0.0)
            # Keep the latest unused dispatch that left 30 s to 2 h before the impact.
            if 30.0 <= gap <= 7200.0 and (flight_sec is None or gap < flight_sec):
                flight_sec = gap
                origin = str(raid.get("origin_coords") or default_origin)
                paired_raid = raid_index
        if paired_raid is not None and flight_sec is not None:
            used_raids.add(paired_raid)
            distance = max(1.0, float(_parse_coords_distance(origin, target)))
            seconds_per_sqrt_distance.append(max(1.0, flight_sec - 10.0) / math.sqrt(distance))
        paired.append((battle, impact_ts, flight_sec, origin))

    calibration = (
        float(np.median(np.asarray(seconds_per_sqrt_distance, dtype=np.float64)))
        if seconds_per_sqrt_distance
        else None
    )
    records: list[dict[str, Any]] = []
    for battle, impact_ts, flight_sec, origin in paired:
        target = str(battle.get("target_coords") or "")
        if flight_sec is not None:
            one_way_sec, time_source = flight_sec, "observed_dispatch_pair"
        elif calibration is not None:
            distance = max(1.0, float(_parse_coords_distance(origin, target)))
            one_way_sec, time_source = 10.0 + calibration * math.sqrt(distance), "calibrated_distance"
        else:
            one_way_sec, time_source = estimate_physics_one_way_seconds(origin, target), "physics_formula"

        outcome = str(battle.get("outcome") or "").strip().lower()
        won = outcome in ("win", "victory")
        if battle.get("target_is_spacia"):
            target_class = "spacia"
        elif battle.get("target_is_inactive") or str(battle.get("target_status_color") or "") == "yellow":
            target_class = "yellow_inactive"
        else:
            target_class = "active_colony"
        records.append(
            attack_record(
                event_ts=impact_ts,
                origin_coords=origin,
                target_coords=target,
                target_class=target_class,
                won=won,
                outcome=outcome or ("win" if won else "loss"),
                loot={resource: float(battle.get(resource) or 0.0) for resource in RESOURCE_KEYS},
                spice_eq=float(battle["loot_spice_eq"]),
                attacker_lost_spice=float(battle.get("attacker_lost_spice") or 0.0),
                debris_spice_eq=float(battle.get("debris_spice_eq") or 0.0),
                one_way_sec=one_way_sec,
                time_source=time_source,
            )
        )
    return records


def attack_records_from_raids(raid_rows: list[dict[str, Any]], default_origin: str) -> list[dict[str, Any]]:
    """Build one record per raid dispatch (used while no battle report exists), timed by the physics estimate."""
    records: list[dict[str, Any]] = []
    for raid in raid_rows:
        target = str(raid.get("target_coords") or "")
        origin = str(raid.get("origin_coords") or default_origin)
        raw_loot = parse_json_object(raid.get("loot_json"))
        loot = {resource: float(raw_loot.get(resource) or 0.0) for resource in RESOURCE_KEYS}
        outcome = str(raid.get("outcome") or "").lower()
        won = "defeat" not in outcome and "loss" not in outcome
        records.append(
            attack_record(
                event_ts=float(raid.get("attacked_ts") or 0.0),
                origin_coords=origin,
                target_coords=target,
                target_class="exempt_farm" if raid.get("is_exempt") else "active_colony",
                won=won,
                outcome="win" if won else "loss",
                loot=loot,
                spice_eq=float(game_data.spice_equivalent(loot)),
                attacker_lost_spice=0.0,
                debris_spice_eq=0.0,
                one_way_sec=estimate_physics_one_way_seconds(origin, target),
                time_source="physics_formula",
            )
        )
    return records


def load_attack_telemetry(db_path: Path) -> dict[str, Any]:
    """Read the origin colony and attack history from a temporary copy of `db_path`; derive one record per attack.

    Without a database, or without a planet snapshot in it, the origin colony and the universe speed fall
    back to `cold_start_colony` in `planner_settings.json`.
    """
    cold_start = PLANNER_SETTINGS["cold_start_colony"]
    origin_coords = str(cold_start["origin_coords"])
    universe_speedup_factor = float(cold_start["speedup_factor"])
    colony_buildings: dict[str, int] = {}
    colony_researches: dict[str, int] = {}
    battle_rows: list[dict[str, Any]] = []
    raid_rows: list[dict[str, Any]] = []
    recycle_rows: list[dict[str, Any]] = []
    active_bash_blocks = 0

    if has_database(db_path):
        with open_readonly_db_copy(db_path) as conn:
            snapshot = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
            if snapshot:
                if snapshot["coords"]:
                    origin_coords = str(snapshot["coords"])
                colony_buildings = parse_level_map(snapshot["buildings_json"])
                colony_researches = parse_level_map(snapshot["researches_json"])

            saved_speed = read_latest_game_state(conn).get("speedup_factor")
            if isinstance(saved_speed, (int, float)) and math.isfinite(float(saved_speed)) and float(saved_speed) > 0.0:
                universe_speedup_factor = float(saved_speed)

            battle_rows = [dict(row) for row in conn.execute(SQL_BATTLE_REPORTS_MER_JOIN)]
            raid_rows = [dict(row) for row in conn.execute(SQL_RAID_DISPATCH_ORIGIN_JOIN, (origin_coords,))]
            recycle_rows = [dict(row) for row in conn.execute(SQL_RECYCLE_HARVESTS)]
            latest_event_ts = max(
                [0.0]
                + [float(raid.get("attacked_ts") or 0.0) for raid in raid_rows]
                + [float(battle.get("ts") or 0.0) for battle in battle_rows]
            )
            active_bash_blocks = int(conn.execute(SQL_ACTIVE_BASH_BLOCKS, (latest_event_ts,)).fetchone()[0])

    if battle_rows:
        server_clock_ts = [parse_server_clock(battle.get("msg_date_text")) for battle in battle_rows]
        tz_offset_sec = infer_server_tz_offset_sec(battle_rows, server_clock_ts, raid_rows)
        records = attack_records_from_battles(battle_rows, server_clock_ts, raid_rows, origin_coords, tz_offset_sec)
    else:
        records = attack_records_from_raids(raid_rows, origin_coords)
    records.sort(key=lambda record: float(record["event_ts"]))

    dispatch_timestamps = sorted(
        float(raid["attacked_ts"])
        for raid in raid_rows
        if isinstance(raid.get("attacked_ts"), (int, float)) and math.isfinite(float(raid["attacked_ts"]))
    )
    if not dispatch_timestamps:
        dispatch_timestamps = [float(record["event_ts"]) for record in records if float(record["event_ts"]) > 0.0]

    fleet_base_level = colony_buildings.get("b10", 0)
    return {
        "records": records,
        "dispatch_timestamps": dispatch_timestamps,
        "origin_coords": origin_coords,
        "colony_buildings": colony_buildings,
        "colony_researches": colony_researches,
        "universe_speedup_factor": universe_speedup_factor,
        "fleet_base_level": fleet_base_level,
        "fleet_slot_capacity": 1 + fleet_base_level,
        "engine_tech_levels": {key: colony_researches.get(key, 0) for key in ("r4", "r5", "r6")},
        "recycle_rows": recycle_rows,
        "active_bash_blocks": active_bash_blocks,
        "source_label": (
            "sqlite:mer_joined(battle_reports+raid_history+spy_targets)" if battle_rows or raid_rows else "cold_start"
        ),
    }

### 2.3 Load Telemetry into DataFrames

Runs the loader and names its results:

- `telemetry_bundle` — everything the loader returned;
- `origin_planet_coords`, `colony_buildings`, `fleet_slot_capacity` (1 + Fleet base level) — the reference colony;
- `colony_researches` — the account's research levels, the same on every planet;
- `attacks_df` — one row per attack; `recycles_df` — one row per recycler harvest;
- `attack_dispatch_ts_arr` — the dispatch times in epoch seconds.

The reference colony is the planet with the newest snapshot. With a single planet that is the home planet; with several planets it is whichever one was snapshotted last. It provides the default origin of an attack, the fleet slots, and the building levels that the graph of Section 7 treats as current. The parameter tables of Section 6 do not use it: they read every colony.

In [59]:
telemetry_bundle: dict[str, Any] = load_attack_telemetry(LIVE_DB_PATH)
telemetry_source: str = telemetry_bundle["source_label"]
origin_planet_coords: str = telemetry_bundle["origin_coords"]
colony_buildings: dict[str, int] = telemetry_bundle["colony_buildings"]
colony_researches: dict[str, int] = telemetry_bundle["colony_researches"]
fleet_slot_capacity: int = int(telemetry_bundle["fleet_slot_capacity"])

attacks_df = to_float_columns(
    pd.DataFrame(telemetry_bundle["records"], columns=list(ATTACK_DF_COLUMNS)),
    ATTACK_FLOAT_COLUMNS,
)
attacks_df["won"] = attacks_df["won"].astype(bool)
recycles_df = to_float_columns(
    pd.DataFrame(telemetry_bundle["recycle_rows"], columns=list(RECYCLE_DF_COLUMNS)),
    RECYCLE_FLOAT_COLUMNS,
)
attack_dispatch_ts_arr: np.ndarray = np.asarray(telemetry_bundle["dispatch_timestamps"], dtype=np.float64)

print(
    pd.DataFrame(
        [
            ("telemetry source", telemetry_source),
            ("origin colony", origin_planet_coords),
            ("universe speed", f"{telemetry_bundle['universe_speedup_factor']:.1f}x"),
            ("fleet slots", f"{fleet_slot_capacity} (Fleet base b10 at level {telemetry_bundle['fleet_base_level']})"),
            (
                "engine research r4 / r5 / r6",
                " / ".join(str(telemetry_bundle["engine_tech_levels"][key]) for key in ("r4", "r5", "r6")),
            ),
            ("attacks_df", f"{len(attacks_df)} attacks on {attacks_df['target_coords'].nunique()} targets"),
            ("targets with an active bash block", telemetry_bundle["active_bash_blocks"]),
            ("recycles_df", f"{len(recycles_df)} recycler harvests"),
        ],
        columns=["property", "value"],
    ).to_string(index=False)
)

                         property                                                      value
                 telemetry source sqlite:mer_joined(battle_reports+raid_history+spy_targets)
                    origin colony                                                  02:380:12
                   universe speed                                                      20.0x
                      fleet slots                              7 (Fleet base b10 at level 6)
     engine research r4 / r5 / r6                                                  8 / 7 / 0
                       attacks_df                                     7 attacks on 2 targets
targets with an active bash block                                                         35
                      recycles_df                                       14 recycler harvests


## 3. Mean & Standard Deviation of Resources Achieved in Attacks

`attack_resource_stats_df` holds, for all attacks and for winning attacks only, the mean, the sample standard deviation (`ddof=1`) and the population standard deviation (`ddof=0`) of:

- the loot per resource (`pig_iron`, `metal`, `kryptonite`, `spice`) and its Spice equivalent `spice_eq` at the Merchant parity `6 : 2.5 : 3 : 1`;
- the net yield `net_spice_eq`: loot minus the attacker's fleet losses;
- the rubble left in orbit, `debris_spice_eq`.

The same numbers are available as dictionaries (`attack_resource_mean`, `attack_resource_std`, `attack_resource_pop_std`, `winning_attack_resource_mean`, `winning_attack_resource_std`). `recycle_spice_eq_mean` and `recycle_spice_eq_std` describe the recycler harvests in `recycles_df`. Everything is computed with `numpy` on the `pandas` tables of Section 2.

In [60]:
METRIC_KEYS: tuple[str, ...] = (*RESOURCE_KEYS, "spice_eq", "net_spice_eq", "debris_spice_eq")


def metric_stats(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Mean, sample std (`ddof=1`) and population std (`ddof=0`) of each metric column; zeros when undefined."""
    values = frame[list(METRIC_KEYS)].to_numpy(dtype=np.float64)
    zeros = np.zeros(len(METRIC_KEYS), dtype=np.float64)
    return {
        "mean": np.mean(values, axis=0) if len(frame) >= 1 else zeros,
        "sample_std": np.std(values, axis=0, ddof=1) if len(frame) >= 2 else zeros,
        "pop_std": np.std(values, axis=0, ddof=0) if len(frame) >= 1 else zeros,
    }


def build_attack_resource_stats(all_attacks: pd.DataFrame, winning_attacks: pd.DataFrame) -> pd.DataFrame:
    """One row per metric with the statistics of all attacks and of winning attacks."""
    all_stats, win_stats = metric_stats(all_attacks), metric_stats(winning_attacks)
    return pd.DataFrame(
        {
            "metric": list(METRIC_KEYS),
            "samples_all": len(all_attacks),
            "mean_all": all_stats["mean"],
            "sample_std_all": all_stats["sample_std"],
            "pop_std_all": all_stats["pop_std"],
            "samples_wins": len(winning_attacks),
            "mean_wins": win_stats["mean"],
            "sample_std_wins": win_stats["sample_std"],
            "pop_std_wins": win_stats["pop_std"],
        }
    )


def stats_column_by_metric(stats: pd.DataFrame, column: str) -> dict[str, float]:
    """`{metric: value}` for one column of an `attack_resource_stats_df` table."""
    return {str(metric): float(value) for metric, value in zip(stats["metric"], stats[column])}


winning_attacks_df = attacks_df.loc[attacks_df["won"]].reset_index(drop=True)
attack_resource_stats_df = build_attack_resource_stats(attacks_df, winning_attacks_df)

attack_resource_mean: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "mean_all")
attack_resource_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "sample_std_all")
attack_resource_pop_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "pop_std_all")
winning_attack_resource_mean: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "mean_wins")
winning_attack_resource_std: dict[str, float] = stats_column_by_metric(attack_resource_stats_df, "sample_std_wins")

recycle_spice_eq_mean, recycle_spice_eq_std, _ = compute_mean_and_std(
    recycles_df["harvested_spice_eq"].to_numpy(dtype=np.float64)
)

print(attack_resource_stats_df.round(2).to_string(index=False))
print(
    f"\nRecycler harvests ({len(recycles_df)}): mean {recycle_spice_eq_mean:,.2f} SpiceEq, "
    f"sample std {recycle_spice_eq_std:,.2f} SpiceEq"
)

         metric  samples_all  mean_all  sample_std_all  pop_std_all  samples_wins  mean_wins  sample_std_wins  pop_std_wins
       pig_iron            7   4065.86         4575.74      4236.31             6    4743.50          4611.68       4209.87
          metal            7   5377.29         5413.95      5012.34             6    6273.50          5331.62       4867.08
     kryptonite            7   5705.14         5930.12      5490.23             6    6656.00          5882.60       5370.05
          spice            7   2328.00         2527.64      2340.14             6    2716.00          2530.25       2309.79
       spice_eq            7   7058.27         6697.76      6200.92             6    8234.65          6496.73       5930.67
   net_spice_eq            7  -2532.16        29097.11     26938.69             6    8234.65          6496.73       5930.67
debris_spice_eq            7   2582.86         6553.12      6067.01             6     106.67           165.25        150.85

Recycle

## 4. Mean & Standard Deviation of Attack Time

How long an attack keeps a fleet slot busy, and how often attacks are sent. `attack_time_stats_df` reports the count, mean, sample standard deviation (`ddof=1`), population standard deviation (`ddof=0`) and median, in seconds, of three `numpy` arrays:

- `one_way_times_arr` — flight time from dispatch to impact;
- `round_trip_times_arr` — twice the one-way time, which is how long the fleet slot stays occupied;
- `inter_attack_intervals_arr` — the positive gaps between consecutive dispatches.

The scalars `attack_one_way_time_*`, `attack_time_*` (round trip) and `inter_attack_time_*` hold the unrounded mean, sample and population standard deviation.

In [61]:
def positive_gaps(timestamps: np.ndarray) -> np.ndarray:
    """Positive differences between consecutive timestamps (sorted first)."""
    gaps = np.diff(np.sort(timestamps))
    return gaps[gaps > 0.0]


def time_stats_row(label: str, samples: np.ndarray) -> dict[str, Any]:
    """One `attack_time_stats_df` row: count, mean, standard deviations and median of `samples` (seconds)."""
    mean, sample_std, pop_std = compute_mean_and_std(samples)
    return {
        "time_metric": label,
        "samples": int(samples.size),
        "mean_sec": round(mean, 2),
        "mean_text": fmt_duration(mean),
        "sample_std_sec": round(sample_std, 2),
        "sample_std_text": fmt_duration(sample_std),
        "pop_std_sec": round(pop_std, 2),
        "median_sec": round(float(np.median(samples)) if samples.size > 0 else 0.0, 2),
    }


one_way_times_arr: np.ndarray = attacks_df["one_way_sec"].to_numpy(dtype=np.float64)
round_trip_times_arr: np.ndarray = attacks_df["round_trip_sec"].to_numpy(dtype=np.float64)
inter_attack_intervals_arr: np.ndarray = positive_gaps(attack_dispatch_ts_arr)

attack_one_way_time_mean_sec, attack_one_way_time_std_sec, attack_one_way_time_pop_std_sec = (
    compute_mean_and_std(one_way_times_arr)
)
attack_time_mean_sec, attack_time_std_sec, attack_time_pop_std_sec = compute_mean_and_std(round_trip_times_arr)
inter_attack_time_mean_sec, inter_attack_time_std_sec, inter_attack_time_pop_std_sec = (
    compute_mean_and_std(inter_attack_intervals_arr)
)

attack_time_stats_df = pd.DataFrame(
    [
        time_stats_row("One-Way Flight Time (Dispatch -> Impact)", one_way_times_arr),
        time_stats_row("Round-Trip Attack Time (Slot Occupation)", round_trip_times_arr),
        time_stats_row("Inter-Attack Dispatch Interval", inter_attack_intervals_arr),
    ]
)

print(attack_time_stats_df.to_string(index=False))

                             time_metric  samples  mean_sec mean_text  sample_std_sec sample_std_text  pop_std_sec  median_sec
One-Way Flight Time (Dispatch -> Impact)        7    151.47    2m 31s          113.25          1m 53s       104.85      115.58
Round-Trip Attack Time (Slot Occupation)        7    302.93    5m 03s          226.49          3m 46s       209.69      231.17
          Inter-Attack Dispatch Interval        6  13249.65    3h 40m        21626.47          6h 00m     19742.18     3354.80


## 5. Attack Success Rate

With `W` wins in `N` attacks:

$$\hat{p} = \frac{W}{N}, \qquad \mathrm{SE}(\hat{p}) = \sqrt{\frac{\hat{p}\,(1 - \hat{p})}{N}}, \qquad \tilde{p} = \frac{W + 1}{N + 2}$$

- `attack_success_rate` is $\hat{p}$, `attack_success_rate_std_err` its binomial standard error, and `attack_success_rate_smoothed` the Laplace-smoothed $\tilde{p}$, which stays away from 0 % and 100 % when `N` is small.
- `attack_class_stats_df` and `attack_target_stats_df` break attempts, wins, losses, success rate and loot down by target class and by target (`pandas` `groupby`). `per_class_success_rate` and `per_target_success_rate` are the same numbers as dictionaries.
- `expected_raids_per_slot_hr` is 3600 divided by the mean round trip. Multiplied by the mean loot it gives the gross and net Spice-equivalent yield per fleet-slot hour, and with all `fleet_slot_capacity` slots the ceiling `max_empire_gross_spice_eq_hr`.
- `ATTACK_SIM_PARAMS` collects the scalars of Sections 3–5 for the simulation; `ATTACK_STATS_TABLES` registers the tables by name.

In [62]:
def summarize_attacks_by(attacks: pd.DataFrame, group_column: str) -> pd.DataFrame:
    """Attempts, wins, losses, success rate and loot statistics of `attacks` per value of `group_column`."""
    if attacks.empty:
        return pd.DataFrame(
            columns=[
                group_column,
                "attempts",
                "wins",
                "success_rate",
                "mean_spice_eq",
                "sample_std_spice_eq",
                "mean_net_spice_eq",
                "sample_std_net_spice_eq",
                "losses",
            ]
        )
    summary = (
        attacks.assign(won_num=attacks["won"].astype(np.float64))
        .groupby(group_column, as_index=False)
        .agg(
            attempts=("won_num", "count"),
            wins=("won_num", "sum"),
            success_rate=("won_num", "mean"),
            mean_spice_eq=("spice_eq", "mean"),
            sample_std_spice_eq=("spice_eq", "std"),
            mean_net_spice_eq=("net_spice_eq", "mean"),
            sample_std_net_spice_eq=("net_spice_eq", "std"),
        )
        .fillna(0.0)
        .sort_values(group_column)
        .reset_index(drop=True)
    )
    summary["wins"] = summary["wins"].astype(int)
    summary["losses"] = summary["attempts"] - summary["wins"]
    return summary


def success_rates_by(summary: pd.DataFrame, group_column: str) -> dict[str, dict[str, Any]]:
    """`{group value: counts, success rate and loot means}` view of a `summarize_attacks_by` table."""
    return {
        str(row[group_column]): {
            "attempts": int(row["attempts"]),
            "wins": int(row["wins"]),
            "losses": int(row["losses"]),
            "success_rate": float(row["success_rate"]),
            "mean_spice_eq": float(row["mean_spice_eq"]),
            "std_spice_eq": float(row["sample_std_spice_eq"]),
            "mean_net_spice_eq": float(row["mean_net_spice_eq"]),
        }
        for row in summary.to_dict("records")
    }


attack_attempts: int = int(len(attacks_df))
attack_wins: int = int(attacks_df["won"].sum())
attack_losses: int = attack_attempts - attack_wins

attack_success_rate: float = float(attacks_df["won"].mean()) if attack_attempts > 0 else 0.0
attack_success_rate_std_err: float = (
    float(np.sqrt(attack_success_rate * (1.0 - attack_success_rate) / float(attack_attempts)))
    if attack_attempts > 0
    else 0.0
)
attack_success_rate_smoothed: float = float(attack_wins + 1) / float(attack_attempts + 2)

attack_class_stats_df = summarize_attacks_by(attacks_df, "target_class")
attack_target_stats_df = summarize_attacks_by(attacks_df, "target_coords")
per_class_success_rate: dict[str, dict[str, Any]] = success_rates_by(attack_class_stats_df, "target_class")
per_target_success_rate: dict[str, dict[str, Any]] = success_rates_by(attack_target_stats_df, "target_coords")

expected_raids_per_slot_hr: float = 3600.0 / attack_time_mean_sec if attack_time_mean_sec > 0.0 else 0.0
expected_spice_eq_per_slot_hr: float = expected_raids_per_slot_hr * attack_resource_mean["spice_eq"]
expected_net_spice_eq_per_slot_hr: float = expected_raids_per_slot_hr * attack_resource_mean["net_spice_eq"]
max_empire_gross_spice_eq_hr: float = expected_spice_eq_per_slot_hr * float(fleet_slot_capacity)

attack_summary_stats_df = pd.DataFrame(
    [
        {
            "metric": "Total Attacks",
            "value": float(attack_attempts),
            "display": f"{attack_attempts} ({attack_wins} wins / {attack_losses} losses)",
        },
        {
            "metric": "Empirical Success Rate",
            "value": round(attack_success_rate, 6),
            "display": f"{attack_success_rate:.2%} (+/- {attack_success_rate_std_err:.2%} SE)",
        },
        {
            "metric": "Laplace-Smoothed Success Rate",
            "value": round(attack_success_rate_smoothed, 6),
            "display": f"{attack_success_rate_smoothed:.2%}",
        },
        {
            "metric": "Max Sorties per Fleet Slot / Hour",
            "value": round(expected_raids_per_slot_hr, 4),
            "display": (
                f"{expected_raids_per_slot_hr:.2f} raids/hr (mean round-trip {fmt_duration(attack_time_mean_sec)})"
            ),
        },
        {
            "metric": "Gross Loot per Slot-Hour (SpiceEq)",
            "value": round(expected_spice_eq_per_slot_hr, 2),
            "display": f"{expected_spice_eq_per_slot_hr:,.1f} SpiceEq/hr",
        },
        {
            "metric": "Net Loot per Slot-Hour (SpiceEq)",
            "value": round(expected_net_spice_eq_per_slot_hr, 2),
            "display": f"{expected_net_spice_eq_per_slot_hr:+,.1f} SpiceEq/hr",
        },
        {
            "metric": f"Full Capacity Ceiling ({fleet_slot_capacity} Slots)",
            "value": round(max_empire_gross_spice_eq_hr, 2),
            "display": f"{max_empire_gross_spice_eq_hr:,.1f} gross SpiceEq/hr",
        },
    ]
)

ATTACK_STATS_TABLES: dict[str, pd.DataFrame] = {
    "attacks": attacks_df,
    "recycles": recycles_df,
    "attack_resource_stats": attack_resource_stats_df,
    "attack_time_stats": attack_time_stats_df,
    "attack_summary_stats": attack_summary_stats_df,
    "attack_class_stats": attack_class_stats_df,
    "attack_target_stats": attack_target_stats_df,
}

ATTACK_SIM_PARAMS: dict[str, Any] = {
    "attempts": attack_attempts,
    "wins": attack_wins,
    "losses": attack_losses,
    "success_rate": attack_success_rate,
    "success_rate_std_err": attack_success_rate_std_err,
    "success_rate_smoothed": attack_success_rate_smoothed,
    "fleet_slot_capacity": fleet_slot_capacity,
    "resource_mean": dict(attack_resource_mean),
    "resource_std": dict(attack_resource_std),
    "winning_resource_mean": dict(winning_attack_resource_mean),
    "winning_resource_std": dict(winning_attack_resource_std),
    "recycle_spice_eq_mean": recycle_spice_eq_mean,
    "recycle_spice_eq_std": recycle_spice_eq_std,
    "one_way_time_mean_sec": attack_one_way_time_mean_sec,
    "one_way_time_std_sec": attack_one_way_time_std_sec,
    "round_trip_time_mean_sec": attack_time_mean_sec,
    "round_trip_time_std_sec": attack_time_std_sec,
    "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
    "inter_attack_time_std_sec": inter_attack_time_std_sec,
    "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
    "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
    "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
    "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
}

print(attack_summary_stats_df[["metric", "display"]].to_string(index=False))
if not attack_class_stats_df.empty:
    print("\nBy target class (attack_class_stats_df)")
    print(attack_class_stats_df.round(3).to_string(index=False))
if not attack_target_stats_df.empty:
    print("\nBy target (attack_target_stats_df)")
    print(attack_target_stats_df.round(3).to_string(index=False))

                            metric                                 display
                     Total Attacks                   7 (6 wins / 1 losses)
            Empirical Success Rate                  85.71% (+/- 13.23% SE)
     Laplace-Smoothed Success Rate                                  77.78%
 Max Sorties per Fleet Slot / Hour 11.88 raids/hr (mean round-trip 5m 03s)
Gross Loot per Slot-Hour (SpiceEq)                     83,879.4 SpiceEq/hr
  Net Loot per Slot-Hour (SpiceEq)                    -30,091.8 SpiceEq/hr
   Full Capacity Ceiling (7 Slots)              587,156.0 gross SpiceEq/hr

By target class (attack_class_stats_df)
   target_class  attempts  wins  success_rate  mean_spice_eq  sample_std_spice_eq  mean_net_spice_eq  sample_std_net_spice_eq  losses
         spacia         3     3          1.00       7821.400             8892.062           7821.400                 8892.062       0
yellow_inactive         4     3          0.75       6485.925             5999.023         -

## 6. Game Parameters Catalog

Tables of the game's parameters, built with `pandas` from `spaceinvasion.core.game_data` and the metadata of Section 1.6: costs, build and research times, production, energy, storage, combat statistics and technology effects of every building, technology, ship and defense.

Build times, the Spice mine's output and the Solar cells' energy depend on the planet. Section 6 therefore works on the whole empire: every colony you own, followed by generic planets (empty colony slots) until the roster has `MAX_EMPIRE_PLANETS` normal planets. It produces two kinds of tables:

- **Catalog and level tables** (`buildings_levels_df`, `technologies_levels_df`, `ships_df`, `defenses_df` and the two catalogs) list every entity and level for a generic planet: universe speed and research levels applied, no accelerator buildings, the temperature `EMPTY_COLONY_DEFAULT_TEMP_C`.
- **Per-planet tables** (`planet_buildings_df`, `planet_technologies_df`, `planet_ships_df`, `planet_defenses_df`) have one row per planet and entity, computed with that planet's own building levels and temperature.

This cell loads the roster and collects the inputs:

- `live_planet_states` and `empire_planet_states` — your colonies, and your colonies followed by the generic planets, as `PlanetState` objects;
- `GAME_PARAM_CONTEXT` — what is the same on every planet: the universe speed and the research levels `r1_mining_tech` to `r10_defense_systems`;
- `PLANET_PARAM_CONTEXTS` — one context per planet of the roster: the account-wide values, the planet's temperature, and the levels of the buildings that shorten build and research times (`b7_dev_center`, `b8_research_lab`, `b9_weapons_factory`, `b17_microsystem`). `GENERIC_PLANET_CONTEXT` is the context of a generic planet;
- `planet_contexts_df` — the planets' own inputs as a table, with the resulting divisors of the base times (formulas in Section 1.3);
- `merchant_rates_df` — the Merchant parity (`6 Pig iron : 2.5 Metal : 3 Kryptonite : 1 Spice`), the ranking value of a resource unit (1 point per 1,000 resources spent) and the mine of each resource.

`MAX_BUILDING_TABLE_LEVEL` and `MAX_RESEARCH_TABLE_LEVEL` cap the level tables. `MAX_EMPIRE_PLANETS`, `EMPTY_COLONY_SIZE_FIELDS` and `EMPTY_COLONY_DEFAULT_TEMP_C` describe the roster. All five come from `planner_settings.json`. The planet limit is a setting, not a value read from the game: 11 is the default of the bot's own colony cap (`StrategyConfig.max_planets`).

Every table builder takes a context, so a full table can be rebuilt for any planet or state: `build_building_tables(PLANET_PARAM_CONTEXTS[0], MAX_BUILDING_TABLE_LEVEL)` returns the catalog and level table of the first planet, and `build_ship_table({**GENERIC_PLANET_CONTEXT, "r8_weapons_tech": 5})` the ships with Weapons technology 5.

In [63]:
MAX_BUILDING_TABLE_LEVEL: int = int(PLANNER_SETTINGS["table_level_caps"]["buildings"])
MAX_RESEARCH_TABLE_LEVEL: int = int(PLANNER_SETTINGS["table_level_caps"]["researches"])
MAX_EMPIRE_PLANETS: int = int(PLANNER_SETTINGS["empire"]["max_planets"])
EMPTY_COLONY_SIZE_FIELDS: int = int(PLANNER_SETTINGS["empire"]["empty_colony_fields"])
EMPTY_COLONY_DEFAULT_TEMP_C: int = int(PLANNER_SETTINGS["empire"]["empty_colony_temperature_c"])


def planet_param_context(
    account_context: dict[str, Any],
    slot_index: int,
    planet: PlanetState,
    *,
    is_empty_slot: bool,
) -> dict[str, Any]:
    """Inputs of the table builders for one planet: the account-wide context plus the planet's own values."""
    return {
        **account_context,
        "slot_index": slot_index,
        "planet_id": int(planet.planet_id),
        "coords": planet.coords,
        "planet_status": "empty_colony_slot" if is_empty_slot else "colonized",
        "planet_name": planet.name,
        "planet_type": planet.planet_type,
        "planet_temperature_c": int(planet.temperature),
        "b7_dev_center": planet.buildings.get("b7", 0),
        "b8_research_lab": planet.buildings.get("b8", 0),
        "b9_weapons_factory": planet.buildings.get("b9", 0),
        "b17_microsystem": planet.buildings.get("b17", 0),
    }


def planet_identity(context: dict[str, Any]) -> dict[str, Any]:
    """The four columns that identify a planet in every per-planet table of Section 6."""
    return {column: context[column] for column in ("slot_index", "planet_id", "coords", "planet_status")}


def planet_context_row(context: dict[str, Any]) -> dict[str, Any]:
    """One `planet_contexts_df` row: a planet's own inputs and the time divisors that follow from them."""
    speedup = context["speedup_factor"]
    return {
        **planet_identity(context),
        "planet_name": context["planet_name"],
        "planet_type": context["planet_type"],
        "temperature_c": context["planet_temperature_c"],
        "b7_dev_center": context["b7_dev_center"],
        "b8_research_lab": context["b8_research_lab"],
        "b9_weapons_factory": context["b9_weapons_factory"],
        "b17_microsystem": context["b17_microsystem"],
        "building_time_divisor": building_time_divisor(context["b7_dev_center"], context["b17_microsystem"], speedup),
        "research_time_divisor": research_time_divisor(context["b8_research_lab"], speedup),
        "shipyard_time_divisor": shipyard_time_divisor(
            context["b9_weapons_factory"], context["b17_microsystem"], speedup
        ),
    }


def next_level_row(
    context: dict[str, Any],
    current_level: int,
    level_row: dict[str, Any],
    extra: dict[str, Any],
) -> dict[str, Any]:
    """Row of a per-planet table: the planet, then a level-table row for the level after `current_level`.

    The `level` column of `level_row` becomes `next_level`; `extra` and `current_level` are placed before it.
    """
    row = planet_identity(context)
    for column, value in level_row.items():
        if column == "level":
            row.update(extra)
            row["current_level"] = current_level
            row["next_level"] = value
        else:
            row[column] = value
    return row


# Every owned colony, then generic planets (empty colony slots) until the empire has MAX_EMPIRE_PLANETS normal planets.
live_planet_states: list[PlanetState] = load_live_planet_states(LIVE_DB_PATH)
empire_planet_states: list[PlanetState] = build_padded_empire_planets(
    live_planet_states,
    max_planets=MAX_EMPIRE_PLANETS,
    empty_planet_size=EMPTY_COLONY_SIZE_FIELDS,
    empty_temp_c=EMPTY_COLONY_DEFAULT_TEMP_C,
)

# Account-wide inputs: the universe speed and the research levels are the same on every planet.
GAME_PARAM_CONTEXT: dict[str, Any] = {
    "wiki_synced_at_utc": game_data.WIKI_SYNCED_AT_UTC,
    "speedup_factor": float(telemetry_bundle["universe_speedup_factor"]),
    "r1_mining_tech": colony_researches.get("r1", 0),
    "r2_energy_tech": colony_researches.get("r2", 0),
    "r4_rocket_engine": colony_researches.get("r4", 0),
    "r5_diffusion_engine": colony_researches.get("r5", 0),
    "r6_warp_engine": colony_researches.get("r6", 0),
    "r8_weapons_tech": colony_researches.get("r8", 0),
    "r9_shield_tech": colony_researches.get("r9", 0),
    "r10_defense_systems": colony_researches.get("r10", 0),
    "points_per_resource_divisor": float(POINTS_PER_RESOURCE_DIVISOR),
}
# One builder context per planet of the roster, and one for a generic planet, the baseline of the level tables.
PLANET_PARAM_CONTEXTS: list[dict[str, Any]] = [
    planet_param_context(GAME_PARAM_CONTEXT, slot_index, planet, is_empty_slot=slot_index > len(live_planet_states))
    for slot_index, planet in enumerate(empire_planet_states, start=1)
]
GENERIC_PLANET_CONTEXT: dict[str, Any] = planet_param_context(
    GAME_PARAM_CONTEXT,
    0,
    empty_colony_slot(0, EMPTY_COLONY_SIZE_FIELDS, EMPTY_COLONY_DEFAULT_TEMP_C),
    is_empty_slot=True,
)
planet_contexts_df = pd.DataFrame([planet_context_row(context) for context in PLANET_PARAM_CONTEXTS])

merchant_rates_df = pd.DataFrame(
    [
        {
            "resource": resource,
            "units_per_1_spice": float(game_data.MERCHANT_RATES_IN_SPICE[resource]),
            "spice_eq_per_unit": round(1.0 / float(game_data.MERCHANT_RATES_IN_SPICE[resource]), 6),
            "points_per_unit": 1.0 / float(POINTS_PER_RESOURCE_DIVISOR),
            "mine_building_key": RESOURCE_METADATA[resource]["mine_building_key"],
            "boosted_by_mining_tech_r1": bool(RESOURCE_METADATA[resource]["boosted_by_mining_tech"]),
            "notes": RESOURCE_METADATA[resource]["notes"],
        }
        for resource in RESOURCE_KEYS
    ]
)

print(
    f"{len(live_planet_states)} owned planet(s) + {len(empire_planet_states) - len(live_planet_states)} generic "
    f"{EMPTY_COLONY_SIZE_FIELDS}-field planet(s), universe speed {GAME_PARAM_CONTEXT['speedup_factor']:.1f}x\n"
)
print(planet_contexts_df.to_string(index=False))
print()
print(merchant_rates_df.to_string(index=False))

1 owned planet(s) + 10 generic 300-field planet(s), universe speed 20.0x

 slot_index  planet_id    coords     planet_status          planet_name planet_type  temperature_c  b7_dev_center  b8_research_lab  b9_weapons_factory  b17_microsystem  building_time_divisor  research_time_divisor  shipyard_time_divisor
          1      12476 02:380:12         colonized               Colony      normal             31              9                3                   4                0                  200.0                   80.0                   80.0
          2         -1  EMPTY:01 empty_colony_slot  Empty Colony Slot 1      normal              0              0                0                   0                0                   20.0                   20.0                   20.0
          3         -2  EMPTY:02 empty_colony_slot  Empty Colony Slot 2      normal              0              0                0                   0                0                   20.0                   20.0  

### 6.1 Buildings (`b1`–`b21`)

- `buildings_catalog_df` — one row per building: category, whether it can be built on an asteroid, number of levels in the catalog, cost scale factor, level-1 cost and time, prerequisites and primary effect.
- `buildings_levels_df` — one row per building level on a generic planet, up to `MAX_BUILDING_TABLE_LEVEL`:
  - cost per resource, `cost_spice_eq`, and `points_gain` (cost / 1,000);
  - `base_time_s` from the catalog and `effective_build_time_s`. On a specific planet, the time is `base_time_s` divided by that planet's `building_time_divisor` in `planet_contexts_df`;
  - mines: `hourly_prod_1x` (universe speed 1, no research) and `hourly_prod_effective` (universe speed, and +2 % per Mining technology level except for the Blast furnace). The Spice mine's output also follows the planet's temperature: the colder, the more;
  - feedstock: `pig_iron_consumed_hr_effective` (the Blast furnace uses 2 Pig iron per Metal) and `spice_consumed_hr` (fuel of the Thermosolar plant);
  - `net_spice_eq_hr_effective`, its gain over the previous level `delta_net_spice_eq_hr`, and `marginal_roi_hours`, the hours the level needs to pay back its cost (empty when the level adds no net output);
  - energy produced (positive) or consumed (negative): `energy_at_level_base`, and `energy_at_level_effective` with +1 % per Energy technology level on production only;
  - `storage_capacity` of the warehouses (`100,000 * 2^level`) and `time_reduction_multiplier` of the accelerator buildings (`1 / (level + 1)` for `b7` and `b8`, `1 / level` for `b9`, `0.5^level` for `b17`).
- `planet_buildings_df` — one row per planet and building: the `current_level` on that planet, the `next_level`, and the level-table columns of that next level computed for the planet, with its own build time and Spice mine output. `delta_net_spice_eq_hr` and `energy_delta_from_prev` are the gains over the current level. `buildable_on_planet` is false for a building that the catalog does not allow on an asteroid. A building already at the last level of the catalog has no row.

In [64]:
def building_time_reduction(key: str, level: int) -> float:
    """Share of the base time that an accelerator building at `level` leaves (1.0 for every other building)."""
    if key in ("b7", "b8"):  # Development center (buildings) and Research lab (research)
        return 1.0 / float(level + 1)
    if key == "b9":  # Weapons factory (ships and defenses)
        return 1.0 / float(max(1, level))
    if key == "b17":  # Microsystem accelerator (buildings, ships and defenses)
        return 0.5 ** float(level)
    return 1.0


def produced_resource(building_key: str) -> str:
    """Resource that a building produces (`"none"` for a building that is not a mine)."""
    return next(
        (resource for resource, meta in RESOURCE_METADATA.items() if meta["mine_building_key"] == building_key),
        "none",
    )


def building_level_economy(building_id: int, level: int, context: dict[str, Any]) -> dict[str, float]:
    """Hourly output, feedstock, net Spice-equivalent value and energy of one building level on the context's planet.

    Level 0, a building that is not built, yields zeros.
    """
    row = (game_data.BUILDING_BY_ID[building_id].get("levels") or {}).get(level) or {}
    merchant_rates = game_data.MERCHANT_RATES_IN_SPICE
    resource = produced_resource(f"b{building_id}")
    speedup = float(context["speedup_factor"])

    # Hourly output scales with universe speed; Mining technology adds +2 % per level (not to the Blast furnace).
    # Only the Spice mine depends on the planet: its output follows the temperature.
    if resource != "none":
        temperature = int(context["planet_temperature_c"])
        production_1x = float(
            game_data.mine_production_per_hour(building_id, level, temperature=temperature, mining_tech_level=0)
        )
        production = speedup * float(
            game_data.mine_production_per_hour(
                building_id, level, temperature=temperature, mining_tech_level=int(context["r1_mining_tech"])
            )
        )
    else:
        production_1x = 0.0
        production = 0.0
    # Feedstock per hour: the Blast furnace's Pig iron (2 per Metal) scales with universe speed like its output;
    # the Thermosolar plant's Spice fuel is the catalog value, not multiplied by universe speed.
    pig_iron_used = float(row.get("pig_iron_used") or 0.0) * speedup
    spice_used = float(row.get("spice_used") or 0.0)
    gross_spice_eq_hr = production / float(merchant_rates[resource]) if resource in merchant_rates else 0.0
    feedstock_spice_eq_hr = pig_iron_used / float(merchant_rates["pig_iron"]) + spice_used / float(
        merchant_rates["spice"]
    )
    # Energy: Energy technology boosts production (positive values) only; consumption is unchanged.
    energy_base = float(row.get("energy") or 0.0)
    energy_multiplier = technology_multiplier("r2", context["r2_energy_tech"])
    return {
        "production_1x": production_1x,
        "production": production,
        "pig_iron_used": pig_iron_used,
        "spice_used": spice_used,
        "net_spice_eq_hr": gross_spice_eq_hr - feedstock_spice_eq_hr,
        "energy_base": energy_base,
        "energy": (energy_base * energy_multiplier) if energy_base > 0.0 else energy_base,
    }


def building_level_row(building_id: int, level: int, base_time: float, context: dict[str, Any]) -> dict[str, Any]:
    """One level-table row: cost, time, production, energy and storage of `level` on the context's planet."""
    info = game_data.BUILDING_BY_ID[building_id]
    key = f"b{building_id}"
    row = (info.get("levels") or {}).get(level) or {}
    cost = game_data.building_cost(building_id, level)
    cost_spice_eq = float(game_data.spice_equivalent(cost))
    build_time = base_time / building_time_divisor(
        context["b7_dev_center"], context["b17_microsystem"], context["speedup_factor"]
    )
    economy = building_level_economy(building_id, level, context)
    previous = building_level_economy(building_id, level - 1, context)
    # Net value gained over the previous level, and how long this level takes to pay for itself.
    delta_net_spice_eq_hr = economy["net_spice_eq_hr"] - previous["net_spice_eq_hr"]
    roi_hours = (cost_spice_eq / delta_net_spice_eq_hr) if delta_net_spice_eq_hr > 0.0 else float("inf")
    return {
        "building_id": building_id,
        "key": key,
        "slug": str(info["slug"]),
        "name": str(info["name"]),
        "category": BUILDING_METADATA[key]["category"],
        "level": level,
        "pig_iron": int(cost["pig_iron"]),
        "metal": int(cost["metal"]),
        "kryptonite": int(cost["kryptonite"]),
        "spice": int(cost["spice"]),
        "cost_spice_eq": round(cost_spice_eq, 2),
        "points_gain": round(cost_points(cost), 3),
        "base_time_s": round(base_time, 2),
        "effective_build_time_s": round(build_time, 2),
        "effective_build_time_text": fmt_duration(build_time),
        "produced_resource": produced_resource(key),
        "hourly_prod_1x": round(economy["production_1x"], 2),
        "hourly_prod_effective": round(economy["production"], 2),
        "pig_iron_consumed_hr_effective": round(economy["pig_iron_used"], 2),
        "spice_consumed_hr": round(economy["spice_used"], 2),
        "net_spice_eq_hr_effective": round(economy["net_spice_eq_hr"], 2),
        "delta_net_spice_eq_hr": round(delta_net_spice_eq_hr, 2),
        "marginal_roi_hours": round(roi_hours, 2) if math.isfinite(roi_hours) else None,
        "energy_at_level_base": round(economy["energy_base"], 2),
        "energy_at_level_effective": round(economy["energy"], 2),
        "energy_delta_from_prev": round(economy["energy"] - previous["energy"], 2),
        "storage_capacity": int(row.get("storage_capacity") or 0),
        "time_reduction_multiplier": building_time_reduction(key, level),
    }


def build_building_tables(context: dict[str, Any], max_level: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build the building catalog (one row per building) and the level table (one row per building level)."""
    time_divisor = building_time_divisor(
        context["b7_dev_center"], context["b17_microsystem"], context["speedup_factor"]
    )
    catalog_rows: list[dict[str, Any]] = []
    level_rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        info = game_data.BUILDING_BY_ID[building_id]
        key = f"b{building_id}"
        levels = info.get("levels") or {}
        table_levels = int(info.get("max_table_level") or len(levels))
        base_times = build_monotonic_base_times(levels, table_levels)
        level_1_cost = game_data.building_cost(building_id, 1)
        level_1_time = float(base_times.get(1, 0.0))
        catalog_rows.append(
            {
                "building_id": building_id,
                "key": key,
                "slug": str(info["slug"]),
                "name": str(info["name"]),
                "category": BUILDING_METADATA[key]["category"],
                "asteroid_ok": bool(info.get("asteroid_ok", False)),
                "max_table_level": table_levels,
                "cost_scale_factor": BUILDING_METADATA[key]["cost_scale_factor"],
                "l1_pig_iron": int(level_1_cost["pig_iron"]),
                "l1_metal": int(level_1_cost["metal"]),
                "l1_kryptonite": int(level_1_cost["kryptonite"]),
                "l1_spice": int(level_1_cost["spice"]),
                "l1_cost_spice_eq": round(float(game_data.spice_equivalent(level_1_cost)), 2),
                "l1_base_time_s": round(level_1_time, 2),
                "l1_effective_time_s": round(level_1_time / time_divisor, 2),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
                "primary_effect": BUILDING_METADATA[key]["primary_effect"],
            }
        )
        level_rows.extend(
            building_level_row(building_id, level, float(base_times.get(level, 0.0)), context)
            for level in range(1, min(table_levels, max_level) + 1)
        )
    return pd.DataFrame(catalog_rows), pd.DataFrame(level_rows)


def build_planet_building_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and building: the level built there and the next level, computed for that planet.

    The columns after `next_level` are the level-table columns of that next level. A building already at the
    last level of the catalog has no row.
    """
    rows: list[dict[str, Any]] = []
    for planet, context in zip(planets, contexts):
        for building_id in sorted(game_data.BUILDING_BY_ID):
            info = game_data.BUILDING_BY_ID[building_id]
            levels = info.get("levels") or {}
            current_level = max(0, int(planet.buildings.get(f"b{building_id}", 0) or 0))
            next_level = current_level + 1
            if next_level > int(info.get("max_table_level") or len(levels)):
                continue
            base_time = float(build_monotonic_base_times(levels, next_level).get(next_level, 0.0))
            # The catalog marks the few buildings an asteroid can hold; a normal planet takes every building.
            buildable = planet.planet_type != "asteroid" or bool(info.get("asteroid_ok", False))
            rows.append(
                next_level_row(
                    context,
                    current_level,
                    building_level_row(building_id, next_level, base_time, context),
                    {"buildable_on_planet": buildable},
                )
            )
    return pd.DataFrame(rows)


buildings_catalog_df, buildings_levels_df = build_building_tables(GENERIC_PLANET_CONTEXT, MAX_BUILDING_TABLE_LEVEL)
planet_buildings_df = build_planet_building_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

print(
    f"buildings_catalog_df: {buildings_catalog_df.shape}, buildings_levels_df: {buildings_levels_df.shape} "
    f"(generic planet), planet_buildings_df: {planet_buildings_df.shape} (next level on each planet)\n"
)
print(
    buildings_catalog_df[
        [
            "key",
            "name",
            "category",
            "asteroid_ok",
            "cost_scale_factor",
            "l1_cost_spice_eq",
            "l1_base_time_s",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nNext level of the mines and the Fusion plant on the owned planets and on the first generic planet:")
print(
    planet_buildings_df[
        planet_buildings_df["key"].isin(["b1", "b2", "b3", "b4", "b5"])
        & planet_buildings_df["buildable_on_planet"]
        & (planet_buildings_df["slot_index"] <= len(live_planet_states) + 1)
    ][
        [
            "slot_index",
            "coords",
            "planet_status",
            "key",
            "name",
            "current_level",
            "next_level",
            "cost_spice_eq",
            "effective_build_time_text",
            "hourly_prod_effective",
            "delta_net_spice_eq_hr",
            "marginal_roi_hours",
            "energy_delta_from_prev",
        ]
    ].to_string(index=False)
)

buildings_catalog_df: (21, 17), buildings_levels_df: (975, 28) (generic planet), planet_buildings_df: (231, 34) (next level on each planet)

key                    name          category  asteroid_ok  cost_scale_factor  l1_cost_spice_eq  l1_base_time_s                                            prerequisites
 b1           Pig iron mine     resource_mine        False               1.51             21.33            93.0                                                     None
 b2           Blast furnace           smelter        False               1.51             20.83            90.0                                                     None
 b3         Kryptonite mine     resource_mine        False               1.51             33.33           136.0                                                     None
 b4              Spice mine     resource_mine        False               1.51            114.67           272.0                                                     None
 b5           

### 6.2 Technologies (`r1`–`r15`)

A research applies to the whole account: every planet, asteroid and fleet. Each level costs twice the previous one, and the research time is divided by `(b8 + 1) * speed` of the planet that researches it.

- `technologies_catalog_df` — one row per research: domain, current level, level-1 cost and time, the quantity it changes (`effect_metric`), `per_level_bonus_pct`, what it applies to, prerequisites and strategic effect. The descriptive columns come from `technologies.json`.
- `technologies_levels_df` — one row per research level, up to `MAX_RESEARCH_TABLE_LEVEL`: cost, `points_gain`, the base research time and the time on a generic planet, the cumulative bonus `bonus_pct_total`, `multiplier_at_level` (`1 + bonus_pct_total / 100`) and a readable `effect_at_level`.
- `planet_technologies_df` — one row per planet and research: the account's `current_level`, the `next_level`, and the level-table columns of that next level with the research time in the planet's own Research lab (`research_lab_level`). It shows where the next level is researched fastest. A planet whose `research_lab_level` is 0 cannot research; its time is the base time at universe speed. A research already at the last level of the catalog has no row.

Per-level bonuses: +2 % mine output for `r1` (not the Blast furnace), +1 % energy output for `r2`, +10 % ship speed for the engine researches `r4`, `r5` and `r6`, and +10 % attack, shield and hull for `r8`, `r9` and `r10`.

In [65]:
def technology_effect_at_level(key: str, level: int) -> str:
    """Human-readable effect of research `key` at `level`."""
    bonus_pct = float(TECHNOLOGY_METADATA[key]["per_level_bonus_pct"])
    if bonus_pct > 0.0:
        return f"+{bonus_pct * float(level):.0f}% ({technology_multiplier(key, level):.2f}x)"
    if key == "r15":  # Black hole research: scanner jam of 4 h + 1 h per level; level 2 unlocks the Imperial starbase
        return f"Scanner jam {4 + level}h" + (" + Starbase unlocked" if level >= 2 else "")
    return f"Tier L{level} unlock / capability"


def technology_identity(key: str, info: dict[str, Any]) -> dict[str, Any]:
    """The columns that identify a research in the catalog and level tables."""
    return {
        "research_id": int(key[1:]),
        "key": key,
        "slug": str(info["slug"]),
        "name": str(info["name"]),
        "domain": str(TECHNOLOGY_METADATA[key]["domain"]),
    }


def technology_level_row(
    key: str,
    info: dict[str, Any],
    level: int,
    base_time: float,
    time_divisor: float,
) -> dict[str, Any]:
    """One level-table row of research `key`: cost, research time and cumulative effect at `level`."""
    cost = full_cost(((info.get("levels") or {}).get(level) or {}).get("cost"))
    research_time = base_time / time_divisor
    bonus_pct = float(TECHNOLOGY_METADATA[key]["per_level_bonus_pct"])
    return {
        **technology_identity(key, info),
        "level": level,
        "pig_iron": int(cost["pig_iron"]),
        "metal": int(cost["metal"]),
        "kryptonite": int(cost["kryptonite"]),
        "spice": int(cost["spice"]),
        "cost_spice_eq": round(float(game_data.spice_equivalent(cost)), 2),
        "points_gain": round(cost_points(cost), 3),
        "base_time_s": round(base_time, 2),
        "effective_research_time_s": round(research_time, 2),
        "effective_research_time_text": fmt_duration(research_time),
        "bonus_pct_total": round(bonus_pct * float(level), 2),
        "multiplier_at_level": round(technology_multiplier(key, level), 4),
        "effect_at_level": technology_effect_at_level(key, level),
    }


def build_technology_tables(
    context: dict[str, Any],
    research_levels: dict[str, int],
    max_level: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build the research catalog (one row per research) and the level table (one row per research level)."""
    time_divisor = research_time_divisor(context["b8_research_lab"], context["speedup_factor"])
    catalog_rows: list[dict[str, Any]] = []
    level_rows: list[dict[str, Any]] = []
    for key, info in research_catalog():
        meta = TECHNOLOGY_METADATA[key]
        levels = info.get("levels") or {}
        table_levels = int(info.get("max_table_level") or len(levels))
        base_times = build_monotonic_base_times(levels, table_levels)
        level_1_cost = full_cost((levels.get(1) or {}).get("cost") or info.get("base_cost"))
        level_1_time = float(base_times.get(1, 0.0))
        catalog_rows.append(
            {
                **technology_identity(key, info),
                "current_level": research_levels.get(key, 0),
                "max_table_level": table_levels,
                "cost_scale_factor": TECHNOLOGY_COST_SCALE_FACTOR,
                "l1_pig_iron": int(level_1_cost["pig_iron"]),
                "l1_metal": int(level_1_cost["metal"]),
                "l1_kryptonite": int(level_1_cost["kryptonite"]),
                "l1_spice": int(level_1_cost["spice"]),
                "l1_cost_spice_eq": round(float(game_data.spice_equivalent(level_1_cost)), 2),
                "l1_base_time_s": round(level_1_time, 2),
                "l1_effective_time_s": round(level_1_time / time_divisor, 2),
                "effect_metric": str(meta["effect_metric"]),
                "per_level_bonus_pct": float(meta["per_level_bonus_pct"]),
                "applies_to": str(meta["applies_to"]),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
                "strategic_effect": str(meta["strategic_effect"]),
            }
        )
        level_rows.extend(
            technology_level_row(key, info, level, float(base_times.get(level, 0.0)), time_divisor)
            for level in range(1, min(table_levels, max_level) + 1)
        )
    return pd.DataFrame(catalog_rows), pd.DataFrame(level_rows)


def build_planet_technology_table(contexts: list[dict[str, Any]], research_levels: dict[str, int]) -> pd.DataFrame:
    """One row per planet and research: the next research level, timed with that planet's Research lab.

    Research levels belong to the account, so `current_level` is the same on every planet; only the research
    time differs. A research already at the last level of the catalog has no row.
    """
    rows: list[dict[str, Any]] = []
    for context in contexts:
        time_divisor = research_time_divisor(context["b8_research_lab"], context["speedup_factor"])
        for key, info in research_catalog():
            levels = info.get("levels") or {}
            current_level = max(0, int(research_levels.get(key, 0) or 0))
            next_level = current_level + 1
            if next_level > int(info.get("max_table_level") or len(levels)):
                continue
            base_time = float(build_monotonic_base_times(levels, next_level).get(next_level, 0.0))
            rows.append(
                next_level_row(
                    context,
                    current_level,
                    technology_level_row(key, info, next_level, base_time, time_divisor),
                    {"research_lab_level": context["b8_research_lab"]},
                )
            )
    return pd.DataFrame(rows)


technologies_catalog_df, technologies_levels_df = build_technology_tables(
    GENERIC_PLANET_CONTEXT, colony_researches, MAX_RESEARCH_TABLE_LEVEL
)
planet_technologies_df = build_planet_technology_table(PLANET_PARAM_CONTEXTS, colony_researches)

print(
    f"technologies_catalog_df: {technologies_catalog_df.shape}, "
    f"technologies_levels_df: {technologies_levels_df.shape} (generic planet), "
    f"planet_technologies_df: {planet_technologies_df.shape} (next level on each planet)\n"
)
print(
    technologies_catalog_df[
        [
            "key",
            "name",
            "domain",
            "current_level",
            "l1_cost_spice_eq",
            "l1_base_time_s",
            "per_level_bonus_pct",
            "prerequisites",
            "strategic_effect",
        ]
    ].to_string(index=False)
)
print("\nNext level of every research on the first planet of the roster:")
print(
    planet_technologies_df[planet_technologies_df["slot_index"] == 1][
        [
            "coords",
            "research_lab_level",
            "key",
            "name",
            "current_level",
            "next_level",
            "cost_spice_eq",
            "effective_research_time_text",
            "effect_at_level",
        ]
    ].to_string(index=False)
)

technologies_catalog_df: (15, 20), technologies_levels_df: (445, 18) (generic planet), planet_technologies_df: (165, 24) (next level on each planet)

key                name           domain  current_level  l1_cost_spice_eq  l1_base_time_s  per_level_bonus_pct                                       prerequisites                                                                   strategic_effect
 r1   Mining technology          economy              9            333.33          2083.0                  2.0     r2 L4 (Energy technology), b8 L2 (Research lab)                            +2%/lvl global Pig iron, Kryptonite & Spice mine output
 r2   Energy technology          economy              4            826.67          1833.0                  1.0                                b8 L1 (Research lab)                  +1%/lvl global energy output; gates b9, b6, r1, shields & turrets
 r3     Engine research       propulsion              6            570.00          1000.0                  0.0  

### 6.3 Spaceships (`s1`–`s16`)

`ships_df` has one row per ship:

- unit cost per resource, `cost_spice_eq`, and `points_value` (cost / 1,000);
- `base_build_time_s`, and `effective_build_time_s` and `units_per_hour_effective` on a generic planet;
- combat: `hull_base`, `shield_base` and `attack_base`, and the `*_effective` values with +10 % per level of Defense systems (`r10`), Shield technology (`r9`) and Weapons technology (`r8`). SpaceInvasion Classic has no rapid fire: every unit fires once per combat round;
- propulsion: `engine`, the research that speeds it up (`engine_tech_key`), `speed_base`, and `speed_effective` with +10 % per level of that research;
- logistics: `cargo`, `fuel`, and the ratios `cargo_per_spice_eq`, `attack_per_spice_eq` and `hull_per_spice_eq`;
- `energy_output_base` and `energy_output_effective`: only Solar cells (`s9`) produce energy. Their output depends on the planet's temperature, and they use no building field.

`planet_ships_df` has one row per planet and ship: the ships `stationed` there, the build time and hourly throughput in that planet's Weapons factory, and the Solar cell energy at that planet's temperature. A planet without a Weapons factory is timed as if it had level 1.

`combat_multipliers` and `planet_unit_rows`, defined here, are used again for the defenses in 6.4.

In [66]:
def combat_multipliers(context: dict[str, Any]) -> dict[str, float]:
    """Attack, shield and hull multipliers from the Weapons (`r8`), Shield (`r9`) and Defense systems (`r10`) levels."""
    return {
        "attack": technology_multiplier("r8", context["r8_weapons_tech"]),
        "shield": technology_multiplier("r9", context["r9_shield_tech"]),
        "hull": technology_multiplier("r10", context["r10_defense_systems"]),
    }


def build_ship_table(context: dict[str, Any]) -> pd.DataFrame:
    """Build `ships_df`: cost, build time, combat stats, propulsion, cargo and orbital energy of every ship."""
    time_divisor = shipyard_time_divisor(
        context["b9_weapons_factory"], context["b17_microsystem"], context["speedup_factor"]
    )
    multiplier = combat_multipliers(context)
    energy_multiplier = technology_multiplier("r2", context["r2_energy_tech"])
    temperature = float(context["planet_temperature_c"])
    # A ship's `engine` stat is the name of the research that speeds it up (Rocket, Diffusion or Warp engine).
    engine_key_by_name = {str(info["name"]): key for key, info in research_catalog()}
    engine_levels = {
        "r4": context["r4_rocket_engine"],
        "r5": context["r5_diffusion_engine"],
        "r6": context["r6_warp_engine"],
    }

    rows: list[dict[str, Any]] = []
    for key in unit_keys(SHIP_SLUG_BY_KEY):
        slug = SHIP_SLUG_BY_KEY[key]
        info = game_data.SHIPS[slug]
        stats = info.get("stats") or {}
        cost = full_cost(info.get("base_cost"))
        cost_spice_eq = float(game_data.spice_equivalent(cost))
        base_time = float(info.get("base_build_time_s") or 0.0)
        build_time = base_time / time_divisor

        hull, shield, attack = (int(stats.get(stat) or 0) for stat in ("hull", "shield", "attack"))
        speed = int(stats.get("speed") or 0)
        cargo = int(stats.get("cargo") or 0)
        engine = str(stats.get("engine") or ("Stationary" if speed == 0 else "Unknown"))
        engine_key = engine_key_by_name.get(engine, "none")
        engine_level = int(engine_levels.get(engine_key, 0))
        # Only Solar cells (`s9`) produce energy; they orbit the planet and use no building field.
        energy_base = solar_cell_energy_per_unit(temperature) if key == "s9" else 0.0

        rows.append(
            {
                "ship_id": int(key[1:]),
                "key": key,
                "slug": slug,
                "name": str(info["name"]),
                "role": str(SHIP_METADATA[key]["role"]),
                "pig_iron": int(cost["pig_iron"]),
                "metal": int(cost["metal"]),
                "kryptonite": int(cost["kryptonite"]),
                "spice": int(cost["spice"]),
                "cost_spice_eq": round(cost_spice_eq, 2),
                "points_value": round(cost_points(cost), 3),
                "base_build_time_s": round(base_time, 2),
                "effective_build_time_s": round(build_time, 2),
                "effective_build_time_text": fmt_duration(build_time),
                "units_per_hour_effective": round((3600.0 / build_time) if build_time > 0.0 else 0.0, 2),
                "hull_base": hull,
                "shield_base": shield,
                "attack_base": attack,
                "hull_effective": round(float(hull) * multiplier["hull"], 1),
                "shield_effective": round(float(shield) * multiplier["shield"], 1),
                "attack_effective": round(float(attack) * multiplier["attack"], 1),
                "engine": engine,
                "engine_tech_key": engine_key,
                "speed_base": speed,
                "speed_effective": round(float(speed) * technology_multiplier(engine_key, engine_level), 1),
                "cargo": cargo,
                "fuel": int(stats.get("fuel") or 0),
                "energy_output_base": round(energy_base, 2),
                "energy_output_effective": round(energy_base * energy_multiplier, 2),
                "fields_used": 0,
                "cargo_per_spice_eq": round(float(cargo) / cost_spice_eq, 3) if cost_spice_eq > 0.0 else 0.0,
                "attack_per_spice_eq": (
                    round((float(attack) * multiplier["attack"]) / cost_spice_eq, 4) if cost_spice_eq > 0.0 else 0.0
                ),
                "hull_per_spice_eq": (
                    round((float(hull) * multiplier["hull"]) / cost_spice_eq, 2) if cost_spice_eq > 0.0 else 0.0
                ),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
            }
        )
    return pd.DataFrame(rows)


SHIP_PLANET_COLUMNS: tuple[str, ...] = (
    "effective_build_time_s",
    "effective_build_time_text",
    "units_per_hour_effective",
    "energy_output_base",
    "energy_output_effective",
)


def planet_unit_rows(
    context: dict[str, Any],
    stationed: dict[str, int],
    unit_table: pd.DataFrame,
    planet_columns: tuple[str, ...],
) -> list[dict[str, Any]]:
    """Rows of a per-planet unit table: the units stationed on the planet and the unit columns that depend on it."""
    identity = planet_identity(context)
    return [
        {
            **identity,
            "key": unit["key"],
            "name": unit["name"],
            "stationed": max(0, int(stationed.get(unit["key"], 0) or 0)),
            **{column: unit[column] for column in planet_columns},
        }
        for unit in unit_table.to_dict("records")
    ]


def build_planet_ship_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and ship: ships stationed there, build time in its Weapons factory, Solar cell output."""
    return pd.DataFrame(
        [
            row
            for planet, context in zip(planets, contexts)
            for row in planet_unit_rows(context, planet.ships, build_ship_table(context), SHIP_PLANET_COLUMNS)
        ]
    )


ships_df = build_ship_table(GENERIC_PLANET_CONTEXT)
planet_ships_df = build_planet_ship_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

print(f"ships_df: {ships_df.shape} (generic planet), planet_ships_df: {planet_ships_df.shape} (each planet)\n")
print(
    ships_df[
        [
            "key",
            "name",
            "role",
            "cost_spice_eq",
            "base_build_time_s",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "speed_effective",
            "cargo",
            "fuel",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nBuild time per planet (seconds), one column per ship:")
print(
    planet_ships_df.pivot(index=["slot_index", "coords"], columns="key", values="effective_build_time_s")[
        list(ships_df["key"])
    ].to_string()
)

ships_df: (15, 34) (generic planet), planet_ships_df: (165, 12) (each planet)

key              name                    role  cost_spice_eq  base_build_time_s  attack_effective  shield_effective  hull_effective  speed_effective   cargo  fuel                                                                prerequisites
 s1   Small transport         cargo_transport        2200.00            1700.00               2.0              10.0          6000.0           5400.0    6000    10                               b9 L2 (Weapons factory), r4 L4 (Rocket engine)
 s2   Large transport         cargo_transport        5200.00            4400.00               5.0              25.0         14000.0           9000.0   30000    15                               b9 L6 (Weapons factory), r4 L6 (Rocket engine)
 s3       Transmitter      armed_heavy_hauler       18333.33           14166.67             130.0              70.0         50000.0          11900.0  250000    25                            b9 L8 (Weap

### 6.4 Planetary Defenses (`d1`–`d11`) & Table Registry

`defenses_df` has one row per defense: unit cost, the base build time and the build time on a generic planet, base and effective hull, shield and attack (the same multipliers as the ships), whether the unit can dig in (`can_dig_in`, from `defenses.json`), and the ratios `attack_per_spice_eq` and `hull_plus_shield_per_spice_eq`.

`planet_defenses_df` has one row per planet and defense: the units `stationed` there, and the build time and hourly throughput in that planet's Weapons factory.

`GAME_PARAMS_TABLES` registers the twelve tables of Section 6 by name, and `game_params_summary_df` lists their shapes and first columns.

In [67]:
def build_defense_table(context: dict[str, Any]) -> pd.DataFrame:
    """Build `defenses_df`: cost, build time and combat stats of every planetary defense."""
    time_divisor = shipyard_time_divisor(
        context["b9_weapons_factory"], context["b17_microsystem"], context["speedup_factor"]
    )
    multiplier = combat_multipliers(context)

    rows: list[dict[str, Any]] = []
    for key in unit_keys(DEFENSE_SLUG_BY_KEY):
        slug = DEFENSE_SLUG_BY_KEY[key]
        info = game_data.DEFENSES[slug]
        stats = info.get("stats") or {}
        cost = full_cost(info.get("base_cost"))
        cost_spice_eq = float(game_data.spice_equivalent(cost))
        base_time = float(info.get("base_build_time_s") or 0.0)
        build_time = base_time / time_divisor

        hull, shield, attack = (int(stats.get(stat) or 0) for stat in ("hull", "shield", "attack"))
        hull_effective = float(hull) * multiplier["hull"]
        shield_effective = float(shield) * multiplier["shield"]
        attack_effective = float(attack) * multiplier["attack"]

        rows.append(
            {
                "defense_id": int(key[1:]),
                "key": key,
                "slug": slug,
                "name": str(info["name"]),
                "role": str(DEFENSE_METADATA[key]["role"]),
                "can_dig_in": bool(DEFENSE_METADATA[key]["can_dig_in"]),
                "pig_iron": int(cost["pig_iron"]),
                "metal": int(cost["metal"]),
                "kryptonite": int(cost["kryptonite"]),
                "spice": int(cost["spice"]),
                "cost_spice_eq": round(cost_spice_eq, 2),
                "points_value": round(cost_points(cost), 3),
                "base_build_time_s": round(base_time, 2),
                "effective_build_time_s": round(build_time, 2),
                "effective_build_time_text": fmt_duration(build_time),
                "units_per_hour_effective": round((3600.0 / build_time) if build_time > 0.0 else 0.0, 2),
                "hull_base": hull,
                "shield_base": shield,
                "attack_base": attack,
                "hull_effective": round(hull_effective, 1),
                "shield_effective": round(shield_effective, 1),
                "attack_effective": round(attack_effective, 1),
                "attack_per_spice_eq": round(attack_effective / cost_spice_eq, 4) if cost_spice_eq > 0.0 else 0.0,
                "hull_plus_shield_per_spice_eq": (
                    round((hull_effective + shield_effective) / cost_spice_eq, 2) if cost_spice_eq > 0.0 else 0.0
                ),
                "prerequisites": format_prerequisites(info.get("requirements") or []),
            }
        )
    return pd.DataFrame(rows)


DEFENSE_PLANET_COLUMNS: tuple[str, ...] = (
    "effective_build_time_s",
    "effective_build_time_text",
    "units_per_hour_effective",
)


def build_planet_defense_table(planets: list[PlanetState], contexts: list[dict[str, Any]]) -> pd.DataFrame:
    """One row per planet and defense: units stationed there and the build time in its Weapons factory."""
    return pd.DataFrame(
        [
            row
            for planet, context in zip(planets, contexts)
            for row in planet_unit_rows(
                context, planet.defense, build_defense_table(context), DEFENSE_PLANET_COLUMNS
            )
        ]
    )


defenses_df = build_defense_table(GENERIC_PLANET_CONTEXT)
planet_defenses_df = build_planet_defense_table(empire_planet_states, PLANET_PARAM_CONTEXTS)

GAME_PARAMS_TABLES: dict[str, pd.DataFrame] = {
    "merchant_rates": merchant_rates_df,
    "planet_contexts": planet_contexts_df,
    "buildings_catalog": buildings_catalog_df,
    "buildings_levels": buildings_levels_df,
    "planet_buildings": planet_buildings_df,
    "technologies_catalog": technologies_catalog_df,
    "technologies_levels": technologies_levels_df,
    "planet_technologies": planet_technologies_df,
    "ships": ships_df,
    "planet_ships": planet_ships_df,
    "defenses": defenses_df,
    "planet_defenses": planet_defenses_df,
}
game_params_summary_df = pd.DataFrame(
    [
        {
            "table_name": name,
            "dataframe_var": f"{name}_df",
            "rows": int(table.shape[0]),
            "columns": int(table.shape[1]),
            "sample_columns": ", ".join(list(table.columns[:6])),
        }
        for name, table in GAME_PARAMS_TABLES.items()
    ]
)

print(
    f"defenses_df: {defenses_df.shape} (generic planet), "
    f"planet_defenses_df: {planet_defenses_df.shape} (each planet)\n"
)
print(
    defenses_df[
        [
            "key",
            "name",
            "role",
            "can_dig_in",
            "cost_spice_eq",
            "base_build_time_s",
            "attack_effective",
            "shield_effective",
            "hull_effective",
            "attack_per_spice_eq",
            "prerequisites",
        ]
    ].to_string(index=False)
)
print("\nBuild time per planet (seconds), one column per defense:")
print(
    planet_defenses_df.pivot(index=["slot_index", "coords"], columns="key", values="effective_build_time_s")[
        list(defenses_df["key"])
    ].to_string()
)
print("\nGAME_PARAMS_TABLES:")
print(game_params_summary_df.to_string(index=False))

defenses_df: (11, 25) (generic planet), planet_defenses_df: (121, 10) (each planet)

key                  name             role  can_dig_in  cost_spice_eq  base_build_time_s  attack_effective  shield_effective  hull_effective  attack_per_spice_eq                                                                     prerequisites
 d1                Mortar planetary_turret        True         800.00            1000.00              35.0              15.0          2000.0               0.0437                                                           b9 L1 (Weapons factory)
 d2  Laser cannon battery planetary_turret        True       15166.67           11333.33            1000.0             400.0         40000.0               0.0659   b9 L8 (Weapons factory), r11 L10 (Targeting systems), r2 L8 (Energy technology)
 d3            Ion cannon planetary_turret        True        3466.67            1533.33             125.0             125.0         10000.0               0.0361                      

## 7. Technology Requirement Graph

A building, research, ship or defense can require other buildings and researches at given levels. This section keeps those requirements in memory as a directed acyclic graph, so questions such as "is this unlocked?" or "what has to be built first?" are answered without reading the catalog again.

### 7.1 `TechnologyRequirementGraph`

A node is an entity key (`b7`, `r2`, `s7`, `d10`). An edge `source -> target` with a level means that the target needs the source at that level. For example, the Weapons factory `b9` needs the Development center `b7` at level 3 and Energy technology `r2` at level 1.

The graph is built once from `game_data` and exposes:

- `nodes` — category, slug, name and highest catalog level of each entity;
- `prereqs_of[key]` and `unlocks_of[key]` — the direct requirements of an entity, and the entities that directly require it, as `(key, level)` pairs;
- `topological_order` and `topological_layer[key]` — an order in which every entity comes after its prerequisites, and each entity's depth in the tree. Kahn's algorithm produces both and fails if the requirements contain a cycle;
- `ancestors_of[key]` and `descendants_of[key]` — every direct and indirect prerequisite, and everything the entity helps to unlock.

The queries take the current building and research levels as arguments:

- `is_unlocked` and `direct_missing` — whether the direct requirements are met, and which ones are not;
- `transitive_requirements` — the minimum level of every direct and indirect prerequisite;
- `plan_unlock_sequence` — the ordered upgrades that unlock a target, with their cost and time.

In [68]:
class TechnologyRequirementGraph:
    """In-memory directed acyclic graph of the prerequisites of every building, research, ship and defense.

    Nodes are entity keys (`b7`, `r2`, `s7`, `d10`). An edge `source -> target` with a level means that the
    target needs the source at that level. Ships and defenses have no levels of their own.
    """

    PLAN_COLUMNS: tuple[str, ...] = (
        "step",
        "key",
        "name",
        "category",
        "topological_layer",
        "from_level",
        "to_level",
        "levels_added",
        "pig_iron",
        "metal",
        "kryptonite",
        "spice",
        "step_cost_spice_eq",
        "step_effective_time_s",
        "step_effective_time_text",
        "cum_cost_spice_eq",
        "cum_effective_time_s",
        "cum_effective_time_text",
    )

    def __init__(self) -> None:
        self.nodes: dict[str, dict[str, Any]] = {}
        self._requirements: dict[str, list[dict[str, Any]]] = {}
        self._base_times: dict[str, dict[int, float]] = {}
        for building_id in sorted(game_data.BUILDING_BY_ID):
            self._add_node(f"b{building_id}", "buildings", game_data.BUILDING_BY_ID[building_id])
        for key, info in research_catalog():
            self._add_node(key, "research", info)
        for key in unit_keys(SHIP_SLUG_BY_KEY):
            self._add_node(key, "ships", game_data.SHIPS[SHIP_SLUG_BY_KEY[key]])
        for key in unit_keys(DEFENSE_SLUG_BY_KEY):
            self._add_node(key, "defense", game_data.DEFENSES[DEFENSE_SLUG_BY_KEY[key]])

        self.prereqs_of, self.unlocks_of = self._link_requirements()
        self.topological_order, self.topological_layer = self._sort_topologically()
        self._topological_rank = {key: rank for rank, key in enumerate(self.topological_order)}
        self.ancestors_of, self.descendants_of = self._close_transitively()

    def __repr__(self) -> str:
        edges = sum(len(requirements) for requirements in self.prereqs_of.values())
        return f"TechnologyRequirementGraph(nodes={len(self.nodes)}, edges={edges})"

    def _add_node(self, key: str, category: str, info: dict[str, Any]) -> None:
        """Register one catalog entity with its requirements and its repaired base times."""
        levels = info.get("levels") or {}
        max_level = int(info.get("max_table_level") or len(levels))
        self.nodes[key] = {
            "key": key,
            "category": category,
            "entity_id": int(key[1:]),
            "slug": str(info["slug"]),
            "name": str(info["name"]),
            "max_level": max_level,
        }
        self._requirements[key] = list(info.get("requirements") or [])
        self._base_times[key] = build_monotonic_base_times(levels, max_level)

    def _link_requirements(
        self,
    ) -> tuple[dict[str, tuple[tuple[str, int], ...]], dict[str, tuple[tuple[str, int], ...]]]:
        """Turn each catalog requirement into an edge, indexed by target (`prereqs_of`) and source (`unlocks_of`)."""
        key_by_slug = {(node["category"], node["slug"]): key for key, node in self.nodes.items()}
        prereqs: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        unlocks: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        for target, requirements in self._requirements.items():
            for requirement in requirements:
                source = key_by_slug.get((str(requirement["category"]), str(requirement["slug"])))
                if source is None:
                    raise KeyError(f"Unresolved tech-tree requirement {requirement!r} on {target}")
                level = int(requirement["level"])
                prereqs[target].append((source, level))
                unlocks[source].append((target, level))
        return (
            {key: tuple(edges) for key, edges in prereqs.items()},
            {key: tuple(edges) for key, edges in unlocks.items()},
        )

    def _sort_topologically(self) -> tuple[tuple[str, ...], dict[str, int]]:
        """Kahn's algorithm: order the nodes so prerequisites come first, and record each node's dependency depth."""
        open_prereqs = {key: len(self.prereqs_of[key]) for key in self.nodes}
        layer = {key: 0 for key in self.nodes}
        queue: deque[str] = deque(key for key, count in open_prereqs.items() if count == 0)
        order: list[str] = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent, _level in self.unlocks_of[node]:
                layer[dependent] = max(layer[dependent], layer[node] + 1)
                open_prereqs[dependent] -= 1
                if open_prereqs[dependent] == 0:
                    queue.append(dependent)
        if len(order) != len(self.nodes):
            raise ValueError(f"Cycle detected in technology requirement graph: {sorted(set(self.nodes) - set(order))}")
        return tuple(order), layer

    def _close_transitively(self) -> tuple[dict[str, frozenset[str]], dict[str, frozenset[str]]]:
        """Collect each node's direct and indirect prerequisites (`ancestors_of`) and dependents (`descendants_of`)."""
        ancestors: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in self.topological_order:
            for prerequisite, _level in self.prereqs_of[node]:
                ancestors[node].add(prerequisite)
                ancestors[node].update(ancestors[prerequisite])
        descendants: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in reversed(self.topological_order):
            for dependent, _level in self.unlocks_of[node]:
                descendants[node].add(dependent)
                descendants[node].update(descendants[dependent])
        return (
            {key: frozenset(found) for key, found in ancestors.items()},
            {key: frozenset(found) for key, found in descendants.items()},
        )

    def current_level(self, key: str, buildings: dict[str, int], researches: dict[str, int]) -> int:
        """Level of `key` in the given state (ships and defenses have no level, so they return 0)."""
        category = self.nodes[key]["category"]
        if category == "buildings":
            return max(0, int(buildings.get(key, 0) or 0))
        if category == "research":
            return max(0, int(researches.get(key, 0) or 0))
        return 0

    def direct_missing(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> list[dict[str, Any]]:
        """Direct prerequisites of `target_key` that the given state does not meet yet."""
        missing: list[dict[str, Any]] = []
        for source, required_level in self.prereqs_of[target_key]:
            level = self.current_level(source, buildings, researches)
            if level < required_level:
                missing.append(
                    {
                        "key": source,
                        "name": self.nodes[source]["name"],
                        "category": self.nodes[source]["category"],
                        "required_level": required_level,
                        "current_level": level,
                        "missing_levels": required_level - level,
                    }
                )
        return missing

    def is_unlocked(self, target_key: str, buildings: dict[str, int], researches: dict[str, int]) -> bool:
        """True when the given state meets every direct prerequisite of `target_key`."""
        return not self.direct_missing(target_key, buildings, researches)

    def transitive_requirements(self, target_key: str, *, include_target_level: int = 0) -> dict[str, int]:
        """Minimum level of every direct and indirect prerequisite of `target_key`, in topological order.

        When several requirement paths reach the same entity, the highest required level wins.
        `include_target_level` also demands that level of the target itself (buildings and researches only).
        """
        if target_key not in self.nodes:
            raise KeyError(f"Unknown node key: {target_key!r}")
        required: dict[str, int] = {}
        if include_target_level > 0 and self.nodes[target_key]["category"] in ("buildings", "research"):
            required[target_key] = int(include_target_level)
        for source, level in self.prereqs_of[target_key]:
            required[source] = max(required.get(source, 0), int(level))
        # Visiting nodes from the last in topological order to the first sees every dependent before its prerequisites.
        for node in reversed(self.topological_order):
            if node not in required:
                continue
            for source, level in self.prereqs_of[node]:
                if level > required.get(source, 0):
                    required[source] = int(level)
        return {key: required[key] for key in sorted(required, key=self._topological_rank.__getitem__)}

    def _level_cost_and_base_time(self, key: str, level: int) -> tuple[dict[str, Any], float]:
        """Cost and base time of taking building or research `key` to `level`."""
        node = self.nodes[key]
        base_time = float(self._base_times[key].get(level, 0.0))
        if node["category"] == "buildings":
            return game_data.building_cost(int(node["entity_id"]), level), base_time
        levels = game_data.RESEARCHES[node["slug"]].get("levels") or {}
        return full_cost((levels.get(level) or {}).get("cost")), base_time

    def plan_unlock_sequence(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
        *,
        speedup_factor: float,
        include_target_level: int = 0,
    ) -> pd.DataFrame:
        """Ordered upgrades that take the given state to the point where `target_key` is unlocked.

        One row per building or research that must be raised, in an order that respects every prerequisite.
        Times use the simulated state, so a Development center, Research lab or Microsystem accelerator
        raised by an earlier step shortens the later ones.
        """
        simulated_buildings = {str(key): max(0, int(level or 0)) for key, level in buildings.items()}
        simulated_researches = {str(key): max(0, int(level or 0)) for key, level in researches.items()}
        speed = max(0.1, float(speedup_factor))

        rows: list[dict[str, Any]] = []
        total_spice_eq = 0.0
        total_time = 0.0
        requirements = self.transitive_requirements(target_key, include_target_level=include_target_level)
        for key, required_level in requirements.items():
            node = self.nodes[key]
            current = self.current_level(key, simulated_buildings, simulated_researches)
            if current >= required_level:
                continue
            step_cost = {resource: 0 for resource in RESOURCE_KEYS}
            step_time = 0.0
            for level in range(current + 1, required_level + 1):
                cost, base_time = self._level_cost_and_base_time(key, level)
                for resource in RESOURCE_KEYS:
                    step_cost[resource] += int(cost.get(resource, 0))
                if node["category"] == "buildings":
                    divisor = building_time_divisor(
                        simulated_buildings.get("b7", 0), simulated_buildings.get("b17", 0), speed
                    )
                    simulated_buildings[key] = level
                else:
                    divisor = research_time_divisor(simulated_buildings.get("b8", 0), speed)
                    simulated_researches[key] = level
                step_time += base_time / divisor

            step_spice_eq = float(game_data.spice_equivalent(step_cost))
            total_spice_eq += step_spice_eq
            total_time += step_time
            rows.append(
                {
                    "step": len(rows) + 1,
                    "key": key,
                    "name": node["name"],
                    "category": node["category"],
                    "topological_layer": self.topological_layer[key],
                    "from_level": current,
                    "to_level": required_level,
                    "levels_added": required_level - current,
                    "pig_iron": step_cost["pig_iron"],
                    "metal": step_cost["metal"],
                    "kryptonite": step_cost["kryptonite"],
                    "spice": step_cost["spice"],
                    "step_cost_spice_eq": round(step_spice_eq, 2),
                    "step_effective_time_s": round(step_time, 2),
                    "step_effective_time_text": fmt_duration(step_time),
                    "cum_cost_spice_eq": round(total_spice_eq, 2),
                    "cum_effective_time_s": round(total_time, 2),
                    "cum_effective_time_text": fmt_duration(total_time),
                }
            )
        return pd.DataFrame(rows, columns=list(self.PLAN_COLUMNS))

### 7.2 Graph Tables

Builds the graph (`TECH_GRAPH`) and two tables that describe it for the reference colony:

- `tech_graph_nodes_df` — one row per entity in topological order: layer, number of direct requirements (`in_degree`) and of direct dependents (`out_degree`), number of indirect ones, current level, and whether it is unlocked now;
- `tech_graph_edges_df` — one row per requirement: source, required level, target, the source's current level, whether the requirement is met, and the levels still missing.

In [69]:
TECH_GRAPH = TechnologyRequirementGraph()

tech_graph_nodes_df = pd.DataFrame(
    [
        {
            "key": key,
            "category": TECH_GRAPH.nodes[key]["category"],
            "slug": TECH_GRAPH.nodes[key]["slug"],
            "name": TECH_GRAPH.nodes[key]["name"],
            "topological_layer": TECH_GRAPH.topological_layer[key],
            "in_degree": len(TECH_GRAPH.prereqs_of[key]),
            "out_degree": len(TECH_GRAPH.unlocks_of[key]),
            "transitive_prereq_count": len(TECH_GRAPH.ancestors_of[key]),
            "transitive_unlock_count": len(TECH_GRAPH.descendants_of[key]),
            "is_root_node": len(TECH_GRAPH.prereqs_of[key]) == 0,
            "current_level": TECH_GRAPH.current_level(key, colony_buildings, colony_researches),
            "unlocked_now": TECH_GRAPH.is_unlocked(key, colony_buildings, colony_researches),
        }
        for key in TECH_GRAPH.topological_order
    ]
)

tech_graph_edges_df = pd.DataFrame(
    [
        {
            "source_key": source,
            "source_name": TECH_GRAPH.nodes[source]["name"],
            "source_category": TECH_GRAPH.nodes[source]["category"],
            "required_level": required_level,
            "target_key": target,
            "target_name": TECH_GRAPH.nodes[target]["name"],
            "target_category": TECH_GRAPH.nodes[target]["category"],
            "target_layer": TECH_GRAPH.topological_layer[target],
            "source_current_level": TECH_GRAPH.current_level(source, colony_buildings, colony_researches),
            "satisfied_now": TECH_GRAPH.current_level(source, colony_buildings, colony_researches) >= required_level,
            "levels_missing": max(
                0, required_level - TECH_GRAPH.current_level(source, colony_buildings, colony_researches)
            ),
        }
        for target in TECH_GRAPH.topological_order
        for source, required_level in TECH_GRAPH.prereqs_of[target]
    ]
)

print(
    f"TECH_GRAPH: {len(TECH_GRAPH.nodes)} nodes, {len(tech_graph_edges_df)} requirement edges, "
    f"deepest layer {max(TECH_GRAPH.topological_layer.values())}\n"
)
print("Entities that unlock the most others:")
print(
    tech_graph_nodes_df.sort_values(["transitive_unlock_count", "out_degree"], ascending=[False, False])
    .head(12)[
        [
            "key",
            "name",
            "category",
            "topological_layer",
            "in_degree",
            "out_degree",
            "transitive_unlock_count",
            "current_level",
            "unlocked_now",
        ]
    ]
    .to_string(index=False)
)

TECH_GRAPH: 62 nodes, 101 requirement edges, deepest layer 5

Entities that unlock the most others:
key                name  category  topological_layer  in_degree  out_degree  transitive_unlock_count  current_level  unlocked_now
 b7  Development center buildings                  0          0           4                       51              9          True
 b8        Research lab buildings                  1          1          12                       49              3          True
 r2   Energy technology  research                  2          1          13                       40              4          True
 b9     Weapons factory buildings                  3          2          28                       28              4          True
 r3     Engine research  research                  2          1           3                       18              6          True
 r9   Shield technology  research                  3          2           4                        7              0     

### 7.3 Unlock Plans

`plan_unlock_sequence` lists the upgrades still missing before a target is unlocked, in an order that respects every prerequisite, with the cost and time of each step.

- When several requirement paths reach the same entity, the highest level wins and is counted once. The Phoenix (`s7`) needs the Weapons factory `b9` at level 7, which needs the Development center `b7` at level 3. It also needs Defense systems `r10` at level 5, which needs the Research lab `b8` at level 5, which needs `b7` at level 2. The plan raises `b7` to level 3, once.
- Times follow the simulated state: after a step raises the Development center, the Research lab or the Microsystem accelerator, the later steps are faster.

Tables:

- `unlock_targets_summary_df` — for every entity: whether it is unlocked, the upgrades and levels still missing, their total cost and time, and the ordered chain of upgrades;
- `phoenix_unlock_plan_df` — the plan for the Phoenix from the reference colony;
- `cold_phoenix_unlock_plan_df` — the same plan from an empty account;
- `TECH_GRAPH_TABLES` — registry of the five tables of Section 7.

In [70]:
def plan_unlock_from_colony(target_key: str) -> pd.DataFrame:
    """Unlock plan for `target_key` starting from the reference colony's current buildings and researches."""
    return TECH_GRAPH.plan_unlock_sequence(
        target_key,
        colony_buildings,
        colony_researches,
        speedup_factor=GAME_PARAM_CONTEXT["speedup_factor"],
    )


def unlock_summary_row(target_key: str) -> dict[str, Any]:
    """One `unlock_targets_summary_df` row: what is still missing before `target_key` is unlocked."""
    node = TECH_GRAPH.nodes[target_key]
    plan = plan_unlock_from_colony(target_key)
    total_time = float(plan["step_effective_time_s"].sum()) if not plan.empty else 0.0
    return {
        "key": target_key,
        "name": node["name"],
        "category": node["category"],
        "topological_layer": TECH_GRAPH.topological_layer[target_key],
        "unlocked_now": bool(plan.empty),
        "missing_upgrade_nodes": int(len(plan)),
        "total_levels_to_add": int(plan["levels_added"].sum()) if not plan.empty else 0,
        "unlock_cost_spice_eq": round(float(plan["step_cost_spice_eq"].sum()) if not plan.empty else 0.0, 2),
        "unlock_effective_time_s": round(total_time, 2),
        "unlock_effective_time_text": fmt_duration(total_time) if total_time > 0.0 else "Ready",
        "ordered_prerequisite_chain": (
            " -> ".join(
                f"{step['key']} L{int(step['from_level'])}->{int(step['to_level'])}"
                for step in plan.to_dict("records")
            )
            if not plan.empty
            else "Unlocked"
        ),
    }


unlock_targets_summary_df = pd.DataFrame([unlock_summary_row(key) for key in TECH_GRAPH.topological_order])
phoenix_unlock_plan_df = plan_unlock_from_colony("s7")
cold_phoenix_unlock_plan_df = TECH_GRAPH.plan_unlock_sequence(
    "s7", {}, {}, speedup_factor=GAME_PARAM_CONTEXT["speedup_factor"]
)

TECH_GRAPH_TABLES: dict[str, pd.DataFrame] = {
    "tech_graph_nodes": tech_graph_nodes_df,
    "tech_graph_edges": tech_graph_edges_df,
    "unlock_targets_summary": unlock_targets_summary_df,
    "phoenix_unlock_plan": phoenix_unlock_plan_df,
    "cold_phoenix_unlock_plan": cold_phoenix_unlock_plan_df,
}

print(
    f"{int(unlock_targets_summary_df['unlocked_now'].sum())} entities are unlocked, "
    f"{int((~unlock_targets_summary_df['unlocked_now']).sum())} are locked\n"
)
print("Cheapest locked entities to unlock:")
print(
    unlock_targets_summary_df[~unlock_targets_summary_df["unlocked_now"]]
    .sort_values(["unlock_cost_spice_eq", "topological_layer"])
    .head(15)[
        [
            "key",
            "name",
            "category",
            "topological_layer",
            "total_levels_to_add",
            "unlock_cost_spice_eq",
            "unlock_effective_time_text",
            "ordered_prerequisite_chain",
        ]
    ]
    .to_string(index=False)
)
print("\nUnlock plan for the Phoenix (s7) from the reference colony:")
if phoenix_unlock_plan_df.empty:
    print("already unlocked")
else:
    print(
        phoenix_unlock_plan_df[
            [
                "step",
                "key",
                "name",
                "from_level",
                "to_level",
                "step_cost_spice_eq",
                "step_effective_time_text",
                "cum_cost_spice_eq",
                "cum_effective_time_text",
            ]
        ].to_string(index=False)
    )

30 entities are unlocked, 32 are locked

Cheapest locked entities to unlock:
key                  name  category  topological_layer  total_levels_to_add  unlock_cost_spice_eq unlock_effective_time_text                    ordered_prerequisite_chain
 s5               Warship     ships                  4                    2               2500.00                     1m 40s                                      r8 L0->2
 d8 Light graviton cannon   defense                  4                    2               3260.00                     2m 19s                                     r13 L0->2
 r9     Shield technology  research                  3                    1               4960.00                        27s                                      b8 L3->4
d10     Small shield dome   defense                  4                    3               7260.00                     2m 16s                          b8 L3->4 -> r9 L0->2
b20         Tunnel system buildings                  4              

## 8. Planet Sizes & States

The size (used and total fields) and the state (resources, storage, energy, buildings, ships, defenses) of every planet of the roster that Section 6 loaded: the planets owned now, plus a generic planet for every colony that can still be founded.

### 8.1 Owned Planets

`live_planets_df` has one summary row per owned planet of `live_planet_states`: size, resources with their Spice equivalent, storage capacities, energy balance, and the totals of building levels, ships and defenses.

In [71]:
def planet_state_to_row(slot_index: int, planet: PlanetState, *, is_empty_slot: bool) -> dict[str, Any]:
    """One row of the planet tables: size, resources, storage, energy and totals of a planet or empty slot."""
    fields_max = int(planet.fields_max)
    fields_used = int(planet.fields_used)
    resources = {resource: int(getattr(planet.resources, resource)) for resource in RESOURCE_KEYS}
    return {
        "slot_index": slot_index,
        "planet_id": int(planet.planet_id),
        "status": "empty_colony_slot" if is_empty_slot else "colonized",
        "is_empty_slot": bool(is_empty_slot),
        "name": planet.name,
        "coords": planet.coords,
        "planet_type": planet.planet_type,
        "temperature_c": int(planet.temperature),
        "fields_used": fields_used,
        "fields_max": fields_max,
        "fields_remaining": int(planet.fields_remaining),
        "field_utilization_pct": round((100.0 * float(fields_used) / float(fields_max)) if fields_max > 0 else 0.0, 2),
        **resources,
        "resources_spice_eq": round(float(game_data.spice_equivalent(resources)), 2),
        **{f"storage_{resource}": int(getattr(planet.storage, resource)) for resource in RESOURCE_KEYS},
        "energy_produced": int(planet.energy.produced),
        "energy_used": int(planet.energy.used),
        "energy_net": int(planet.energy.produced) - int(planet.energy.used),
        "total_building_levels": int(sum(planet.buildings.values())),
        "total_ships": int(sum(planet.ships.values())),
        "total_defenses": int(sum(planet.defense.values())),
    }


live_planets_df = pd.DataFrame(
    [
        planet_state_to_row(index, planet, is_empty_slot=False)
        for index, planet in enumerate(live_planet_states, start=1)
    ]
)

print(f"{len(live_planet_states)} owned planet(s); the empire holds up to {MAX_EMPIRE_PLANETS}")
if not live_planets_df.empty:
    print(
        live_planets_df[
            [
                "planet_id",
                "name",
                "coords",
                "planet_type",
                "temperature_c",
                "fields_used",
                "fields_max",
                "fields_remaining",
                "field_utilization_pct",
                "resources_spice_eq",
                "energy_net",
                "total_building_levels",
                "total_ships",
            ]
        ].to_string(index=False)
    )

1 owned planet(s); the empire holds up to 11
 planet_id   name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining  field_utilization_pct  resources_spice_eq  energy_net  total_building_levels  total_ships
     12476 Colony 02:380:12      normal             31          137         230                93                  59.57           668897.67         965                    137           75


### 8.2 Full Empire Roster with Empty Colony Slots

The same tables for the whole roster in `empire_planet_states`: the owned planets followed by the generic planets, each an empty colony slot with `EMPTY_COLONY_SIZE_FIELDS` free fields.

- `empire_planets_df` — the summary columns of `live_planets_df` for every slot, with `status` and `is_empty_slot`;
- `empire_planet_buildings_df` — the level of every building on every slot;
- `planet_sizes_arr`, `planet_fields_used_arr`, `planet_fields_remaining_arr` — total, used and free fields per slot, as `numpy` arrays;
- `EMPIRE_PLANET_STATE_SUMMARY` — the totals; `EMPIRE_PLANET_TABLES` — registry of the three planet tables.

In [72]:
def build_planet_tables(planets: list[PlanetState], owned_count: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Summary table and building-level table of a roster whose first `owned_count` planets are owned."""
    building_keys = [f"b{building_id}" for building_id in sorted(game_data.BUILDING_BY_ID)]
    planet_rows: list[dict[str, Any]] = []
    building_rows: list[dict[str, Any]] = []
    for slot_index, planet in enumerate(planets, start=1):
        is_empty_slot = slot_index > owned_count
        planet_rows.append(planet_state_to_row(slot_index, planet, is_empty_slot=is_empty_slot))
        building_rows.append(
            {
                "slot_index": slot_index,
                "planet_id": int(planet.planet_id),
                "coords": planet.coords,
                "status": "empty_colony_slot" if is_empty_slot else "colonized",
                "fields_used": int(planet.fields_used),
                "fields_max": int(planet.fields_max),
                **{key: max(0, int(planet.buildings.get(key, 0) or 0)) for key in building_keys},
            }
        )
    return pd.DataFrame(planet_rows), pd.DataFrame(building_rows)


empire_planets_df, empire_planet_buildings_df = build_planet_tables(empire_planet_states, len(live_planet_states))

planet_sizes_arr: np.ndarray = empire_planets_df["fields_max"].to_numpy(dtype=np.int64)
planet_fields_used_arr: np.ndarray = empire_planets_df["fields_used"].to_numpy(dtype=np.int64)
planet_fields_remaining_arr: np.ndarray = empire_planets_df["fields_remaining"].to_numpy(dtype=np.int64)

EMPIRE_PLANET_STATE_SUMMARY: dict[str, Any] = {
    "max_empire_planets": int(MAX_EMPIRE_PLANETS),
    "empty_colony_size_fields": int(EMPTY_COLONY_SIZE_FIELDS),
    "colonized_planets": int((~empire_planets_df["is_empty_slot"]).sum()),
    "empty_colony_slots": int(empire_planets_df["is_empty_slot"].sum()),
    "total_empire_slots": int(len(empire_planets_df)),
    "total_fields_capacity": int(np.sum(planet_sizes_arr)),
    "total_fields_used": int(np.sum(planet_fields_used_arr)),
    "total_fields_remaining": int(np.sum(planet_fields_remaining_arr)),
    "mean_planet_size_fields": float(np.mean(planet_sizes_arr)) if planet_sizes_arr.size > 0 else 0.0,
}

EMPIRE_PLANET_TABLES: dict[str, pd.DataFrame] = {
    "live_planets": live_planets_df,
    "empire_planets": empire_planets_df,
    "empire_planet_buildings": empire_planet_buildings_df,
}

print(
    f"{EMPIRE_PLANET_STATE_SUMMARY['colonized_planets']} owned + "
    f"{EMPIRE_PLANET_STATE_SUMMARY['empty_colony_slots']} empty {EMPTY_COLONY_SIZE_FIELDS}-field slots = "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_empire_slots']} planets; "
    f"fields: {EMPIRE_PLANET_STATE_SUMMARY['total_fields_used']:,} used of "
    f"{EMPIRE_PLANET_STATE_SUMMARY['total_fields_capacity']:,}\n"
)
print(
    empire_planets_df[
        [
            "slot_index",
            "planet_id",
            "status",
            "name",
            "coords",
            "temperature_c",
            "fields_used",
            "fields_max",
            "fields_remaining",
            "resources_spice_eq",
            "energy_net",
            "total_building_levels",
        ]
    ].to_string(index=False)
)
print("\nBuilding levels per planet (b1-b14):")
print(
    empire_planet_buildings_df[
        ["slot_index", "coords", "status", "fields_used", "fields_max", *(f"b{number}" for number in range(1, 15))]
    ].to_string(index=False)
)

1 owned + 10 empty 300-field slots = 11 planets; fields: 137 used of 3,230

 slot_index  planet_id            status                 name    coords  temperature_c  fields_used  fields_max  fields_remaining  resources_spice_eq  energy_net  total_building_levels
          1      12476         colonized               Colony 02:380:12             31          137         230                93           668897.67         965                    137
          2         -1 empty_colony_slot  Empty Colony Slot 1  EMPTY:01              0            0         300               300                0.00           0                      0
          3         -2 empty_colony_slot  Empty Colony Slot 2  EMPTY:02              0            0         300               300                0.00           0                      0
          4         -3 empty_colony_slot  Empty Colony Slot 3  EMPTY:03              0            0         300               300                0.00           0                      0

## 9. Maximum Research Levels in the Universe

The highest level of every research that anyone in the universe is known to have. Section 11 plans with these levels as its reference.

The bot keeps them in the `universe_research_max` table of its database. The notebook reads that ledger from the temporary database copy and combines it with the sources it can check itself:

- **Server records** (`server_record_level`) — the level that the game's statistics pages list for a research, once the bot has synchronized them into the ledger;
- **Spy reports** (`spied_max_level`) — the highest level found in the archived reports of `spy_reports`;
- **Your own research** (`own_level`) — the levels of the newest planet snapshot.

`universe_max_level` is the highest of these and of the maximum that the ledger stored earlier; the ledger never lowers a level. `max_source` names the source that holds the maximum (`server_record`, `spy_report`, `own_level`, `ledger` or `none`), and `holder` the player, when one is known. A level of 0 means that no source reported the research.

Only the bot fills the ledger; this notebook reads it. Until the bot has synchronized the server records, or spied a player whose report lists researches, every maximum is your own level.

- `universe_research_sources` — the raw inputs: the ledger rows, the highest spied level per research, and the number of spy reports that list researches;
- `universe_research_df` — one row per research: the level from each source, `universe_max_level`, `max_source`, `holder`, the distance from your own level (`gap_to_max`, `own_percent_of_max`), the last level of the catalog, and the time of the ledger's last update. A research that the ledger tracks but the catalog does not list comes last;
- `UNIVERSE_RESEARCH_MAX_LEVELS` — `{research key: universe_max_level}`;
- `UNIVERSE_RESEARCH_SUMMARY` — the number of ledger rows, of spy reports with researches, and of researches known from each source or above your own level.

In [73]:
SQL_UNIVERSE_RESEARCH_LEDGER = """
SELECT
    research_key,
    research_id,
    slug,
    name,
    own_level,
    spied_max_level,
    server_record_level,
    universe_max_level,
    holder,
    holder_coords,
    updated_ts
FROM universe_research_max
ORDER BY research_id ASC, slug ASC
"""

SQL_SPY_REPORT_RESEARCHES = """
SELECT target_coords, owner, researches_json
FROM spy_reports
WHERE researches_json != '' AND researches_json != '{}'
ORDER BY ts ASC, id ASC
"""


def research_key_aliases() -> dict[str, str]:
    """Key, slug and name of every catalog research in lower case, each mapped to the key (`r1`, `r2`, ...)."""
    aliases: dict[str, str] = {}
    for key, info in research_catalog():
        for alias in (key, str(info["slug"]), str(info["name"])):
            aliases[alias.lower()] = key
    return aliases


def load_universe_research_sources(db_path: Path) -> dict[str, Any]:
    """Read what the bot recorded about research levels in the universe from a temporary copy of `db_path`.

    `ledger_rows` are the rows of the bot's `universe_research_max` table. `spied_levels` holds, per research,
    the highest level found in the archived spy reports and the planet it was seen on. Both stay empty without
    a database.
    """
    sources: dict[str, Any] = {"ledger_rows": [], "spied_levels": {}, "spy_reports_with_research": 0}
    if not has_database(db_path):
        return sources
    aliases = research_key_aliases()
    spied_levels: dict[str, dict[str, Any]] = sources["spied_levels"]
    with open_readonly_db_copy(db_path) as conn:
        sources["ledger_rows"] = [dict(row) for row in conn.execute(SQL_UNIVERSE_RESEARCH_LEDGER)]
        for report in conn.execute(SQL_SPY_REPORT_RESEARCHES):
            levels = {
                aliases[name.strip().lower()]: level
                for name, level in parse_level_map(report["researches_json"]).items()
                if level > 0 and name.strip().lower() in aliases
            }
            if levels:
                sources["spy_reports_with_research"] += 1
            for key, level in levels.items():
                # Reports are read oldest first, so the newest sighting of the highest level is kept.
                if level >= spied_levels.get(key, {}).get("level", 0):
                    spied_levels[key] = {
                        "level": level,
                        "holder": str(report["owner"] or ""),
                        "holder_coords": str(report["target_coords"] or ""),
                    }
    return sources


def universe_research_row(
    key: str,
    info: dict[str, Any] | None,
    ledger_row: dict[str, Any],
    spied: dict[str, Any],
    own_level: int,
) -> dict[str, Any]:
    """One `universe_research_df` row: the levels each source reports for a research and the maximum of them."""
    levels = (info or {}).get("levels") or {}
    own = max(int(own_level), int(ledger_row.get("own_level") or 0))
    spied_level = max(int(spied.get("level") or 0), int(ledger_row.get("spied_max_level") or 0))
    server_record = int(ledger_row.get("server_record_level") or 0)
    ledger_max = int(ledger_row.get("universe_max_level") or 0)
    universe_max = max(own, spied_level, server_record, ledger_max)

    if universe_max == 0:
        max_source = "none"
    elif server_record == universe_max:
        max_source = "server_record"
    elif spied_level == universe_max:
        max_source = "spy_report"
    elif own == universe_max:
        max_source = "own_level"
    else:  # the ledger never lowers a maximum, so it can remember more than today's sources show
        max_source = "ledger"
    if max_source == "spy_report" and int(spied.get("level") or 0) == universe_max:
        holder, holder_coords = str(spied["holder"]), str(spied["holder_coords"])
    elif max_source != "own_level" and ledger_max == universe_max > 0:
        holder, holder_coords = str(ledger_row.get("holder") or ""), str(ledger_row.get("holder_coords") or "")
    else:
        holder, holder_coords = "", ""

    updated_ts = float(ledger_row.get("updated_ts") or 0.0)
    return {
        "research_id": int(key[1:]) if info is not None else int(ledger_row.get("research_id") or 0),
        "key": key,
        "slug": str(info["slug"]) if info is not None else str(ledger_row.get("slug") or key),
        "name": str(info["name"]) if info is not None else str(ledger_row.get("name") or key),
        "own_level": own,
        "spied_max_level": spied_level,
        "server_record_level": server_record,
        "universe_max_level": universe_max,
        "max_source": max_source,
        "holder": holder,
        "holder_coords": holder_coords,
        "gap_to_max": universe_max - own,
        "own_percent_of_max": round(100.0 * float(own) / float(universe_max), 1) if universe_max > 0 else 0.0,
        "catalog_max_level": int((info or {}).get("max_table_level") or len(levels)),
        "ledger_updated_utc": (
            datetime.datetime.fromtimestamp(updated_ts, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            if updated_ts > 0.0
            else ""
        ),
    }


def build_universe_research_table(sources: dict[str, Any], own_levels: dict[str, int]) -> pd.DataFrame:
    """One row per research of the catalog, followed by any other research the bot's ledger tracks."""
    catalog = dict(research_catalog())
    ledger = {str(row["research_key"]): row for row in sources["ledger_rows"]}
    spied_levels = sources["spied_levels"]
    return pd.DataFrame(
        [
            universe_research_row(
                key,
                catalog.get(key),
                ledger.get(key, {}),
                spied_levels.get(key, {}),
                max(0, int(own_levels.get(key, 0) or 0)),
            )
            for key in (*catalog, *(key for key in ledger if key not in catalog))
        ]
    )


universe_research_sources: dict[str, Any] = load_universe_research_sources(LIVE_DB_PATH)
universe_research_df = build_universe_research_table(universe_research_sources, colony_researches)

# The reference levels of Section 11: the highest known level of every research in the universe.
UNIVERSE_RESEARCH_MAX_LEVELS: dict[str, int] = {
    str(key): int(level) for key, level in zip(universe_research_df["key"], universe_research_df["universe_max_level"])
}
UNIVERSE_RESEARCH_SUMMARY: dict[str, Any] = {
    "researches": int(len(universe_research_df)),
    "ledger_rows": len(universe_research_sources["ledger_rows"]),
    "spy_reports_with_research": int(universe_research_sources["spy_reports_with_research"]),
    "researches_with_server_record": int((universe_research_df["server_record_level"] > 0).sum()),
    "researches_seen_in_spy_reports": int((universe_research_df["spied_max_level"] > 0).sum()),
    "researches_above_own_level": int((universe_research_df["gap_to_max"] > 0).sum()),
    "highest_level": int(universe_research_df["universe_max_level"].max()),
}

print(
    f"{UNIVERSE_RESEARCH_SUMMARY['ledger_rows']} ledger row(s), "
    f"{UNIVERSE_RESEARCH_SUMMARY['spy_reports_with_research']} spy report(s) with research levels, "
    f"{UNIVERSE_RESEARCH_SUMMARY['researches_with_server_record']} server record(s); "
    f"{UNIVERSE_RESEARCH_SUMMARY['researches_above_own_level']} research(es) known above your own level\n"
)
if UNIVERSE_RESEARCH_SUMMARY["researches_above_own_level"] == 0:
    print("No other player's research is recorded yet, so every universe maximum is your own level.\n")
print(
    universe_research_df[
        [
            "key",
            "name",
            "own_level",
            "spied_max_level",
            "server_record_level",
            "universe_max_level",
            "max_source",
            "holder",
            "gap_to_max",
            "own_percent_of_max",
            "catalog_max_level",
        ]
    ].to_string(index=False)
)

16 ledger row(s), 0 spy report(s) with research levels, 13 server record(s); 13 research(es) known above your own level

                key                name  own_level  spied_max_level  server_record_level  universe_max_level    max_source   holder  gap_to_max  own_percent_of_max  catalog_max_level
                 r1   Mining technology          9                0                   30                  30 server_record ventkx86          21                30.0                 40
                 r2   Energy technology          4                0                   28                  28 server_record ventkx86          24                14.3                 40
                 r3     Engine research          6                0                   20                  20 server_record ventkx86          14                30.0                 35
                 r4       Rocket engine          8                0                   26                  26 server_record ventkx86          18    

## 10. Print Every Generated Data Set (One Cell per Section)

Each cell below prints, in full, everything the matching cell of Sections 1 to 9 generated: every row and column of each `pandas` table, every element of each `numpy` array, and every dictionary and scalar. Nothing is shortened with `...`, large numbers are written out instead of switching to scientific notation, and every printout starts with the date and time at which it was printed.

- Cells that only install packages, import modules, or define functions, classes or SQL text generate no data, so they have no print cell.
- The `SQL_*` query texts and the `*_COLUMNS` tuples are constants of the source, not generated data, and are not repeated.
- All printed text passes through `redact_sensitive`, so a session token can never appear in a printout.
- Section 11 comes after this section; its cells print their own data in the same way.

### 10.1 Section 1.1 Data — Workspace Paths & Shared Constants

`REPO_ROOT`, `ACTIVE_DATA_ROOT`, `LIVE_DB_PATH`, `METADATA_DIR` and `RESOURCE_KEYS`.

In [74]:
print_section_data(
    "Section 1.1 data: workspace paths and shared constants",
    [
        ("REPO_ROOT", REPO_ROOT),
        ("ACTIVE_DATA_ROOT", ACTIVE_DATA_ROOT),
        ("LIVE_DB_PATH", LIVE_DB_PATH),
        ("METADATA_DIR", METADATA_DIR),
        ("RESOURCE_KEYS", RESOURCE_KEYS),
    ],
)

Section 1.1 data: workspace paths and shared constants
Printed: 2026-11-17 09:59:32 -0300 | 5 item(s)
[REPO_ROOT] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer')
----------------------------------------------------------------------------------------------------
[ACTIVE_DATA_ROOT] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer')
----------------------------------------------------------------------------------------------------
[LIVE_DB_PATH] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/state/spaceinvasion.db')
----------------------------------------------------------------------------------------------------
[METADATA_DIR] PosixPath = PosixPath('/usr/local/google/home/vieiravitor/workspace/opensource/spaceinvasion-autoplayer/notebooks/metadata')
--------------------------------------------------------------------

### 10.2 Section 1.6 Data — Static Metadata

The content of the JSON files as loaded: `BUILDING_METADATA`, `TECHNOLOGY_METADATA`, `TECHNOLOGY_COST_SCALE_FACTOR`, `SHIP_METADATA`, `DEFENSE_METADATA`, `RESOURCE_METADATA` and `PLANNER_SETTINGS`.

In [75]:
print_section_data(
    "Section 1.6 data: static metadata",
    [
        ("BUILDING_METADATA", BUILDING_METADATA),
        ("TECHNOLOGY_METADATA", TECHNOLOGY_METADATA),
        ("TECHNOLOGY_COST_SCALE_FACTOR", TECHNOLOGY_COST_SCALE_FACTOR),
        ("SHIP_METADATA", SHIP_METADATA),
        ("DEFENSE_METADATA", DEFENSE_METADATA),
        ("RESOURCE_METADATA", RESOURCE_METADATA),
        ("PLANNER_SETTINGS", PLANNER_SETTINGS),
    ],
)

Section 1.6 data: static metadata
Printed: 2026-11-17 09:59:32 -0300 | 7 item(s)
[BUILDING_METADATA] dict, len=21
{
  "b1": {"name": "Pig iron mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Pig iron (+2%/lvl r1); consumes energy"},
  "b2": {"name": "Blast furnace", "category": "smelter", "cost_scale_factor": 1.51, "primary_effect": "Smelts Pig iron -> Metal at 2:1 loss (NOT boosted by r1); consumes energy"},
  "b3": {"name": "Kryptonite mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Kryptonite (+2%/lvl r1); consumes energy"},
  "b4": {"name": "Spice mine", "category": "resource_mine", "cost_scale_factor": 1.51, "primary_effect": "Mines Spice on planet-temperature curve (+2%/lvl r1); consumes energy"},
  "b5": {"name": "Fusion plant", "category": "power_plant", "cost_scale_factor": 1.51, "primary_effect": "Fuel-free planetary power (+1%/lvl r2); 1 field/level"},
  "b6": {"name": "Thermosolar plant", "categ

### 10.3 Section 2.3 Data — Attack Telemetry

`telemetry_source`, `origin_planet_coords`, `fleet_slot_capacity`, `colony_buildings`, `colony_researches`, the other entries of `telemetry_bundle`, `attacks_df`, `recycles_df` and `attack_dispatch_ts_arr`. The three row lists of `telemetry_bundle` are not printed separately: they are the rows of the two tables and the values of the array.

In [76]:
print_section_data(
    "Section 2.3 data: attack telemetry",
    [
        ("telemetry_source", telemetry_source),
        ("origin_planet_coords", origin_planet_coords),
        ("fleet_slot_capacity", fleet_slot_capacity),
        ("colony_buildings", colony_buildings),
        ("colony_researches", colony_researches),
        (
            "telemetry_bundle (every other key; its three row lists are the tables and array below)",
            {
                key: value
                for key, value in telemetry_bundle.items()
                if key
                not in ("records", "recycle_rows", "dispatch_timestamps", "colony_buildings", "colony_researches")
            },
        ),
        ("attacks_df", attacks_df),
        ("recycles_df", recycles_df),
        ("attack_dispatch_ts_arr", attack_dispatch_ts_arr),
    ],
)

Section 2.3 data: attack telemetry
Printed: 2026-11-17 09:59:32 -0300 | 9 item(s)
[telemetry_source] str = 'sqlite:mer_joined(battle_reports+raid_history+spy_targets)'
----------------------------------------------------------------------------------------------------
[origin_planet_coords] str = '02:380:12'
----------------------------------------------------------------------------------------------------
[fleet_slot_capacity] int = 7
----------------------------------------------------------------------------------------------------
[colony_buildings] dict, len=19
{
  "b1": 21,
  "b10": 6,
  "b11": 5,
  "b12": 3,
  "b13": 4,
  "b14": 3,
  "b15": 0,
  "b16": 0,
  "b17": 0,
  "b18": 0,
  "b2": 20,
  "b21": 0,
  "b3": 21,
  "b4": 15,
  "b5": 23,
  "b6": 0,
  "b7": 9,
  "b8": 3,
  "b9": 4
}
----------------------------------------------------------------------------------------------------
[colony_researches] dict, len=15
{
  "r1": 9,
  "r10": 0,
  "r11": 0,
  "r12": 0,
  "r13": 0,
  "r

### 10.4 Section 3 Data — Attack Resource Means & Standard Deviations

`METRIC_KEYS`, `attack_resource_stats_df`, the five dictionaries of means and standard deviations per resource, the mean and standard deviation of the recycler harvests, and `winning_attacks_df`.

In [77]:
print_section_data(
    "Section 3 data: attack resource means and standard deviations",
    [
        ("METRIC_KEYS", METRIC_KEYS),
        ("attack_resource_stats_df", attack_resource_stats_df),
        ("attack_resource_mean", attack_resource_mean),
        ("attack_resource_std", attack_resource_std),
        ("attack_resource_pop_std", attack_resource_pop_std),
        ("winning_attack_resource_mean", winning_attack_resource_mean),
        ("winning_attack_resource_std", winning_attack_resource_std),
        ("recycle_spice_eq_mean", recycle_spice_eq_mean),
        ("recycle_spice_eq_std", recycle_spice_eq_std),
        ("winning_attacks_df", winning_attacks_df),
    ],
)

Section 3 data: attack resource means and standard deviations
Printed: 2026-11-17 09:59:32 -0300 | 10 item(s)
[METRIC_KEYS] tuple, len=7
["pig_iron", "metal", "kryptonite", "spice", "spice_eq", "net_spice_eq", "debris_spice_eq"]
----------------------------------------------------------------------------------------------------
[attack_resource_stats_df] pandas.DataFrame, shape=(7, 9)
         metric  samples_all     mean_all sample_std_all  pop_std_all  samples_wins   mean_wins sample_std_wins pop_std_wins
       pig_iron            7  4065.857143    4575.737880  4236.310100             6 4743.500000     4611.677992  4209.866773
          metal            7  5377.285714    5413.949443  5012.343213             6 6273.500000     5331.616819  4867.077999
     kryptonite            7  5705.142857    5930.124547  5490.228500             6 6656.000000     5882.598609  5370.053259
          spice            7  2328.000000    2527.637237  2340.137359             6 2716.000000     2530.251055 

### 10.5 Section 4 Data — Attack Flight & Cadence Times

`attack_time_stats_df`, the nine unrounded means and standard deviations, and the arrays `one_way_times_arr`, `round_trip_times_arr` and `inter_attack_intervals_arr`.

In [78]:
print_section_data(
    "Section 4 data: attack flight and cadence times",
    [
        ("attack_time_stats_df", attack_time_stats_df),
        (
            "exact time statistics in seconds",
            {
                "attack_one_way_time_mean_sec": attack_one_way_time_mean_sec,
                "attack_one_way_time_std_sec": attack_one_way_time_std_sec,
                "attack_one_way_time_pop_std_sec": attack_one_way_time_pop_std_sec,
                "attack_time_mean_sec": attack_time_mean_sec,
                "attack_time_std_sec": attack_time_std_sec,
                "attack_time_pop_std_sec": attack_time_pop_std_sec,
                "inter_attack_time_mean_sec": inter_attack_time_mean_sec,
                "inter_attack_time_std_sec": inter_attack_time_std_sec,
                "inter_attack_time_pop_std_sec": inter_attack_time_pop_std_sec,
            },
        ),
        ("one_way_times_arr", one_way_times_arr),
        ("round_trip_times_arr", round_trip_times_arr),
        ("inter_attack_intervals_arr", inter_attack_intervals_arr),
    ],
)

Section 4 data: attack flight and cadence times
Printed: 2026-11-17 09:59:32 -0300 | 5 item(s)
[attack_time_stats_df] pandas.DataFrame, shape=(3, 8)
                             time_metric  samples mean_sec mean_text sample_std_sec sample_std_text pop_std_sec median_sec
One-Way Flight Time (Dispatch -> Impact)        7   151.47    2m 31s         113.25          1m 53s      104.85     115.58
Round-Trip Attack Time (Slot Occupation)        7   302.93    5m 03s         226.49          3m 46s      209.69     231.17
          Inter-Attack Dispatch Interval        6 13249.65    3h 40m       21626.47          6h 00m    19742.18    3354.80
----------------------------------------------------------------------------------------------------
[exact time statistics in seconds] dict, len=9
{
  "attack_one_way_time_mean_sec": 151.4660808500405,
  "attack_one_way_time_std_sec": 113.24717030740591,
  "attack_one_way_time_pop_std_sec": 104.84650651296167,
  "attack_time_mean_sec": 302.932161700081,
  

### 10.6 Section 5 Data — Attack Success Rate & Simulation Parameters

`attack_summary_stats_df`, the success-rate and expected-yield scalars, `attack_class_stats_df` with `per_class_success_rate`, `attack_target_stats_df` with `per_target_success_rate`, `ATTACK_SIM_PARAMS`, and the `ATTACK_STATS_TABLES` registry.

In [79]:
print_section_data(
    "Section 5 data: attack success rate and simulation parameters",
    [
        ("attack_summary_stats_df", attack_summary_stats_df),
        (
            "success-rate and expected-yield scalars",
            {
                "attack_attempts": attack_attempts,
                "attack_wins": attack_wins,
                "attack_losses": attack_losses,
                "attack_success_rate": attack_success_rate,
                "attack_success_rate_std_err": attack_success_rate_std_err,
                "attack_success_rate_smoothed": attack_success_rate_smoothed,
                "expected_raids_per_slot_hr": expected_raids_per_slot_hr,
                "expected_spice_eq_per_slot_hr": expected_spice_eq_per_slot_hr,
                "expected_net_spice_eq_per_slot_hr": expected_net_spice_eq_per_slot_hr,
                "max_empire_gross_spice_eq_hr": max_empire_gross_spice_eq_hr,
            },
        ),
        ("attack_class_stats_df", attack_class_stats_df),
        ("per_class_success_rate", per_class_success_rate),
        ("attack_target_stats_df", attack_target_stats_df),
        ("per_target_success_rate", per_target_success_rate),
        ("ATTACK_SIM_PARAMS", ATTACK_SIM_PARAMS),
        ("ATTACK_STATS_TABLES", ATTACK_STATS_TABLES),
    ],
)

Section 5 data: attack success rate and simulation parameters
Printed: 2026-11-17 09:59:32 -0300 | 8 item(s)
[attack_summary_stats_df] pandas.DataFrame, shape=(7, 3)
                            metric         value                                 display
                     Total Attacks      7.000000                   7 (6 wins / 1 losses)
            Empirical Success Rate      0.857143                  85.71% (+/- 13.23% SE)
     Laplace-Smoothed Success Rate      0.777778                                  77.78%
 Max Sorties per Fleet Slot / Hour     11.883800 11.88 raids/hr (mean round-trip 5m 03s)
Gross Loot per Slot-Hour (SpiceEq)  83879.430000                     83,879.4 SpiceEq/hr
  Net Loot per Slot-Hour (SpiceEq) -30091.770000                    -30,091.8 SpiceEq/hr
   Full Capacity Ceiling (7 Slots) 587156.010000              587,156.0 gross SpiceEq/hr
----------------------------------------------------------------------------------------------------
[success-rate and exp

### 10.7 Section 6 Data — Colonies, Generic Planets & Contexts

The table level caps and empire settings, `GAME_PARAM_CONTEXT`, `GENERIC_PLANET_CONTEXT`, `planet_contexts_df`, `PLANET_PARAM_CONTEXTS`, `merchant_rates_df`, and the typed `live_planet_states` and `empire_planet_states`: one JSON line per planet with its resources, storage, energy, buildings, ships, defenses and economy rates.

In [80]:
print_section_data(
    "Section 6 data: colonies, generic planets and contexts",
    [
        (
            "table level caps and empire settings",
            {
                "MAX_BUILDING_TABLE_LEVEL": MAX_BUILDING_TABLE_LEVEL,
                "MAX_RESEARCH_TABLE_LEVEL": MAX_RESEARCH_TABLE_LEVEL,
                "MAX_EMPIRE_PLANETS": MAX_EMPIRE_PLANETS,
                "EMPTY_COLONY_SIZE_FIELDS": EMPTY_COLONY_SIZE_FIELDS,
                "EMPTY_COLONY_DEFAULT_TEMP_C": EMPTY_COLONY_DEFAULT_TEMP_C,
            },
        ),
        ("GAME_PARAM_CONTEXT", GAME_PARAM_CONTEXT),
        ("GENERIC_PLANET_CONTEXT", GENERIC_PLANET_CONTEXT),
        ("planet_contexts_df", planet_contexts_df),
        ("PLANET_PARAM_CONTEXTS", PLANET_PARAM_CONTEXTS),
        ("merchant_rates_df", merchant_rates_df),
        ("live_planet_states", live_planet_states),
        ("empire_planet_states", empire_planet_states),
    ],
)

Section 6 data: colonies, generic planets and contexts
Printed: 2026-11-17 09:59:32 -0300 | 8 item(s)
[table level caps and empire settings] dict, len=5
{
  "MAX_BUILDING_TABLE_LEVEL": 50,
  "MAX_RESEARCH_TABLE_LEVEL": 30,
  "MAX_EMPIRE_PLANETS": 11,
  "EMPTY_COLONY_SIZE_FIELDS": 300,
  "EMPTY_COLONY_DEFAULT_TEMP_C": 0
}
----------------------------------------------------------------------------------------------------
[GAME_PARAM_CONTEXT] dict, len=11
{
  "wiki_synced_at_utc": "2026-11-15T02:03:10Z",
  "speedup_factor": 20.0,
  "r1_mining_tech": 9,
  "r2_energy_tech": 4,
  "r4_rocket_engine": 8,
  "r5_diffusion_engine": 7,
  "r6_warp_engine": 0,
  "r8_weapons_tech": 0,
  "r9_shield_tech": 0,
  "r10_defense_systems": 0,
  "points_per_resource_divisor": 1000.0
}
----------------------------------------------------------------------------------------------------
[GENERIC_PLANET_CONTEXT] dict, len=22
{
  "wiki_synced_at_utc": "2026-11-15T02:03:10Z",
  "speedup_factor": 20.0,
  "r1_mining

### 10.8 Section 6.1 Data — Buildings

`buildings_catalog_df` (one row per building), `planet_buildings_df` (one row per planet and building) and `buildings_levels_df` (every level of every building, all columns).

**Precision of high-level costs.** `game_data` copies its costs from the wiki, which rounds every cost of `1,000,000,000` or more. Compared with exact doubling for the buildings whose cost doubles per level: every cost below `10^9` is exact, larger costs are up to 0.42 % off, and from `10^15` on only 5 significant digits remain (hence values such as `281470000000000032768`).

In [81]:
print_section_data(
    "Section 6.1 data: buildings",
    [
        ("buildings_catalog_df", buildings_catalog_df),
        ("planet_buildings_df", planet_buildings_df),
        ("buildings_levels_df", buildings_levels_df),
    ],
)

Section 6.1 data: buildings
Printed: 2026-11-17 09:59:32 -0300 | 3 item(s)
[buildings_catalog_df] pandas.DataFrame, shape=(21, 17)
 building_id key                    slug                    name          category  asteroid_ok  max_table_level cost_scale_factor  l1_pig_iron  l1_metal  l1_kryptonite  l1_spice l1_cost_spice_eq l1_base_time_s l1_effective_time_s                                            prerequisites                                                                                           primary_effect
           1  b1           pig-iron-mine           Pig iron mine     resource_mine        False              100              1.51           78         0             25         0            21.33             93                4.65                                                     None                                                             Mines Pig iron (+2%/lvl r1); consumes energy
           2  b2           blast-furnace           Blast furnace           smelter 

### 10.9 Section 6.2 Data — Technologies

`technologies_catalog_df` (one row per research), `planet_technologies_df` (one row per planet and research) and `technologies_levels_df` (every level of every research).

The wiki rounding described in 10.8 applies here too: research costs of `1,000,000,000` or more are up to 0.39 % off exact doubling, and costs below `10^9` are exact.

In [82]:
print_section_data(
    "Section 6.2 data: technologies",
    [
        ("technologies_catalog_df", technologies_catalog_df),
        ("planet_technologies_df", planet_technologies_df),
        ("technologies_levels_df", technologies_levels_df),
    ],
)

Section 6.2 data: technologies
Printed: 2026-11-17 09:59:32 -0300 | 3 item(s)
[technologies_catalog_df] pandas.DataFrame, shape=(15, 20)
 research_id key                slug                name           domain  current_level  max_table_level cost_scale_factor  l1_pig_iron  l1_metal  l1_kryptonite  l1_spice l1_cost_spice_eq l1_base_time_s l1_effective_time_s                  effect_metric per_level_bonus_pct                                                                                          applies_to                                       prerequisites                                                                   strategic_effect
           1  r1   mining-technology   Mining technology          economy              9               40                 2         1000         0            500         0           333.33           2083              104.15              mine_output_bonus                   2                      b1 Pig iron, b3 Kryptonite, b4 Spice across all planets (

### 10.10 Section 6.3 Data — Spaceships

`ships_df` (every ship, all columns) and `planet_ships_df` (one row per planet and ship).

In [83]:
print_section_data(
    "Section 6.3 data: spaceships",
    [
        ("ships_df", ships_df),
        ("planet_ships_df", planet_ships_df),
    ],
)

Section 6.3 data: spaceships
Printed: 2026-11-17 09:59:32 -0300 | 2 item(s)
[ships_df] pandas.DataFrame, shape=(15, 34)
 ship_id key              slug              name                    role  pig_iron   metal  kryptonite  spice cost_spice_eq points_value base_build_time_s effective_build_time_s effective_build_time_text units_per_hour_effective  hull_base  shield_base  attack_base hull_effective shield_effective attack_effective           engine engine_tech_key  speed_base speed_effective   cargo  fuel energy_output_base energy_output_effective  fields_used cargo_per_spice_eq attack_per_spice_eq hull_per_spice_eq                                                                prerequisites
       1  s1   small-transport   Small transport         cargo_transport         0    3000        3000      0       2200.00          6.0           1700.00                  85.00                    1m 25s                    42.35       6000           10            2           6000               10   

### 10.11 Section 6.4 Data — Planetary Defenses & Table Registry

`defenses_df` (every defense, all columns), `planet_defenses_df` (one row per planet and defense), `game_params_summary_df`, and the `GAME_PARAMS_TABLES` registry (name and shape of the twelve tables of Section 6).

In [84]:
print_section_data(
    "Section 6.4 data: planetary defenses and table registry",
    [
        ("defenses_df", defenses_df),
        ("planet_defenses_df", planet_defenses_df),
        ("game_params_summary_df", game_params_summary_df),
        ("GAME_PARAMS_TABLES", GAME_PARAMS_TABLES),
    ],
)

Section 6.4 data: planetary defenses and table registry
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[defenses_df] pandas.DataFrame, shape=(11, 25)
 defense_id key                  slug                  name             role  can_dig_in  pig_iron  metal  kryptonite  spice cost_spice_eq points_value base_build_time_s effective_build_time_s effective_build_time_text units_per_hour_effective  hull_base  shield_base  attack_base hull_effective shield_effective attack_effective attack_per_spice_eq hull_plus_shield_per_spice_eq                                                                     prerequisites
          1  d1                mortar                Mortar planetary_turret        True         0   2000           0      0        800.00          2.0           1000.00                  50.00                       50s                    72.00       2000           15           35           2000               15               35              0.0437                          2.52         

### 10.12 Section 7.2 Data — Technology Requirement Graph

The content of `TECH_GRAPH` (node and edge counts, `topological_order`, the `nodes` store, and the `ancestors_of` and `descendants_of` closure of every node) with `tech_graph_nodes_df` and `tech_graph_edges_df`.

In [85]:
print_section_data(
    "Section 7.2 data: technology requirement graph",
    [
        (
            "TECH_GRAPH (TechnologyRequirementGraph summary)",
            {
                "nodes": len(TECH_GRAPH.nodes),
                "edges": sum(len(requirements) for requirements in TECH_GRAPH.prereqs_of.values()),
                "max_topological_layer": max(TECH_GRAPH.topological_layer.values()),
            },
        ),
        ("TECH_GRAPH.topological_order", list(TECH_GRAPH.topological_order)),
        ("TECH_GRAPH.nodes", pd.DataFrame(list(TECH_GRAPH.nodes.values()))),
        ("tech_graph_nodes_df", tech_graph_nodes_df),
        ("tech_graph_edges_df", tech_graph_edges_df),
        (
            "TECH_GRAPH.ancestors_of / TECH_GRAPH.descendants_of (transitive closure per node)",
            {
                node: {
                    "all_prerequisites": [
                        key for key in TECH_GRAPH.topological_order if key in TECH_GRAPH.ancestors_of[node]
                    ],
                    "all_unlocks": [
                        key for key in TECH_GRAPH.topological_order if key in TECH_GRAPH.descendants_of[node]
                    ],
                }
                for node in TECH_GRAPH.topological_order
            },
        ),
    ],
)

Section 7.2 data: technology requirement graph
Printed: 2026-11-17 09:59:32 -0300 | 6 item(s)
[TECH_GRAPH (TechnologyRequirementGraph summary)] dict, len=3
{
  "nodes": 62,
  "edges": 101,
  "max_topological_layer": 5
}
----------------------------------------------------------------------------------------------------
[TECH_GRAPH.topological_order] list, len=62
["b1", "b2", "b3", "b4", "b5", "b7", "b10", "b11", "b12", "b13", "b14", "b8", "b19", "r2", "r3", "r7", "r8", "r10", "r14", "b6", "b9", "b17", "r1", "r9", "r11", "r13", "r4", "r5", "r6", "b20", "s9", "d1", "d5", "b18", "b21", "r12", "d10", "d11", "d2", "d3", "d4", "d7", "d8", "d9", "s1", "s2", "s4", "s5", "s8", "s10", "s3", "s6", "s7", "s11", "r15", "s13", "s14", "s15", "b15", "b16", "d6", "s16"]
----------------------------------------------------------------------------------------------------
[TECH_GRAPH.nodes] pandas.DataFrame, shape=(62, 6)
key  category  entity_id                    slug                    name  max_level


### 10.13 Section 7.3 Data — Unlock Plans

`unlock_targets_summary_df` (remaining cost, time and ordered chain of upgrades of every entity), `phoenix_unlock_plan_df` (from the reference colony), `cold_phoenix_unlock_plan_df` (from an empty account), and the `TECH_GRAPH_TABLES` registry.

In [86]:
print_section_data(
    "Section 7.3 data: unlock plans",
    [
        ("unlock_targets_summary_df", unlock_targets_summary_df),
        ("phoenix_unlock_plan_df", phoenix_unlock_plan_df),
        ("cold_phoenix_unlock_plan_df", cold_phoenix_unlock_plan_df),
        ("TECH_GRAPH_TABLES", TECH_GRAPH_TABLES),
    ],
)

Section 7.3 data: unlock plans
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[unlock_targets_summary_df] pandas.DataFrame, shape=(62, 11)
key                    name  category  topological_layer  unlocked_now  missing_upgrade_nodes  total_levels_to_add unlock_cost_spice_eq unlock_effective_time_s unlock_effective_time_text                                                ordered_prerequisite_chain
 b1           Pig iron mine buildings                  0          True                      0                    0                 0.00                    0.00                      Ready                                                                  Unlocked
 b2           Blast furnace buildings                  0          True                      0                    0                 0.00                    0.00                      Ready                                                                  Unlocked
 b3         Kryptonite mine buildings                  0          True        

### 10.14 Section 8.1 Data — Owned Planets

`live_planets_df` (one row per owned planet, all columns).

In [87]:
print_section_data(
    "Section 8.1 data: owned planets",
    [
        ("live_planets_df", live_planets_df),
    ],
)

Section 8.1 data: owned planets
Printed: 2026-11-17 09:59:32 -0300 | 1 item(s)
[live_planets_df] pandas.DataFrame, shape=(1, 27)
 slot_index  planet_id    status  is_empty_slot   name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining field_utilization_pct  pig_iron  metal  kryptonite  spice resources_spice_eq  storage_pig_iron  storage_metal  storage_kryptonite  storage_spice  energy_produced  energy_used  energy_net  total_building_levels  total_ships  total_defenses
          1      12476 colonized          False Colony 02:380:12      normal             31          137         230                93                 59.57    195068  40720      433519 475592          668897.67           3200000         800000             1600000         800000             5954         4989         965                    137           75               0


### 10.15 Section 8.2 Data — Full Empire Roster with Empty Colony Slots

`EMPIRE_PLANET_STATE_SUMMARY`, `empire_planets_df`, `empire_planet_buildings_df` (the level of every building per slot), the three field arrays, and the `EMPIRE_PLANET_TABLES` registry.

In [88]:
print_section_data(
    f"Section 8.2 data: full empire roster with empty {EMPTY_COLONY_SIZE_FIELDS}-field slots",
    [
        ("EMPIRE_PLANET_STATE_SUMMARY", EMPIRE_PLANET_STATE_SUMMARY),
        ("empire_planets_df", empire_planets_df),
        ("empire_planet_buildings_df", empire_planet_buildings_df),
        ("planet_sizes_arr", planet_sizes_arr),
        ("planet_fields_used_arr", planet_fields_used_arr),
        ("planet_fields_remaining_arr", planet_fields_remaining_arr),
        ("EMPIRE_PLANET_TABLES", EMPIRE_PLANET_TABLES),
    ],
)

Section 8.2 data: full empire roster with empty 300-field slots
Printed: 2026-11-17 09:59:32 -0300 | 7 item(s)
[EMPIRE_PLANET_STATE_SUMMARY] dict, len=9
{
  "max_empire_planets": 11,
  "empty_colony_size_fields": 300,
  "colonized_planets": 1,
  "empty_colony_slots": 10,
  "total_empire_slots": 11,
  "total_fields_capacity": 3230,
  "total_fields_used": 137,
  "total_fields_remaining": 3093,
  "mean_planet_size_fields": 293.6363636363636
}
----------------------------------------------------------------------------------------------------
[empire_planets_df] pandas.DataFrame, shape=(11, 27)
 slot_index  planet_id            status  is_empty_slot                 name    coords planet_type  temperature_c  fields_used  fields_max  fields_remaining field_utilization_pct  pig_iron  metal  kryptonite  spice resources_spice_eq  storage_pig_iron  storage_metal  storage_kryptonite  storage_spice  energy_produced  energy_used  energy_net  total_building_levels  total_ships  total_defenses
      

### 10.16 Section 9 Data — Maximum Research Levels in the Universe

`universe_research_sources` (ledger rows, highest spied levels, number of spy reports with researches), `universe_research_df`, `UNIVERSE_RESEARCH_MAX_LEVELS` and `UNIVERSE_RESEARCH_SUMMARY`.

In [89]:
print_section_data(
    "Section 9 data: maximum research levels in the universe",
    [
        ("universe_research_sources", universe_research_sources),
        ("universe_research_df", universe_research_df),
        ("UNIVERSE_RESEARCH_MAX_LEVELS", UNIVERSE_RESEARCH_MAX_LEVELS),
        ("UNIVERSE_RESEARCH_SUMMARY", UNIVERSE_RESEARCH_SUMMARY),
    ],
)

Section 9 data: maximum research levels in the universe
Printed: 2026-11-17 09:59:32 -0300 | 4 item(s)
[universe_research_sources] dict, len=3
{
  "ledger_rows": [{"research_key": "r1", "research_id": 1, "slug": "mining-technology", "name": "Mining technology", "own_level": 9, "spied_max_level": 0, "server_record_level": 30, "universe_max_level": 30, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r2", "research_id": 2, "slug": "energy-technology", "name": "Energy technology", "own_level": 4, "spied_max_level": 0, "server_record_level": 28, "universe_max_level": 28, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r3", "research_id": 3, "slug": "engine-research", "name": "Engine research", "own_level": 6, "spied_max_level": 0, "server_record_level": 20, "universe_max_level": 20, "holder": "ventkx86", "holder_coords": "", "updated_ts": 1794900668.659}, {"research_key": "r4", "research_id": 4, "slug": 

## 11. Best Empire Configuration for Imperial Starbase Production (Genetic Algorithm)

Which building levels, on which planets, build the most Imperial starbases per month? This section answers the question for the roster of Section 6: the colonies you own, plus the generic planets that stand for the colonies you can still found. Asteroids take no part.

**The model** describes a steady state: every building of the configuration stands, and each month's production is spent on the target ship.

- A planet's mines run at the share of the energy that the planet produces. Its Blast furnace smelts the planet's own Pig iron, 2 per Metal.
- Resources are pooled, so the empire pays for as many ships as its scarcest resource allows. Hauling resources to the shipyards, Merchant trades and raids are not modelled.
- Every planet has one shipyard queue. It builds the target ship only where the ship's building prerequisites are met (Weapons factory 13 for the Imperial starbase), at the speed of its Weapons factory and Microsystem accelerator.
- The number of ships per month is the smallest of four **limits**: the ships that the Metal, the Kryptonite and the Spice pay for, and the ships that the shipyards can finish.
- Every building level takes one field of its planet, and so does every warehouse level.

**Warehouses.** The model sets the four warehouses of every planet itself. Each one gets the lowest level that meets all of these rules:

1. It holds `WAREHOUSE_HOURS_OF_PRODUCTION` hours of the planet's own net production of its resource.
2. It holds the cost of the most expensive thing that the planet still has to pay for: a level of one of its buildings (a warehouse level included), a ship that it builds or, on the planet with the Research lab, a level of the planned research.
3. On the central planet, it also holds that many hours of the production of the whole empire, and it does not fall below `CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL`.
4. It does not fall below its level in `MISC_BUILDING_LEVELS`, nor below the level already built.

Rule 2 is what limits the level of a mine. A warehouse holds twice as much with every level, and the last level of a mine costs twice as much about every 1.7 levels. A mine of level 50 therefore needs warehouses of level 18 to 21 for the resources it is paid in, and a mine of level 70 needs levels 30 to 33.

The cost of building the configuration does not enter the search. Section 11.5 reports it.

**Fixed and searched.** The meta parameters fix the research levels, the fleet and the miscellaneous buildings of every planet. The genetic algorithm decides the levels of the other buildings on every planet, and the warehouses follow from them. Unlike the cells of Sections 1 to 9, each cell of this section prints its own data in full.

### 11.1 Meta Parameters

Everything you can tune is in this cell.

- `TARGET_SHIP_KEY`, `PRODUCTION_MONTH_DAYS` — the ship to count, and the length of the month.
- `RESEARCH_PERCENT_OF_UNIVERSE_MAX` — the research levels of the plan, as a percentage of the universe maximum of Section 9. Use a lower percentage when the universe maximum is far beyond your current game. A planned level never falls below your current level or below a level that the plan requires.
- `MIN_FLEET_PER_PLANET` — ships that every planet keeps. The plan gives every planet the Weapons factory level needed to build them, counts the energy of Solar cells among them, and reports what the missing ships cost.
- `MISC_BUILDING_LEVELS`, `MISC_BUILDING_LEVELS_BY_SLOT` — the level of every building that the algorithm does not decide: on all planets, and exceptions for single planets. A colony keeps a level that it has already built beyond these. For the four warehouses the level is only a minimum.
- `WAREHOUSE_HOURS_OF_PRODUCTION` — the hours of production that the warehouses hold (rules 1 and 3).
- `CENTRAL_PLANET_SLOT`, `CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL` — the roster slot of the central planet, your home planet unless you change it, and the lowest level of its four warehouses (rule 3).
- `OPTIMIZED_BUILDING_MAX_LEVELS` — the buildings whose levels the algorithm decides, each with the highest level it may choose. Together with the miscellaneous buildings they must cover every building once.
- `CATALOGUE_*`, `HILL_CLIMB_STEP_SIZES`, `GA_*` — settings of the search, explained in Section 11.4.

In [ ]:
# --- Objective -------------------------------------------------------------------------------------------------
TARGET_SHIP_KEY: str = "s16"  # the ship to mass-produce: the Imperial starbase
PRODUCTION_MONTH_DAYS: float = 30.0  # length of the month that the production is counted over

# --- Research reference ----------------------------------------------------------------------------------------
# Research levels of the plan, as a percentage of the highest level known in the universe (Section 9).
# 100 plans with the universe maximum; lower it to stay closer to your current game. A planned level never falls
# below your current level, nor below the level that the ships and buildings of the plan require.
RESEARCH_PERCENT_OF_UNIVERSE_MAX: float = 100.0

# --- Fixed part of every planet --------------------------------------------------------------------------------
# Ships that every planet keeps (ship key -> count). Every planet must be able to build them itself.
MIN_FLEET_PER_PLANET: dict[str, int] = {
    "s2": 10,  # Large transport
    "s10": 10,  # Spy probe
    "s11": 10,  # Recycler
}
# Level of the miscellaneous buildings, the ones the genetic algorithm does not decide, on every planet.
# For the four warehouses it is only the lowest level: the plan raises each one to what its planet needs.
MISC_BUILDING_LEVELS: dict[str, int] = {
    "b6": 0,  # Thermosolar plant
    "b7": 12,  # Development center (the Microsystem accelerator requires level 12)
    "b8": 0,  # Research lab
    "b10": 5,  # Fleet base
    "b11": 0,  # Pig iron warehouse
    "b12": 0,  # Metal warehouse
    "b13": 0,  # Kryptonite warehouse
    "b14": 0,  # Spice warehouse
    "b15": 0,  # Particle cannon
    "b16": 0,  # Particle shield
    "b18": 0,  # Teleporter
    "b19": 0,  # Galaxy scanner
    "b20": 0,  # Tunnel system
    "b21": 0,  # Quantum Array
}
# Exceptions for single planets (roster slot -> building key -> level); slot 1 is the first planet of the roster.
MISC_BUILDING_LEVELS_BY_SLOT: dict[int, dict[str, int]] = {
    1: {"b8": 12},  # one Research lab for the whole empire
}

# --- Warehouses ------------------------------------------------------------------------------------------------
# Every warehouse gets the lowest level that holds this many hours of its planet's own production, and the cost of
# the most expensive thing that the planet still has to pay for: a building level, a ship or, on the planet with
# the Research lab, a research level.
WAREHOUSE_HOURS_OF_PRODUCTION: float = 24.0
# The central planet collects the production of the empire. Its warehouses also hold that many hours of the
# production of all planets together, and none of the four falls below the minimum level.
CENTRAL_PLANET_SLOT: int = 1  # roster slot of the central planet: the home planet
CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL: int = 15  # lowest level of each of its four warehouses

# --- What the genetic algorithm decides ------------------------------------------------------------------------
# Buildings whose level it chooses on every planet, each with the highest level it may choose. A maximum beyond
# the last level of the catalog (100 for the mines and the Fusion plant, 60 for the Weapons factory, 55 for the
# Microsystem accelerator) is cut to that level.
OPTIMIZED_BUILDING_MAX_LEVELS: dict[str, int] = {
    "b1": 100,  # Pig iron mine
    "b2": 100,  # Blast furnace
    "b3": 100,  # Kryptonite mine
    "b4": 100,  # Spice mine
    "b5": 100,  # Fusion plant
    "b9": 100,  # Weapons factory
    "b17": 100,  # Microsystem accelerator
}

# --- Search settings -------------------------------------------------------------------------------------------
CATALOGUE_GRID_SIZE: int = 100_000  # coarse designs tried per planet before its design catalogue is refined
CATALOGUE_PRIORITY_STEPS: int = 4  # priorities between the limits vary in steps of 1/4
HILL_CLIMB_STEP_SIZES: tuple[int, ...] = (1, 2, 4, 8)  # level changes that the hill-climb tries
GA_POPULATION_SIZE: int = 400  # chromosomes per generation
GA_GENERATIONS: int = 300  # generations to evolve
GA_TOURNAMENT_SIZE: int = 3  # chromosomes compared to choose one parent
GA_CROSSOVER_RATE: float = 0.9  # share of children that mix two parents
GA_MUTATION_RATE: float = 0.1  # chance that a planet of a child switches to a random design
GA_ELITE_COUNT: int = 4  # best chromosomes copied unchanged into the next generation
GA_RANDOM_SEED: int = 20261117  # the same seed gives the same result

print_section_data(
    "Section 11.1 data: meta parameters",
    [
        ("TARGET_SHIP_KEY", TARGET_SHIP_KEY),
        ("PRODUCTION_MONTH_DAYS", PRODUCTION_MONTH_DAYS),
        ("RESEARCH_PERCENT_OF_UNIVERSE_MAX", RESEARCH_PERCENT_OF_UNIVERSE_MAX),
        ("MIN_FLEET_PER_PLANET", MIN_FLEET_PER_PLANET),
        ("MISC_BUILDING_LEVELS", MISC_BUILDING_LEVELS),
        ("MISC_BUILDING_LEVELS_BY_SLOT", MISC_BUILDING_LEVELS_BY_SLOT),
        ("WAREHOUSE_HOURS_OF_PRODUCTION", WAREHOUSE_HOURS_OF_PRODUCTION),
        ("CENTRAL_PLANET_SLOT", CENTRAL_PLANET_SLOT),
        ("CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL", CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL),
        ("OPTIMIZED_BUILDING_MAX_LEVELS", OPTIMIZED_BUILDING_MAX_LEVELS),
        (
            "search settings",
            {
                "CATALOGUE_GRID_SIZE": CATALOGUE_GRID_SIZE,
                "CATALOGUE_PRIORITY_STEPS": CATALOGUE_PRIORITY_STEPS,
                "HILL_CLIMB_STEP_SIZES": HILL_CLIMB_STEP_SIZES,
                "GA_POPULATION_SIZE": GA_POPULATION_SIZE,
                "GA_GENERATIONS": GA_GENERATIONS,
                "GA_TOURNAMENT_SIZE": GA_TOURNAMENT_SIZE,
                "GA_CROSSOVER_RATE": GA_CROSSOVER_RATE,
                "GA_MUTATION_RATE": GA_MUTATION_RATE,
                "GA_ELITE_COUNT": GA_ELITE_COUNT,
                "GA_RANDOM_SEED": GA_RANDOM_SEED,
            },
        ),
    ],
)

Section 11.1 data: meta parameters
Printed: 2026-11-17 09:59:32 -0300 | 8 item(s)
[TARGET_SHIP_KEY] str = 's16'
----------------------------------------------------------------------------------------------------
[PRODUCTION_MONTH_DAYS] float = 30.0
----------------------------------------------------------------------------------------------------
[RESEARCH_PERCENT_OF_UNIVERSE_MAX] float = 100.0
----------------------------------------------------------------------------------------------------
[MIN_FLEET_PER_PLANET] dict, len=3
{
  "s2": 10,
  "s10": 10,
  "s11": 10
}
----------------------------------------------------------------------------------------------------
[MISC_BUILDING_LEVELS] dict, len=14
{
  "b6": 0,
  "b7": 12,
  "b8": 0,
  "b10": 5,
  "b11": 6,
  "b12": 6,
  "b13": 6,
  "b14": 6,
  "b15": 0,
  "b16": 0,
  "b18": 0,
  "b19": 0,
  "b20": 0,
  "b21": 0
}
----------------------------------------------------------------------------------------------------
[MISC_BUILDING_L

### 11.2 Planned Research and the Fixed Part of Every Planet

This cell turns the meta parameters into the inputs of the search. `check_optimizer_settings` stops with an error when they contradict the catalog or each other.

**Research.** `plan_research_levels` sets every research to `RESEARCH_PERCENT_OF_UNIVERSE_MAX` percent of its universe maximum, and raises it to your current level and to the level that the plan requires (`PLAN_REQUIRED_RESEARCH`: the prerequisites of the target ship, the fleet, and the buildings of the plan). In the model, Mining technology raises mine output and Energy technology raises energy output; the other researches only unlock ships and buildings.

**Planets.** `build_planet_plans` plans every normal planet of the roster:

- a miscellaneous building gets the highest of its meta-parameter level, the level already built, and the level that the planet's fleet and other miscellaneous buildings require;
- an optimized building may range from the level that those require (`min_level`) to its maximum (`max_level`). It stays at level 0, with the reason in `locked_by`, when the miscellaneous levels do not meet one of its own building prerequisites: the Microsystem accelerator needs Development center 12;
- a warehouse gets no level here, because its level depends on the design of the planet. The plan records its lowest level (`warehouse_min_levels`, with the reason in `warehouse_min_set_by`) and the most of each resource that the planet must be able to store, whatever its design (`capacity_needed`, with the cost that asks for it in `capacity_needed_for`). That is the cost of its fixed buildings and of the ships of its fleet. On the planet with the highest Research lab, it is also the cost of the planned research levels;
- the fields that the fixed buildings leave free are the budget of the optimized buildings and the warehouses (`free_fields`).

Tables and values:

- `WAREHOUSE_KEYS` — the warehouse of each resource;
- `planned_research_df` — per research: your level, the universe maximum, the percentage level, the required level, the planned level, and what set it (`set_by`: `current level`, `universe maximum`, `prerequisite` or `not needed`);
- `PLANNED_RESEARCH_LEVELS` — `{research key: planned level}`; `PLAN_ACCOUNT_CONTEXT` — the account context of Section 6 with these levels;
- `plan_planets_df` — per planet: whether it is the central planet and whether it pays for the research, fields, fields of the fixed buildings, free fields, the fields that the minimum levels of the optimized buildings and the lowest warehouses need, Solar cells counted, and the missing ships of the minimum fleet with their cost;
- `plan_building_levels_df` — per planet and building: fixed, optimized or warehouse, the level built today, and the level range of the plan;
- `PLANET_PLANS` — the same plan as one record per planet, the input of the model.

In [91]:
# The warehouse of each resource, in `RESOURCE_KEYS` order.
WAREHOUSE_KEYS: tuple[str, ...] = tuple(
    str(RESOURCE_METADATA[resource]["warehouse_building_key"]) for resource in RESOURCE_KEYS
)


def same_planet_building_requirements(key: str, level: int = 0) -> dict[str, int]:
    """Building levels a planet needs before it can hold `key`, following building prerequisites only.

    `key` is a ship, a defense or a building; `level > 0` also demands that level of the building itself.
    Research prerequisites belong to the account, not to a planet: see `research_requirements`.
    """
    required: dict[str, int] = {}
    if level > 0 and TECH_GRAPH.nodes[key]["category"] == "buildings":
        required[key] = int(level)
    pending = [key]
    while pending:
        for source, source_level in TECH_GRAPH.prereqs_of[pending.pop()]:
            if TECH_GRAPH.nodes[source]["category"] == "buildings" and source_level > required.get(source, 0):
                required[source] = int(source_level)
                pending.append(source)
    return required


def research_requirements(keys: Iterable[str]) -> dict[str, int]:
    """Highest research level that any of `keys` (ships, defenses or buildings) needs, directly or indirectly."""
    required: dict[str, int] = {}
    for key in keys:
        for node, level in TECH_GRAPH.transitive_requirements(key).items():
            if TECH_GRAPH.nodes[node]["category"] == "research":
                required[node] = max(required.get(node, 0), int(level))
    return required


def catalog_top_level(key: str) -> int:
    """Last level of building `key` that the catalog lists."""
    info = game_data.BUILDING_BY_ID[int(key[1:])]
    return int(info.get("max_table_level") or len(info.get("levels") or {}))


def warehouse_level_for(amount: float) -> int:
    """Lowest warehouse level whose capacity holds `amount` units of a resource."""
    level = 0
    while warehouse_capacity(level) < amount:
        level += 1
    return level


def largest_costs(costs: Iterable[tuple[str, dict[str, Any]]]) -> tuple[dict[str, float], dict[str, str]]:
    """Per resource, the largest amount that any of the labelled `costs` asks for, and the label of that cost."""
    amounts = {resource: 0.0 for resource in RESOURCE_KEYS}
    labels = {resource: "" for resource in RESOURCE_KEYS}
    for label, cost in costs:
        for resource in RESOURCE_KEYS:
            amount = float(cost.get(resource, 0) or 0)
            if amount > amounts[resource]:
                amounts[resource], labels[resource] = amount, label
    return amounts, labels


def building_upgrade_costs(key: str, from_level: int, to_level: int) -> list[tuple[str, dict[str, Any]]]:
    """`(label, cost)` of every level of building `key` above `from_level`, up to `to_level`."""
    return [
        (f"{key} L{level}", game_data.building_cost(int(key[1:]), level))
        for level in range(from_level + 1, to_level + 1)
    ]


def research_upgrade_costs(
    own_levels: dict[str, int], planned_levels: dict[str, int]
) -> list[tuple[str, dict[str, Any]]]:
    """`(label, cost)` of every research level between the current levels and the planned ones."""
    return [
        (f"{key} L{level}", ((game_data.RESEARCH_BY_ID[key].get("levels") or {}).get(level) or {}).get("cost") or {})
        for key, planned_level in planned_levels.items()
        for level in range(max(0, int(own_levels.get(key, 0) or 0)) + 1, planned_level + 1)
    ]


def check_optimizer_settings(
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    misc_levels_by_slot: dict[int, dict[str, int]],
    min_fleet: dict[str, int],
    target_ship_key: str,
    warehouse_hours: float,
    central_min_warehouse_level: int,
) -> None:
    """Raise `ValueError` when the meta parameters contradict the catalog or each other."""
    building_keys = {f"b{building_id}" for building_id in game_data.BUILDING_BY_ID}
    optimized, fixed = set(optimized_max_levels), set(misc_levels)
    if (optimized | fixed) != building_keys or (optimized & fixed):
        raise ValueError(
            "OPTIMIZED_BUILDING_MAX_LEVELS and MISC_BUILDING_LEVELS must list every building exactly once: "
            f"missing {sorted(building_keys - optimized - fixed)}, "
            f"unknown {sorted((optimized | fixed) - building_keys)}, in both {sorted(optimized & fixed)}"
        )
    if optimized & set(WAREHOUSE_KEYS):
        raise ValueError(
            f"The warehouses {sorted(optimized & set(WAREHOUSE_KEYS))} belong in MISC_BUILDING_LEVELS: "
            "the plan sets their level to what each planet needs"
        )
    for slot, levels in misc_levels_by_slot.items():
        if not set(levels) <= fixed:
            raise ValueError(
                f"MISC_BUILDING_LEVELS_BY_SLOT[{slot}] sets {sorted(set(levels) - fixed)}, "
                "which MISC_BUILDING_LEVELS does not list"
            )
    if target_ship_key not in SHIP_SLUG_BY_KEY or not set(min_fleet) <= set(SHIP_SLUG_BY_KEY):
        raise ValueError(
            f"TARGET_SHIP_KEY and MIN_FLEET_PER_PLANET need ship keys: {sorted({target_ship_key, *min_fleet})}"
        )
    counts = [
        *optimized_max_levels.values(),
        *misc_levels.values(),
        *(level for levels in misc_levels_by_slot.values() for level in levels.values()),
        *min_fleet.values(),
        central_min_warehouse_level,
    ]
    if any(int(count) != count or count < 0 for count in counts):
        raise ValueError("Building levels and fleet sizes must be whole numbers of 0 or more")
    if not 0.0 <= warehouse_hours < math.inf:
        raise ValueError("WAREHOUSE_HOURS_OF_PRODUCTION must be a number of hours of 0 or more")
    for key in optimized_max_levels:
        linked = sorted((set(same_planet_building_requirements(key)) & optimized) - {key})
        if linked:
            raise ValueError(
                f"{key} needs {linked} on the same planet, and both are optimized: "
                "move one of them to MISC_BUILDING_LEVELS"
            )


def plan_research_levels(
    universe_max_levels: dict[str, int],
    own_levels: dict[str, int],
    required_levels: dict[str, int],
    percent_of_universe_max: float,
) -> pd.DataFrame:
    """Research levels of the plan: a share of the universe maximum, never below the current or required level."""
    rows: list[dict[str, Any]] = []
    for key, info in research_catalog():
        catalog_max = int(info.get("max_table_level") or len(info.get("levels") or {}))
        own_level = max(0, int(own_levels.get(key, 0) or 0))
        universe_max = int(universe_max_levels.get(key, 0))
        # Half a level rounds up: 50 % of level 5 is level 3.
        percent_level = min(catalog_max, int(math.floor(universe_max * float(percent_of_universe_max) / 100.0 + 0.5)))
        required_level = int(required_levels.get(key, 0))
        planned_level = max(own_level, percent_level, required_level)
        if planned_level == 0:
            set_by = "not needed"
        elif planned_level == own_level:
            set_by = "current level"
        elif planned_level == percent_level:
            set_by = "universe maximum"
        else:
            set_by = "prerequisite"
        rows.append(
            {
                "key": key,
                "name": str(info["name"]),
                "own_level": own_level,
                "universe_max_level": universe_max,
                "percent_level": percent_level,
                "required_level": required_level,
                "planned_level": planned_level,
                "set_by": set_by,
            }
        )
    return pd.DataFrame(rows)


def context_with_research(context: dict[str, Any], research_levels: dict[str, int]) -> dict[str, Any]:
    """Copy of an account context whose research entries (`r1_mining_tech`, ...) hold `research_levels`."""
    return {name: research_levels.get(name.partition("_")[0], value) for name, value in context.items()}


def build_planet_plan(
    context: dict[str, Any],
    planet: PlanetState,
    *,
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    min_fleet: dict[str, int],
    central_min_warehouse_level: int | None = None,
) -> dict[str, Any]:
    """Fixed part and search range of one planet.

    A miscellaneous building gets the highest of its meta-parameter level, the level already built, and the
    level that the planet's fleet and other miscellaneous buildings require. An optimized building ranges
    from the level those require up to its maximum; it stays at 0 (`locked_by`) when the miscellaneous
    levels do not meet one of its own building prerequisites.

    A warehouse is neither: the model sets its level for every design. The plan only gives its lowest level
    (`warehouse_min_levels`: the meta-parameter level, the level already built and, on the central planet,
    `central_min_warehouse_level`) and the most of each resource that the planet must be able to store to
    pay for its fixed buildings and its fleet (`capacity_needed`).
    """
    built = {key: max(0, int(planet.buildings.get(key, 0) or 0)) for key in (*misc_levels, *optimized_max_levels)}
    wanted = {key: max(int(level), built[key]) for key, level in misc_levels.items()}
    required: dict[str, int] = {}
    for needs in (
        *(same_planet_building_requirements(ship) for ship, count in min_fleet.items() if count > 0),
        *(same_planet_building_requirements(key, level) for key, level in wanted.items() if level > 0),
    ):
        for key, level in needs.items():
            required[key] = max(required.get(key, 0), level)
    fixed_levels = {
        key: max(level, required.get(key, 0)) for key, level in wanted.items() if key not in WAREHOUSE_KEYS
    }

    min_levels: dict[str, int] = {}
    max_levels: dict[str, int] = {}
    locked_by: dict[str, str] = {}
    for key, max_level in optimized_max_levels.items():
        unmet = [
            f"{source} L{level}"
            for source, level in same_planet_building_requirements(key).items()
            if fixed_levels.get(source, level) < level
        ]
        min_levels[key] = required.get(key, 0)
        max_levels[key] = max(min_levels[key], 0 if unmet else min(int(max_level), catalog_top_level(key)))
        if unmet:
            locked_by[key] = ", ".join(unmet)

    # The lowest level of each warehouse, and what sets it; between equal levels the first reason is named.
    warehouse_min_levels: dict[str, int] = {}
    warehouse_min_set_by: dict[str, str] = {}
    for key in WAREHOUSE_KEYS:
        level, reason = max(
            [
                (built[key], "level already built"),
                (int(central_min_warehouse_level or 0), "central planet minimum"),
                (int(misc_levels[key]), "MISC_BUILDING_LEVELS"),
            ],
            key=lambda candidate: candidate[0],
        )
        warehouse_min_levels[key], warehouse_min_set_by[key] = level, reason if level > 0 else "no minimum"

    stationed = {ship: max(0, int(planet.ships.get(ship, 0) or 0)) for ship in {*min_fleet, "s9"}}
    fleet_missing = {ship: max(0, int(count) - stationed[ship]) for ship, count in min_fleet.items()}
    capacity_needed, capacity_needed_for = largest_costs(
        [
            *(cost for key, level in fixed_levels.items() for cost in building_upgrade_costs(key, built[key], level)),
            *(
                (ship, game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost") or {})
                for ship, missing in fleet_missing.items()
                if missing > 0
            ),
        ]
    )
    fixed_fields = sum(fixed_levels.values())
    return {
        **planet_identity(context),
        "temperature_c": int(planet.temperature),
        "fields_max": int(planet.fields_max),
        "fixed_fields": fixed_fields,
        "free_fields": int(planet.fields_max) - fixed_fields,
        "built_levels": built,
        "fixed_levels": fixed_levels,
        "min_levels": min_levels,
        "max_levels": max_levels,
        "locked_by": locked_by,
        "warehouse_min_levels": warehouse_min_levels,
        "warehouse_min_set_by": warehouse_min_set_by,
        "capacity_needed": capacity_needed,
        "capacity_needed_for": capacity_needed_for,
        "is_central_planet": central_min_warehouse_level is not None,
        "pays_for_research": False,
        "solar_cells": max(int(min_fleet.get("s9", 0)), stationed["s9"]),
        "fleet_missing": fleet_missing,
        "fleet_cost_spice_eq": float(
            sum(
                count * game_data.spice_equivalent(full_cost(game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost")))
                for ship, count in fleet_missing.items()
            )
        ),
    }


def build_planet_plans(
    planets: list[PlanetState],
    contexts: list[dict[str, Any]],
    *,
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    misc_levels_by_slot: dict[int, dict[str, int]],
    min_fleet: dict[str, int],
    central_planet_slot: int,
    central_min_warehouse_level: int,
    research_costs: list[tuple[str, dict[str, Any]]],
) -> list[dict[str, Any]]:
    """Plan every normal planet of the roster; asteroids take no part in the optimization.

    The planet in roster slot `central_planet_slot` is the central planet. Research is paid for on the planet
    with the highest Research lab, the first of them between equals: its `capacity_needed` also covers the
    `research_costs`.
    """
    plans = [
        build_planet_plan(
            context,
            planet,
            optimized_max_levels=optimized_max_levels,
            misc_levels={**misc_levels, **misc_levels_by_slot.get(context["slot_index"], {})},
            min_fleet=min_fleet,
            central_min_warehouse_level=(
                central_min_warehouse_level if context["slot_index"] == central_planet_slot else None
            ),
        )
        for planet, context in zip(planets, contexts)
        if planet.planet_type == "normal"
    ]
    if not any(plan["is_central_planet"] for plan in plans):
        raise ValueError(
            f"CENTRAL_PLANET_SLOT {central_planet_slot} is not a normal planet of the roster: "
            f"choose one of the slots {[plan['slot_index'] for plan in plans]}"
        )
    lab_planet = max(plans, key=lambda plan: plan["fixed_levels"].get("b8", 0))
    if lab_planet["fixed_levels"].get("b8", 0) > 0:
        lab_planet["capacity_needed"], lab_planet["capacity_needed_for"] = largest_costs(
            [
                *(
                    (lab_planet["capacity_needed_for"][resource], {resource: lab_planet["capacity_needed"][resource]})
                    for resource in RESOURCE_KEYS
                ),
                *research_costs,
            ]
        )
        lab_planet["pays_for_research"] = True
    return plans


def planet_plan_level_rows(plan: dict[str, Any]) -> list[dict[str, Any]]:
    """Rows of `plan_building_levels_df` for one planet: the level range of every building."""
    rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        key = f"b{building_id}"
        if key in plan["min_levels"]:
            role, min_level, max_level = "optimized", plan["min_levels"][key], plan["max_levels"][key]
        elif key in plan["warehouse_min_levels"]:
            role, min_level, max_level = "warehouse", plan["warehouse_min_levels"][key], catalog_top_level(key)
        else:
            role, min_level, max_level = "fixed", plan["fixed_levels"][key], plan["fixed_levels"][key]
        rows.append(
            {
                "slot_index": plan["slot_index"],
                "coords": plan["coords"],
                "key": key,
                "name": str(game_data.BUILDING_BY_ID[building_id]["name"]),
                "role": role,
                "built_level": plan["built_levels"][key],
                "min_level": min_level,
                "max_level": max_level,
                "locked_by": plan["locked_by"].get(key, ""),
            }
        )
    return rows


check_optimizer_settings(
    OPTIMIZED_BUILDING_MAX_LEVELS,
    MISC_BUILDING_LEVELS,
    MISC_BUILDING_LEVELS_BY_SLOT,
    MIN_FLEET_PER_PLANET,
    TARGET_SHIP_KEY,
    WAREHOUSE_HOURS_OF_PRODUCTION,
    CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL,
)

# Research: whatever the target ship, the fleet and the buildings of the plan need is the floor of the plan.
PLAN_REQUIRED_RESEARCH: dict[str, int] = research_requirements(
    [
        TARGET_SHIP_KEY,
        *(ship for ship, count in MIN_FLEET_PER_PLANET.items() if count > 0),
        *OPTIMIZED_BUILDING_MAX_LEVELS,
        *(key for key, level in MISC_BUILDING_LEVELS.items() if level > 0),
        *(key for levels in MISC_BUILDING_LEVELS_BY_SLOT.values() for key, level in levels.items() if level > 0),
    ]
)
planned_research_df = plan_research_levels(
    UNIVERSE_RESEARCH_MAX_LEVELS, colony_researches, PLAN_REQUIRED_RESEARCH, RESEARCH_PERCENT_OF_UNIVERSE_MAX
)
PLANNED_RESEARCH_LEVELS: dict[str, int] = {
    str(key): int(level) for key, level in zip(planned_research_df["key"], planned_research_df["planned_level"])
}
PLAN_ACCOUNT_CONTEXT: dict[str, Any] = context_with_research(GAME_PARAM_CONTEXT, PLANNED_RESEARCH_LEVELS)

# Planets: the fixed buildings, the level range of every optimized building, the lowest warehouses, and the
# fields left for the optimized buildings and the warehouses.
PLANET_PLANS: list[dict[str, Any]] = build_planet_plans(
    empire_planet_states,
    PLANET_PARAM_CONTEXTS,
    optimized_max_levels=OPTIMIZED_BUILDING_MAX_LEVELS,
    misc_levels=MISC_BUILDING_LEVELS,
    misc_levels_by_slot=MISC_BUILDING_LEVELS_BY_SLOT,
    min_fleet=MIN_FLEET_PER_PLANET,
    central_planet_slot=CENTRAL_PLANET_SLOT,
    central_min_warehouse_level=CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL,
    research_costs=research_upgrade_costs(colony_researches, PLANNED_RESEARCH_LEVELS),
)
plan_planets_df = pd.DataFrame(
    [
        {
            **{
                column: plan[column]
                for column in (
                    "slot_index",
                    "planet_id",
                    "coords",
                    "planet_status",
                    "is_central_planet",
                    "pays_for_research",
                    "temperature_c",
                    "fields_max",
                    "fixed_fields",
                    "free_fields",
                )
            },
            "min_optimized_fields": sum(plan["min_levels"].values()),
            "min_warehouse_fields": sum(
                max(plan["warehouse_min_levels"][key], warehouse_level_for(plan["capacity_needed"][resource]))
                for resource, key in zip(RESOURCE_KEYS, WAREHOUSE_KEYS)
            ),
            "solar_cells": plan["solar_cells"],
            "fleet_units_missing": sum(plan["fleet_missing"].values()),
            "fleet_cost_spice_eq": round(plan["fleet_cost_spice_eq"], 2),
        }
        for plan in PLANET_PLANS
    ]
)
plan_building_levels_df = pd.DataFrame([row for plan in PLANET_PLANS for row in planet_plan_level_rows(plan)])

print_section_data(
    "Section 11.2 data: planned research and the fixed part of every planet",
    [
        ("WAREHOUSE_KEYS", WAREHOUSE_KEYS),
        ("PLAN_REQUIRED_RESEARCH", PLAN_REQUIRED_RESEARCH),
        ("planned_research_df", planned_research_df),
        ("PLANNED_RESEARCH_LEVELS", PLANNED_RESEARCH_LEVELS),
        ("PLAN_ACCOUNT_CONTEXT", PLAN_ACCOUNT_CONTEXT),
        ("plan_planets_df", plan_planets_df),
        ("plan_building_levels_df", plan_building_levels_df),
        ("PLANET_PLANS", PLANET_PLANS),
    ],
)

Section 11.2 data: planned research and the fixed part of every planet
Printed: 2026-11-17 09:59:32 -0300 | 7 item(s)
[PLAN_REQUIRED_RESEARCH] dict, len=8
{
  "r2": 10,
  "r3": 8,
  "r13": 10,
  "r6": 10,
  "r15": 2,
  "r4": 6,
  "r7": 2,
  "r5": 4
}
----------------------------------------------------------------------------------------------------
[planned_research_df] pandas.DataFrame, shape=(15, 8)
key                name  own_level  universe_max_level  percent_level  required_level  planned_level           set_by
 r1   Mining technology          9                  30             30               0             30 universe maximum
 r2   Energy technology          4                  28             28              10             28 universe maximum
 r3     Engine research          6                  20             20               8             20 universe maximum
 r4       Rocket engine          8                  26             26               6             26 universe maximum
 r5 

### 11.3 Production Model

`EmpireProductionModel` computes, for any building levels, what each planet adds to each limit. It is built once from `PLANET_PLANS` and then evaluates thousands of designs at a time with `numpy`. A **design** is one planet's levels of the optimized buildings; a **configuration** is one design per planet.

For a planet and a design, `simulate` works through the following steps, with the per-level numbers of Section 6 (`building_level_economy`) at the planned research levels:

1. Sum the hourly flows of the optimized and the fixed buildings: output per resource, Pig iron and Spice used as feedstock, energy produced and used. The Solar cells of the planet's fleet add their energy.
2. `power_ratio` = energy produced / energy used, at most 1. Every mine and the Blast furnace run at that share.
3. `smelting_ratio` = Pig iron mined / Pig iron the Blast furnace wants, at most 1. The furnace makes that share of its Metal.
4. The planet's net output per hour follows. Divided by the ship's cost, and counted over a month, it is the number of ships that the planet's Metal, Kryptonite or Spice pays for.
5. The shipyard finishes `month / (base time / (max(1, b9) * 2^b17 * speed))` ships, or none when the planet lacks a building that the ship requires.
6. The warehouses get the lowest levels that meet the rules of Section 11: the hours of the planet's net output, the most expensive level that the design still has to build (`upgrade_cost_table`), a target ship where the shipyard builds it, what the plan asks the planet to store, and the reserve of the central planet. A warehouse level has a cost of its own, so the levels are raised until the warehouses hold that as well.
7. The design fits the planet (`planet_fits`) when its levels and its warehouse levels together stay within the planet's free fields.

`planet_limits[n, planet, limit]` holds the numbers of steps 4 and 5, in ships per month. `ships_per_month` adds them up over the planets and takes the smallest sum, because resources are pooled. `is_feasible` checks the level ranges and the fields of a design.

The model stops with an error when a planet has no room for its minimum levels and their warehouses. `with_central_reserve` returns a copy of the model whose central planet keeps a given capacity free for the production of the empire; Section 11.4 uses it. `PRODUCTION_MODEL` itself reserves nothing yet.

Planets with identical inputs, such as the generic planets, are marked as twins (`twin_planets`), so that Section 11.4 searches designs for one of them only. The central planet has no twin.

In [92]:
PRODUCTION_FLOW_KEYS: tuple[str, ...] = (
    *RESOURCE_KEYS,
    "pig_iron_used",
    "spice_used",
    "energy_produced",
    "energy_used",
)


def building_level_flows(building_key: str, level: int, context: dict[str, Any]) -> list[float]:
    """Hourly flows of one building level at full power on the context's planet, in `PRODUCTION_FLOW_KEYS` order."""
    economy = building_level_economy(int(building_key[1:]), level, context)
    output = {produced_resource(building_key): economy["production"]}
    return [
        *(float(output.get(resource, 0.0)) for resource in RESOURCE_KEYS),
        economy["pig_iron_used"],
        economy["spice_used"],
        max(0.0, economy["energy"]),
        max(0.0, -economy["energy"]),
    ]


def upgrade_cost_table(key: str, built_level: int, top_level: int) -> np.ndarray:
    """`table[level, resource]`: the most of each resource that a single level of building `key` costs on the way
    from `built_level` to `level`. It is 0 up to `built_level`, where nothing is left to build."""
    table = np.zeros((top_level + 1, len(RESOURCE_KEYS)), dtype=np.float64)
    for level in range(built_level + 1, top_level + 1):
        cost = full_cost(game_data.building_cost(int(key[1:]), level))
        table[level] = np.maximum(table[level - 1], [float(cost[resource]) for resource in RESOURCE_KEYS])
    return table


def upgrade_cost_tables(planet_plans: list[dict[str, Any]], keys: tuple[str, ...], top_level: int) -> np.ndarray:
    """`tables[planet, building, level, resource]`: the `upgrade_cost_table` of each of `keys` on every planet."""
    tables = {
        (key, built_level): upgrade_cost_table(key, built_level, top_level)
        for key in keys
        for built_level in {plan["built_levels"][key] for plan in planet_plans}
    }
    return np.array([[tables[key, plan["built_levels"][key]] for key in keys] for plan in planet_plans])


class EmpireProductionModel:
    """Steady-state output of an empire: what its planets add to each limit of the target ship's production.

    A design is one planet's levels of the optimized buildings, in `optimized_keys` order. The methods take
    many designs at once as `levels[n, planet, building]`; `planets` lists the planet of every column and
    defaults to all planets in order, which makes each `levels[n]` a configuration of the whole empire.
    The warehouses are not part of a design: the model gives every design the lowest warehouses it needs.
    """

    def __init__(
        self,
        planet_plans: list[dict[str, Any]],
        planet_contexts: list[dict[str, Any]],
        *,
        optimized_keys: tuple[str, ...],
        target_ship_key: str,
        month_days: float,
        warehouse_hours: float,
    ) -> None:
        self.optimized_keys = tuple(optimized_keys)
        self.target_ship_key = target_ship_key
        self.building_index = {key: index for index, key in enumerate(self.optimized_keys)}
        self.planet_names = [f"{plan['coords']} (slot {plan['slot_index']})" for plan in planet_plans]
        self.min_levels = np.array(
            [[plan["min_levels"][key] for key in self.optimized_keys] for plan in planet_plans], dtype=np.int64
        )
        self.max_levels = np.array(
            [[plan["max_levels"][key] for key in self.optimized_keys] for plan in planet_plans], dtype=np.int64
        )
        self.fields_max = np.array([plan["fields_max"] for plan in planet_plans], dtype=np.int64)
        # free_fields[planet]: the fields that the fixed buildings leave to the optimized ones and the warehouses.
        self.free_fields = np.array([plan["free_fields"] for plan in planet_plans], dtype=np.int64)
        self.fixed_levels = {
            key: np.array([plan["fixed_levels"][key] for plan in planet_plans], dtype=np.int64)
            for key in planet_plans[0]["fixed_levels"]
        }

        # flow_tables[flow, planet, building, level]: what each optimized building adds at each level.
        top_level = int(self.max_levels.max())
        self.flow_tables = np.array(
            [
                [
                    [building_level_flows(key, level, context) for level in range(top_level + 1)]
                    for key in self.optimized_keys
                ]
                for context in planet_contexts
            ],
            dtype=np.float64,
        ).transpose(3, 0, 1, 2)
        # fixed_flows[flow, planet]: the buildings at fixed levels and the Solar cells of the planet's fleet.
        self.fixed_flows = np.zeros((len(PRODUCTION_FLOW_KEYS), len(planet_plans)), dtype=np.float64)
        for planet, (plan, context) in enumerate(zip(planet_plans, planet_contexts)):
            for key, level in plan["fixed_levels"].items():
                self.fixed_flows[:, planet] += building_level_flows(key, level, context)
            self.fixed_flows[PRODUCTION_FLOW_KEYS.index("energy_produced"), planet] += (
                plan["solar_cells"]
                * solar_cell_energy_per_unit(context["planet_temperature_c"])
                * technology_multiplier("r2", context["r2_energy_tech"])
            )

        # Warehouses. warehouse_capacities[level] is what a warehouse of that level holds.
        self.warehouse_hours = float(warehouse_hours)
        self.top_warehouse_level = min(catalog_top_level(key) for key in WAREHOUSE_KEYS)
        self.warehouse_capacities = np.array(
            [warehouse_capacity(level) for level in range(self.top_warehouse_level + 1)], dtype=np.float64
        )
        self.min_warehouse_levels = np.array(
            [[plan["warehouse_min_levels"][key] for key in WAREHOUSE_KEYS] for plan in planet_plans], dtype=np.int64
        )
        # capacity_needed[planet, resource]: what the plan asks the planet to store, whatever its design.
        self.capacity_needed = np.array(
            [[plan["capacity_needed"][resource] for resource in RESOURCE_KEYS] for plan in planet_plans],
            dtype=np.float64,
        )
        # cost_tables[planet, building, level, resource] and warehouse_cost_tables[planet, warehouse, level, resource]:
        # the most that a single level still to build costs, for the optimized buildings and for the warehouses.
        self.cost_tables = upgrade_cost_tables(planet_plans, self.optimized_keys, top_level)
        self.warehouse_cost_tables = upgrade_cost_tables(planet_plans, WAREHOUSE_KEYS, self.top_warehouse_level)
        # The central planet also keeps central_reserve[resource] free for the production of the whole empire.
        self.central_planet = next(
            (planet for planet, plan in enumerate(planet_plans) if plan["is_central_planet"]), -1
        )
        self.central_reserve = np.zeros(len(RESOURCE_KEYS), dtype=np.float64)

        # The limits: one per resource the ship costs, then the shipyards.
        ship = game_data.SHIPS[SHIP_SLUG_BY_KEY[target_ship_key]]
        cost = full_cost(ship.get("base_cost"))
        self.cost_resources = tuple(resource for resource in RESOURCE_KEYS if float(cost[resource]) > 0.0)
        self.ship_cost = np.array([float(cost[resource]) for resource in self.cost_resources], dtype=np.float64)
        # ship_capacity[resource]: what a planet must store to pay for one target ship.
        self.ship_capacity = np.array([float(cost[resource]) for resource in RESOURCE_KEYS], dtype=np.float64)
        self.limit_names = (*self.cost_resources, "shipyards")
        self.target_requirements = same_planet_building_requirements(target_ship_key)
        self.hours_per_month = 24.0 * float(month_days)
        # shipyard_ships[weapons factory level, microsystem level]: target ships one shipyard finishes per month.
        factory_levels = max(top_level, int(self.fixed_levels.get("b9", np.zeros(1)).max())) + 1
        microsystem_levels = max(top_level, int(self.fixed_levels.get("b17", np.zeros(1)).max())) + 1
        speedup = float(planet_contexts[0]["speedup_factor"])
        seconds_per_month = 3600.0 * self.hours_per_month
        self.shipyard_ships = np.array(
            [
                [
                    seconds_per_month * shipyard_time_divisor(factory, microsystem, speedup)
                    / float(ship["base_build_time_s"])
                    for microsystem in range(microsystem_levels)
                ]
                for factory in range(factory_levels)
            ],
            dtype=np.float64,
        )

        # Planets with identical inputs, such as the generic planets, can share their results:
        # twin_planets[planet] is the first planet that equals it. The central planet has no twin.
        signatures = [
            (
                self.flow_tables[:, planet].tobytes(),
                self.fixed_flows[:, planet].tobytes(),
                self.min_levels[planet].tobytes(),
                self.max_levels[planet].tobytes(),
                int(self.free_fields[planet]),
                tuple(int(levels[planet]) for levels in self.fixed_levels.values()),
                self.min_warehouse_levels[planet].tobytes(),
                self.capacity_needed[planet].tobytes(),
                self.cost_tables[planet].tobytes(),
                self.warehouse_cost_tables[planet].tobytes(),
                planet == self.central_planet,
            )
            for planet in range(len(planet_plans))
        ]
        self.twin_planets = [signatures.index(signature) for signature in signatures]
        self.check_minimum_designs()

    def check_minimum_designs(self) -> None:
        """Raise `ValueError` when a planet has no room for its minimum levels and the warehouses they need."""
        outcome = self.simulate(self.min_levels[None])
        overfull = np.flatnonzero(~outcome["planet_fits"][0])
        if len(overfull) > 0:
            planet = int(overfull[0])
            if (outcome["planet_warehouse_levels"][0, planet] > self.top_warehouse_level).any():
                raise ValueError(
                    f"Planet {self.planet_names[planet]} would need a warehouse beyond level "
                    f"{self.top_warehouse_level}, the last level of the catalog: lower WAREHOUSE_HOURS_OF_PRODUCTION"
                )
            fixed_fields = int(self.fields_max[planet] - self.free_fields[planet])
            raise ValueError(
                f"Planet {self.planet_names[planet]} has {self.fields_max[planet]} fields, but its fixed buildings, "
                "the minimum levels of the optimized ones and their warehouses need "
                f"{fixed_fields + int(outcome['planet_fields_needed'][0, planet])}: lower MISC_BUILDING_LEVELS, "
                "CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL or WAREHOUSE_HOURS_OF_PRODUCTION"
            )

    def with_central_reserve(self, capacity: np.ndarray) -> "EmpireProductionModel":
        """A copy of the model whose central planet keeps `capacity[resource]` free for the empire's production.

        The copy shares the tables of the model, which never change.
        """
        model = copy.copy(self)
        model.central_reserve = np.asarray(capacity, dtype=np.float64)
        model.check_minimum_designs()
        return model

    def planet_index(self, planets: Iterable[int] | None) -> np.ndarray:
        """The planet of every column of `levels[n, planet, building]`: `planets`, or all planets in order."""
        return np.arange(len(self.free_fields)) if planets is None else np.asarray(list(planets), dtype=np.int64)

    def building_levels(self, key: str, levels: np.ndarray, planets: np.ndarray) -> np.ndarray:
        """Level of building `key` for every design, whether the building is optimized or fixed."""
        if key in self.building_index:
            return levels[:, :, self.building_index[key]]
        return np.broadcast_to(self.fixed_levels[key][planets], levels.shape[:2])

    def warehouse_levels_for(self, capacity: np.ndarray) -> np.ndarray:
        """Lowest warehouse level that holds each amount of `capacity`; one above the top level when none does."""
        return np.searchsorted(self.warehouse_capacities, capacity)

    def simulate(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> dict[str, np.ndarray]:
        """Evaluate designs `levels[n, planet, building]`; every result is indexed `[n, planet]` first.

        `planet_limits[n, planet, limit]` is what the planet adds to each limit, in target ships per month:
        the ships its net output of each cost resource pays for, and the ships its shipyard can finish.
        `planet_warehouse_levels[n, planet, resource]` are the warehouses that the design needs, and
        `planet_fits[n, planet]` says whether the design and those warehouses fit the planet's free fields.
        """
        planets = self.planet_index(planets)
        planet_axis = planets[None, :, None]
        building_axis = np.arange(levels.shape[2])[None, None, :]
        totals = (
            self.fixed_flows[:, None, planets]
            + self.flow_tables[:, planet_axis, building_axis, levels].sum(axis=3)
        )
        flows = dict(zip(PRODUCTION_FLOW_KEYS, totals))

        # A planet short of energy runs every mine at the share of the energy it has.
        power_ratio = np.ones_like(flows["energy_used"])
        np.divide(flows["energy_produced"], flows["energy_used"], out=power_ratio, where=flows["energy_used"] > 0.0)
        power_ratio = np.minimum(1.0, power_ratio)
        # The Blast furnace smelts the planet's own Pig iron, 2 per Metal, and idles for the part it lacks.
        pig_iron_mined = flows["pig_iron"] * power_ratio
        pig_iron_wanted = flows["pig_iron_used"] * power_ratio
        smelting_ratio = np.ones_like(pig_iron_wanted)
        np.divide(pig_iron_mined, pig_iron_wanted, out=smelting_ratio, where=pig_iron_wanted > 0.0)
        smelting_ratio = np.minimum(1.0, smelting_ratio)
        net = {
            "pig_iron": pig_iron_mined - pig_iron_wanted * smelting_ratio,
            "metal": flows["metal"] * power_ratio * smelting_ratio,
            "kryptonite": flows["kryptonite"] * power_ratio,
            "spice": flows["spice"] * power_ratio - flows["spice_used"],
        }
        net_per_hour = np.stack([net[resource] for resource in RESOURCE_KEYS], axis=2)

        # A shipyard builds the ship only where the ship's building prerequisites are met.
        can_build = np.ones(levels.shape[:2], dtype=bool)
        for key, required_level in self.target_requirements.items():
            can_build &= self.building_levels(key, levels, planets) >= required_level
        shipyard_ships = np.where(
            can_build,
            self.shipyard_ships[
                self.building_levels("b9", levels, planets), self.building_levels("b17", levels, planets)
            ],
            0.0,
        )
        planet_limits = np.stack(
            [
                *(
                    net[resource] * self.hours_per_month / cost
                    for resource, cost in zip(self.cost_resources, self.ship_cost)
                ),
                shipyard_ships,
            ],
            axis=2,
        )

        # Warehouses. Each one holds `warehouse_hours` of the planet's own net output, the most expensive level
        # that the design still has to build, one target ship where the shipyard builds it, what the plan asks
        # the planet to store and, on the central planet, the reserve for the empire.
        capacity = np.maximum(self.warehouse_hours * np.maximum(net_per_hour, 0.0), self.capacity_needed[planets])
        capacity = np.maximum(capacity, self.cost_tables[planet_axis, building_axis, levels].max(axis=2))
        capacity = np.maximum(capacity, np.where(can_build[:, :, None], self.ship_capacity, 0.0))
        capacity = np.maximum(
            capacity, np.where((planets == self.central_planet)[:, None], self.central_reserve, 0.0)
        )
        warehouses = np.maximum(self.warehouse_levels_for(capacity), self.min_warehouse_levels[planets])
        # A warehouse level costs resources too, and the warehouses must hold that cost: raise them until they do.
        warehouse_axis = np.arange(len(WAREHOUSE_KEYS))[None, None, :]
        while True:
            own_cost = self.warehouse_cost_tables[
                planet_axis, warehouse_axis, np.minimum(warehouses, self.top_warehouse_level)
            ].max(axis=2)
            raised = np.maximum(warehouses, self.warehouse_levels_for(own_cost))
            if (raised == warehouses).all():
                break
            warehouses = raised
        fields_needed = levels.sum(axis=2) + warehouses.sum(axis=2)
        return {
            "planet_limits": planet_limits,
            "planet_net_per_hour": net_per_hour,
            "planet_energy_produced": flows["energy_produced"],
            "planet_energy_used": flows["energy_used"],
            "planet_power_ratio": power_ratio,
            "planet_smelting_ratio": smelting_ratio,
            "planet_warehouse_levels": warehouses,
            "planet_fields_needed": fields_needed,
            "planet_fits": (fields_needed <= self.free_fields[planets])
            & (warehouses <= self.top_warehouse_level).all(axis=2),
        }

    def ships_per_month(self, levels: np.ndarray) -> np.ndarray:
        """Target ships per month of whole-empire configurations `levels[n, planet, building]`.

        Resources are pooled, so the empire builds what its scarcest limit allows: the sum over the planets of
        each limit, then the smallest of those sums.
        """
        return self.simulate(levels)["planet_limits"].sum(axis=1).min(axis=1)

    def in_range(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> np.ndarray:
        """True per `levels[n]` when every level is inside the range of its building."""
        planets = self.planet_index(planets)
        return ((levels >= self.min_levels[planets]) & (levels <= self.max_levels[planets])).all(axis=(1, 2))

    def is_feasible(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> np.ndarray:
        """True per `levels[n]` when every level is inside its range and every planet has the fields for its
        optimized buildings and the warehouses they need."""
        planets = self.planet_index(planets)
        inside = np.clip(levels, self.min_levels[planets], self.max_levels[planets])
        return self.in_range(levels, planets) & self.simulate(inside, planets)["planet_fits"].all(axis=1)


PRODUCTION_MODEL = EmpireProductionModel(
    PLANET_PLANS,
    [
        planet_param_context(
            PLAN_ACCOUNT_CONTEXT,
            plan["slot_index"],
            empire_planet_states[plan["slot_index"] - 1],
            is_empty_slot=plan["planet_status"] == "empty_colony_slot",
        )
        for plan in PLANET_PLANS
    ],
    optimized_keys=tuple(OPTIMIZED_BUILDING_MAX_LEVELS),
    target_ship_key=TARGET_SHIP_KEY,
    month_days=PRODUCTION_MONTH_DAYS,
    warehouse_hours=WAREHOUSE_HOURS_OF_PRODUCTION,
)

print_section_data(
    "Section 11.3 data: production model",
    [
        ("PRODUCTION_FLOW_KEYS", PRODUCTION_FLOW_KEYS),
        (
            "PRODUCTION_MODEL (EmpireProductionModel summary)",
            {
                "planets": len(PRODUCTION_MODEL.free_fields),
                "optimized_buildings": list(PRODUCTION_MODEL.optimized_keys),
                "limits": list(PRODUCTION_MODEL.limit_names),
                "ship_cost": dict(zip(PRODUCTION_MODEL.cost_resources, PRODUCTION_MODEL.ship_cost.tolist())),
                "ship_building_requirements": PRODUCTION_MODEL.target_requirements,
                "hours_per_month": PRODUCTION_MODEL.hours_per_month,
                "free_fields": PRODUCTION_MODEL.free_fields,
                "first_planet_with_the_same_inputs": PRODUCTION_MODEL.twin_planets,
                "warehouse_hours": PRODUCTION_MODEL.warehouse_hours,
                "central_planet": PRODUCTION_MODEL.planet_names[PRODUCTION_MODEL.central_planet],
            },
        ),
    ],
)

Section 11.3 data: production model
Printed: 2026-11-17 09:59:32 -0300 | 2 item(s)
[PRODUCTION_FLOW_KEYS] tuple, len=8
["pig_iron", "metal", "kryptonite", "spice", "pig_iron_used", "spice_used", "energy_produced", "energy_used"]
----------------------------------------------------------------------------------------------------
[PRODUCTION_MODEL (EmpireProductionModel summary)] dict, len=8
{
  "planets": 11,
  "optimized_buildings": ["b1", "b2", "b3", "b4", "b5", "b9", "b17"],
  "limits": ["metal", "kryptonite", "spice", "shipyards"],
  "ship_cost": {"metal": 3000000.0, "kryptonite": 2000000.0, "spice": 500000.0},
  "ship_building_requirements": {"b9": 13, "b7": 3},
  "hours_per_month": 720.0,
  "free_fields": [176, 259, 259, 259, 259, 259, 259, 259, 259, 259, 259],
  "first_planet_with_the_same_inputs": [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
}


### 11.4 Design Catalogues, Genetic Algorithm and Polish

The search (`search_best_configuration`) has three steps.

**1. A catalogue of designs per planet** (`build_design_catalogues`). No single design is best for a planet: one planet may do most for the empire by making only Metal, another by making Kryptonite and Spice. The catalogue therefore holds one design for each mix of priorities between the limits, such as "only Metal", "Metal and Kryptonite in equal parts" or "a quarter each". With `CATALOGUE_PRIORITY_STEPS = 4` and four limits there are 35 mixes. For each mix, `planet_design_catalogue` takes the best of about `CATALOGUE_GRID_SIZE` coarse designs (`coarse_designs`) and improves it with a hill-climb (`improve_design`), which changes one, two or three buildings at a time by the `HILL_CLIMB_STEP_SIZES` and prefers fewer levels between equals. One more design balances all limits, for a planet that has to do everything itself. Only designs that fit the planet together with their warehouses are considered. Twin planets share a catalogue.

A shipyard counts only up to the ships that the empire could pay for if every planet made nothing but its scarcest resource (`resource_peaks`). A design that gives the shipyard any priority therefore gets the smallest shipyard that is large enough, and keeps the rest of its fields for mines.

**2. The genetic algorithm** (`run_genetic_algorithm`) decides which design each planet takes.

- A chromosome is one catalogue number per planet.
- Its fitness is the number of target ships per month of that configuration: the limits of the chosen designs, added up over the planets, and then the smallest sum.
- Each generation keeps the `GA_ELITE_COUNT` best chromosomes unchanged. Parents are picked by tournament: the best of `GA_TOURNAMENT_SIZE` random chromosomes. A child takes each planet's design from either parent (`GA_CROSSOVER_RATE` of the children; the others copy one parent), and then each of its planets switches to a random design of its catalogue with probability `GA_MUTATION_RATE`.
- `GA_RANDOM_SEED` makes the run repeatable.

**3. The polish** (`polish_both_ways`) hill-climbs the winner level by level, one planet at a time with the others held fixed, until a full round changes nothing (`polish_configuration`). It does this twice, once counting ships directly and once following `balanced_score` first, and keeps the better result. The polish can only add ships, and it removes levels that add none.

**Rounds for the central planet** (`search_in_rounds`). The central planet stores `WAREHOUSE_HOURS_OF_PRODUCTION` hours of the production of the whole empire, and that production is only known once a search has finished. The first round therefore reserves nothing for it. When the central planet of the result does not hold what the result produces (`central_reserve_needed`), the search runs again on a copy of the model that reserves that much on the central planet. The rounds end with the first result that the central planet can hold. `BEST_CONFIG_MODEL` is the model in which the central planet reserves exactly what that result produces; Sections 11.5 and 11.6 describe the result with it.

The result is a good configuration, not a proven optimum: a genetic algorithm gives no such guarantee. `ga_history_df` shows whether the run had settled; another `GA_RANDOM_SEED` or more `GA_GENERATIONS` are a cheap second opinion.

Data of this cell, all from the last round:

- `central_reserve_rounds_df` — one row per round: the ships per month, the capacity reserved on the central planet going in, the capacity that the result's production needs, and the capacity that the central planet of the result holds;
- `BEST_CONFIG_MODEL` — the production model whose central planet keeps room for the production of the result;
- `DESIGN_CATALOGUES` — per planet, the levels of every catalogue design; `design_catalogue_df` — the same designs with their warehouse levels, the fields they take, and what each adds to every limit, in ships per month;
- `ga_designs_arr` — the catalogue number that the algorithm chose for each planet; `ga_levels_arr` — the levels of that choice, per planet and optimized building;
- `ga_history_df` — best, mean and worst fitness of every generation;
- `best_levels_arr` — the levels after the polish: the configuration that Section 11.5 describes;
- `GA_SUMMARY` — the number of rounds, catalogue sizes, size of the search, and the ships per month before and after the polish.

In [93]:
def balanced_score(limits: np.ndarray, sharpness: float = 8.0) -> np.ndarray:
    """Smooth stand-in for the smallest limit of each row of `limits[n, limit]`.

    It is the power mean of the row with exponent `-sharpness`. It never falls below the smallest limit and
    stays close to it, but unlike the smallest limit it also grows when any other limit grows. A search that
    follows it can trade a little of one limit for a lot of another, where the plain minimum would call
    every such step a loss.
    """
    floor = np.maximum(limits.min(axis=1, keepdims=True), 1e-9)
    return floor[:, 0] * np.mean((np.maximum(limits, 1e-9) / floor) ** -sharpness, axis=1) ** (-1.0 / sharpness)


def smallest_limit(limits: np.ndarray) -> np.ndarray:
    """The smallest limit of each row of `limits[n, limit]`: the target ships per month that the limits allow."""
    return limits.min(axis=1)


def planet_moves(buildings: int, step_sizes: tuple[int, ...]) -> np.ndarray:
    """Changes of one design that the hill-climb tries, as rows to add to its levels.

    For every step size: one building up or down; two buildings at once, each up or down; and two buildings
    up by the step while a third goes down by twice the step, which keeps the number of fields.
    """
    moves: set[tuple[int, ...]] = set()
    for size in step_sizes:
        for first in range(buildings):
            for first_step in (size, -size):
                moves.add(tuple(first_step if index == first else 0 for index in range(buildings)))
                for second in range(first + 1, buildings):
                    for second_step in (size, -size):
                        moves.add(
                            tuple(
                                first_step if index == first else second_step if index == second else 0
                                for index in range(buildings)
                            )
                        )
            for second in range(first + 1, buildings):
                for third in range(buildings):
                    if third not in (first, second):
                        moves.add(
                            tuple(
                                size if index in (first, second) else -2 * size if index == third else 0
                                for index in range(buildings)
                            )
                        )
    return np.array(sorted(moves), dtype=np.int64)


def improve_design(
    model: EmpireProductionModel,
    planet: int,
    design: np.ndarray,
    score: Any,
    moves: np.ndarray,
) -> np.ndarray:
    """Hill-climb one planet's design: keep taking the best neighbour until no neighbour is better.

    `score` maps the planet's limits `[n, limit]` to one number per row. A neighbour is better when it scores
    higher, or the same with fewer building levels, so the result carries no level that does nothing.
    Only neighbours that fit the planet, together with the warehouses they need, are considered.
    """
    current = np.asarray(design, dtype=np.int64)
    current_score = score(model.simulate(current[None, None, :], [planet])["planet_limits"][:, 0])[0]
    current_rank = (float(current_score), -int(current.sum()))
    while True:
        candidates = current + moves
        candidates = candidates[model.in_range(candidates[:, None, :], [planet])]
        if len(candidates) == 0:
            return current
        outcome = model.simulate(candidates[:, None, :], [planet])
        fits = outcome["planet_fits"][:, 0]
        if not fits.any():
            return current
        candidates, scores = candidates[fits], score(outcome["planet_limits"][fits, 0])
        level_totals = candidates.sum(axis=1)
        best = int(np.lexsort((-level_totals, scores))[-1])
        best_rank = (float(scores[best]), -int(level_totals[best]))
        if not best_rank > current_rank:  # also true for a score that is not a number, so the loop always ends
            return current
        current, current_rank = candidates[best], best_rank


def coarse_designs(model: EmpireProductionModel, planet: int, grid_size: int) -> tuple[np.ndarray, np.ndarray]:
    """A coarse grid of at most about `grid_size` designs of one planet: those that fit it, and their limits."""
    lower, upper = model.min_levels[planet], model.max_levels[planet]
    values_per_building = max(2, int(grid_size ** (1.0 / len(lower))))
    axes = [
        np.unique(np.linspace(low, high, values_per_building).round().astype(np.int64))
        for low, high in zip(lower, upper)
    ]
    grid = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, len(lower))
    outcome = model.simulate(grid[:, None, :], [planet])
    fits = outcome["planet_fits"][:, 0]
    return grid[fits], outcome["planet_limits"][fits, 0]


def resource_peaks(
    model: EmpireProductionModel,
    planet: int,
    grid: np.ndarray,
    grid_limits: np.ndarray,
    moves: np.ndarray,
) -> np.ndarray:
    """The most that one planet can add to each resource limit when it makes nothing else.

    For every resource limit, the best design of the coarse grid is hill-climbed with that limit as its score.
    """
    peaks: list[float] = []
    for column in range(len(model.cost_resources)):

        def this_limit(limits: np.ndarray, column: int = column) -> np.ndarray:
            return limits[:, column]

        design = improve_design(model, planet, grid[int(grid_limits[:, column].argmax())], this_limit, moves)
        peaks.append(float(model.simulate(design[None, None, :], [planet])["planet_limits"][0, 0, column]))
    return np.array(peaks)


def planet_design_catalogue(
    model: EmpireProductionModel,
    planet: int,
    grid: np.ndarray,
    grid_limits: np.ndarray,
    moves: np.ndarray,
    *,
    priority_steps: int,
    shipyard_ceiling: float,
) -> np.ndarray:
    """Candidate designs of one planet: for each mix of priorities between the limits, the design that serves it best.

    For every priority mix (each way to split `priority_steps` equal shares among the limits) the best design
    of the coarse grid is hill-climbed to a local optimum. One more design balances all limits, for a planet
    that has to do everything itself.

    A shipyard counts up to `shipyard_ceiling` ships and no further. Without that ceiling the shipyard limit,
    which doubles with each Microsystem accelerator level, would swallow the fields of every design that gives
    it any priority; with it, such a design gets the smallest shipyard that is large enough.
    """
    ceiling = np.full(grid_limits.shape[1], np.inf)
    ceiling[-1] = shipyard_ceiling
    # Each limit is measured against the best that a grid design reaches, so that the priorities compare like with like.
    scale = np.maximum(np.minimum(grid_limits, ceiling).max(axis=0), 1e-12)

    shares = np.stack(np.meshgrid(*[np.arange(priority_steps + 1)] * len(scale), indexing="ij"), axis=-1)
    shares = shares.reshape(-1, len(scale))
    designs: list[np.ndarray] = []
    for priorities in shares[shares.sum(axis=1) == priority_steps] / float(priority_steps):

        def priority_score(limits: np.ndarray, priorities: np.ndarray = priorities) -> np.ndarray:
            return (np.minimum(limits, ceiling) / scale) @ priorities

        start = grid[int(priority_score(grid_limits).argmax())]
        designs.append(improve_design(model, planet, start, priority_score, moves))
    start = grid[int(balanced_score(grid_limits).argmax())]
    designs.append(improve_design(model, planet, start, balanced_score, moves))
    return np.unique(np.array(designs), axis=0)


def build_design_catalogues(
    model: EmpireProductionModel,
    *,
    grid_size: int,
    priority_steps: int,
    step_sizes: tuple[int, ...],
) -> list[np.ndarray]:
    """One design catalogue per planet; a planet with a twin takes the twin's catalogue."""
    moves = planet_moves(len(model.optimized_keys), step_sizes)
    grids = {
        planet: coarse_designs(model, planet, grid_size)
        for planet, twin in enumerate(model.twin_planets)
        if twin == planet
    }
    # No empire builds more ships than its scarcest resource pays for when every planet makes nothing else:
    # that is the most a shipyard can be asked to finish.
    peaks = {planet: resource_peaks(model, planet, *grid, moves) for planet, grid in grids.items()}
    shipyard_ceiling = float(np.sum([peaks[twin] for twin in model.twin_planets], axis=0).min())
    catalogues = {
        planet: planet_design_catalogue(
            model, planet, *grid, moves, priority_steps=priority_steps, shipyard_ceiling=shipyard_ceiling
        )
        for planet, grid in grids.items()
    }
    return [catalogues[twin] for twin in model.twin_planets]


def catalogue_limits(model: EmpireProductionModel, catalogues: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """`design_limits[planet, design, limit]` of every catalogue design, and the number of designs per planet."""
    design_counts = np.array([len(catalogue) for catalogue in catalogues], dtype=np.int64)
    design_limits = np.zeros((len(catalogues), int(design_counts.max()), len(model.limit_names)), dtype=np.float64)
    for planet, catalogue in enumerate(catalogues):
        limits = model.simulate(catalogue[:, None, :], [planet])["planet_limits"][:, 0]
        design_limits[planet, : len(catalogue)] = limits
    return design_limits, design_counts


def design_catalogue_table(
    model: EmpireProductionModel,
    planet_plans: list[dict[str, Any]],
    catalogues: list[np.ndarray],
) -> pd.DataFrame:
    """One row per planet and catalogue design: its building levels, the fields that it takes together with its
    warehouses, and what it adds to each limit."""
    rows: list[dict[str, Any]] = []
    for planet, (plan, catalogue) in enumerate(zip(planet_plans, catalogues)):
        outcome = model.simulate(catalogue[:, None, :], [planet])
        for design, levels in enumerate(catalogue):
            rows.append(
                {
                    "slot_index": plan["slot_index"],
                    "coords": plan["coords"],
                    "design": design,
                    **dict(zip(model.optimized_keys, (int(level) for level in levels))),
                    "warehouse_levels": int(outcome["planet_warehouse_levels"][design, 0].sum()),
                    "fields": int(outcome["planet_fields_needed"][design, 0]),
                    **{
                        f"{limit}_ships_per_month": round(float(ships), 2)
                        for limit, ships in zip(model.limit_names, outcome["planet_limits"][design, 0])
                    },
                }
            )
    return pd.DataFrame(rows)


def run_genetic_algorithm(
    design_limits: np.ndarray,
    design_counts: np.ndarray,
    *,
    population_size: int,
    generations: int,
    tournament_size: int,
    crossover_rate: float,
    mutation_rate: float,
    elite_count: int,
    seed: int,
) -> tuple[np.ndarray, pd.DataFrame]:
    """Evolve the choice of one catalogue design per planet; return the best choice and the history.

    A chromosome holds one design number per planet. `design_limits[planet, design, limit]` is what that
    design adds to each limit, so the fitness of a chromosome, its target ships per month, is the smallest
    of the limit sums over its planets. Every generation keeps the `elite_count` best chromosomes, picks
    parents by tournament, takes each planet's design of a child from one of its two parents, and switches
    random planets of the children to a random design of their catalogue.
    """
    rng = np.random.default_rng(seed)
    planets = np.arange(len(design_counts))
    population = rng.integers(0, design_counts, size=(population_size, len(design_counts)))
    history: list[dict[str, Any]] = []
    for generation in range(generations + 1):
        fitness = design_limits[planets, population].sum(axis=1).min(axis=1)
        ranking = np.argsort(-fitness, kind="stable")
        history.append(
            {
                "generation": generation,
                "best_ships_per_month": float(fitness[ranking[0]]),
                "mean_ships_per_month": float(fitness.mean()),
                "worst_ships_per_month": float(fitness[ranking[-1]]),
            }
        )
        if generation == generations:
            break
        # Tournament selection: each parent is the best of `tournament_size` random chromosomes.
        contenders = rng.integers(0, population_size, size=(2 * population_size, tournament_size))
        winners = contenders[np.arange(2 * population_size), fitness[contenders].argmax(axis=1)]
        mothers, fathers = population[winners[:population_size]], population[winners[population_size:]]
        # Uniform crossover: a child takes each planet's design from its mother or its father.
        from_father = (rng.random(mothers.shape) < 0.5) & (rng.random((population_size, 1)) < crossover_rate)
        children = np.where(from_father, fathers, mothers)
        # Mutation: a planet switches to a random design of its catalogue.
        mutated = rng.random(children.shape) < mutation_rate
        children = np.where(mutated, rng.integers(0, design_counts, size=children.shape), children)
        population = np.concatenate([population[ranking[:elite_count]], children[: population_size - elite_count]])
    return population[ranking[0]].copy(), pd.DataFrame(history)


def polish_configuration(
    model: EmpireProductionModel,
    levels: np.ndarray,
    score: Any,
    step_sizes: tuple[int, ...],
) -> np.ndarray:
    """Hill-climb a whole configuration, one planet at a time, until a full round changes nothing.

    `score` maps the empire's limits `[n, limit]` to one number per row. Each planet's design is improved
    with `improve_design` while the other planets' contributions stay fixed.
    """
    moves = planet_moves(levels.shape[1], step_sizes)
    levels = np.array(levels, dtype=np.int64)
    planet_limits = model.simulate(levels[None])["planet_limits"][0]
    changed = True
    while changed:
        changed = False
        for planet in range(len(levels)):
            others = planet_limits.sum(axis=0) - planet_limits[planet]

            def empire_score(own: np.ndarray, others: np.ndarray = others) -> np.ndarray:
                return score(others + own)

            design = improve_design(model, planet, levels[planet], empire_score, moves)
            if (design != levels[planet]).any():
                levels[planet] = design
                planet_limits[planet] = model.simulate(design[None, None, :], [planet])["planet_limits"][0, 0]
                changed = True
    return levels


def polish_both_ways(model: EmpireProductionModel, levels: np.ndarray, step_sizes: tuple[int, ...]) -> np.ndarray:
    """Polish a configuration twice and keep the result with more target ships (fewer levels between equals).

    Once the polish counts ships directly. Once it first follows `balanced_score`, which can rearrange the
    planets more freely, and only then counts ships.
    """
    direct = polish_configuration(model, levels, smallest_limit, step_sizes)
    rebalanced = polish_configuration(
        model, polish_configuration(model, levels, balanced_score, step_sizes), smallest_limit, step_sizes
    )
    ships = model.ships_per_month(np.array([direct, rebalanced]))
    return rebalanced if (ships[1], -rebalanced.sum()) > (ships[0], -direct.sum()) else direct


def search_best_configuration(
    model: EmpireProductionModel,
    *,
    grid_size: int,
    priority_steps: int,
    step_sizes: tuple[int, ...],
    **ga_settings: Any,
) -> tuple[list[np.ndarray], np.ndarray, pd.DataFrame, np.ndarray, np.ndarray]:
    """One search on `model`: design catalogues, genetic algorithm, polish.

    Returns the catalogues, the catalogue design that the genetic algorithm chose for each planet, the history
    of its generations, the levels of that choice, and the levels after the polish. `ga_settings` are the
    keyword arguments of `run_genetic_algorithm`.
    """
    catalogues = build_design_catalogues(
        model, grid_size=grid_size, priority_steps=priority_steps, step_sizes=step_sizes
    )
    ga_designs, ga_history = run_genetic_algorithm(*catalogue_limits(model, catalogues), **ga_settings)
    ga_levels = np.array([catalogues[planet][design] for planet, design in enumerate(ga_designs)], dtype=np.int64)
    return catalogues, ga_designs, ga_history, ga_levels, polish_both_ways(model, ga_levels, step_sizes)


def central_reserve_needed(model: EmpireProductionModel, levels: np.ndarray) -> np.ndarray:
    """Capacity per resource that holds `warehouse_hours` of the net production of the whole empire."""
    net_per_hour = model.simulate(levels[None])["planet_net_per_hour"][0].sum(axis=0)
    return model.warehouse_hours * np.maximum(net_per_hour, 0.0)


def search_in_rounds(model: EmpireProductionModel, **search_settings: Any) -> tuple[Any, ...]:
    """Repeat the search until the warehouses of the central planet hold the production of the result.

    The central planet stores `warehouse_hours` of the production of the whole empire, and that production is
    only known once a search has finished. The first round therefore reserves nothing for it. Every further
    round reserves, on the central planet, room for the largest production that the rounds before came to.
    The rounds end with the first result whose central planet holds what the result itself produces; the
    reserve only grows from round to round, so they do end.

    Returns a copy of the model whose central planet reserves exactly what the last result produces, a table
    with one row per round, and the result of the last round as `search_best_configuration` returns it.
    `search_settings` are that function's keyword arguments.
    """
    reserve = np.zeros(len(RESOURCE_KEYS), dtype=np.float64)
    rounds: list[dict[str, Any]] = []
    while True:
        round_model = model.with_central_reserve(reserve)
        result = search_best_configuration(round_model, **search_settings)
        best_levels = result[-1]
        needed = central_reserve_needed(round_model, best_levels)
        warehouses = round_model.simulate(best_levels[None])["planet_warehouse_levels"][0]
        # A model without a central planet has nothing to hold.
        held = (
            round_model.warehouse_capacities[warehouses[round_model.central_planet]]
            if round_model.central_planet >= 0
            else np.full(len(RESOURCE_KEYS), np.inf)
        )
        holds_the_production = not (needed > held).any()
        rounds.append(
            {
                "round": len(rounds) + 1,
                "ships_per_month": round(float(round_model.ships_per_month(best_levels[None])[0]), 2),
                **{f"{resource}_reserved": float(amount) for resource, amount in zip(RESOURCE_KEYS, reserve)},
                **{f"{resource}_needed": round(float(amount), 1) for resource, amount in zip(RESOURCE_KEYS, needed)},
                **{f"{resource}_held": float(amount) for resource, amount in zip(RESOURCE_KEYS, held)},
                "holds_the_production": holds_the_production,
            }
        )
        if holds_the_production:
            return model.with_central_reserve(needed), pd.DataFrame(rounds), *result
        reserve = np.maximum(reserve, needed)


(
    BEST_CONFIG_MODEL,
    central_reserve_rounds_df,
    DESIGN_CATALOGUES,
    ga_designs_arr,
    ga_history_df,
    ga_levels_arr,
    best_levels_arr,
) = search_in_rounds(
    PRODUCTION_MODEL,
    grid_size=CATALOGUE_GRID_SIZE,
    priority_steps=CATALOGUE_PRIORITY_STEPS,
    step_sizes=HILL_CLIMB_STEP_SIZES,
    population_size=GA_POPULATION_SIZE,
    generations=GA_GENERATIONS,
    tournament_size=GA_TOURNAMENT_SIZE,
    crossover_rate=GA_CROSSOVER_RATE,
    mutation_rate=GA_MUTATION_RATE,
    elite_count=GA_ELITE_COUNT,
    seed=GA_RANDOM_SEED,
)
design_catalogue_df = design_catalogue_table(BEST_CONFIG_MODEL, PLANET_PLANS, DESIGN_CATALOGUES)

GA_SUMMARY: dict[str, Any] = {
    "search_rounds": int(len(central_reserve_rounds_df)),
    "designs_per_planet": [len(catalogue) for catalogue in DESIGN_CATALOGUES],
    "possible_choices_log10": round(float(sum(math.log10(len(catalogue)) for catalogue in DESIGN_CATALOGUES)), 2),
    "chromosomes_evaluated": GA_POPULATION_SIZE * (GA_GENERATIONS + 1) * int(len(central_reserve_rounds_df)),
    "first_generation_with_the_final_best": int(
        (ga_history_df["best_ships_per_month"] >= ga_history_df["best_ships_per_month"].iloc[-1]).idxmax()
    ),
    "ga_ships_per_month": round(float(BEST_CONFIG_MODEL.ships_per_month(ga_levels_arr[None])[0]), 2),
    "polished_ships_per_month": round(float(BEST_CONFIG_MODEL.ships_per_month(best_levels_arr[None])[0]), 2),
    "levels_changed_by_polish": int(np.abs(best_levels_arr - ga_levels_arr).sum()),
}

print_section_data(
    "Section 11.4 data: design catalogues, genetic algorithm and polish",
    [
        ("GA_SUMMARY", GA_SUMMARY),
        ("central_reserve_rounds_df", central_reserve_rounds_df),
        (
            "BEST_CONFIG_MODEL (capacity that the central planet keeps for the production of the empire)",
            dict(zip(RESOURCE_KEYS, BEST_CONFIG_MODEL.central_reserve.tolist())),
        ),
        ("ga_designs_arr", ga_designs_arr),
        ("ga_levels_arr", ga_levels_arr),
        ("best_levels_arr", best_levels_arr),
        ("ga_history_df", ga_history_df),
        ("design_catalogue_df", design_catalogue_df),
        ("DESIGN_CATALOGUES", DESIGN_CATALOGUES),
    ],
)

Section 11.4 data: design catalogues, genetic algorithm and polish
Printed: 2026-11-17 09:59:33 -0300 | 7 item(s)
[GA_SUMMARY] dict, len=7
{
  "designs_per_planet": [18, 18, 18, 18, 18, 18, 18, 18, 18, 18, 18],
  "possible_choices_log10": 13.81,
  "chromosomes_evaluated": 120400,
  "first_generation_with_the_final_best": 4,
  "ga_ships_per_month": 30038.98,
  "polished_ships_per_month": 30038.98,
  "levels_changed_by_polish": 430
}
----------------------------------------------------------------------------------------------------
[ga_designs_arr] numpy.ndarray, shape=(11,), dtype=int64
[15 12 16 11 15 12 17 17 14 13 16]
----------------------------------------------------------------------------------------------------
[ga_levels_arr] numpy.ndarray, shape=(11, 7), dtype=int64
[[47 50  0  0 44 13  9]
 [47 50  0 50 50  6  0]
 [47 50 50  0 48 13  9]
 [47 50  0  0 44 13  9]
 [47 50 50  0 48  6  0]
 [47 50  0 50 50  6  0]
 [47 50 50 45 50  6  0]
 [47 50 50 45 50  6  0]
 [47 50 49 45 50 13 

### 11.5 Best Configuration

The configuration that the search found, described by `build_best_configuration_tables` and `summarize_best_configuration`.

- `BEST_CONFIG_SUMMARY` — the result in numbers: target ships per month (`ships_per_month`, and the whole ships in `whole_ships_per_month`), the limit that holds the others back (`bottleneck`), the central planet, fields used and free, the fields that the warehouses take, the Pig iron that nothing uses, and the investment.
- `best_config_limits_df` — one row per limit: what one ship needs, what the empire has per month, the ships per month that it allows, and the share of it that the result uses. The bottleneck is used to 100 %.
- `best_config_buildings_df` — one row per planet: fields, and the level of every building: optimized, fixed or warehouse.
- `best_config_planets_df` — one row per planet: energy produced and used, `power_ratio` and `smelting_ratio` (1 means that nothing is held back), net output per hour of each resource, the ships that its shipyard could finish and the ships it builds, the fields of its warehouses, and the cost of getting there.
- `best_config_warehouses_df` — one row per planet and resource, from `warehouse_requirement_rows`: the level of the warehouse, its capacity, and the level that each rule of Section 11 asks for: the largest cost (`level_for_largest_cost`, with the level or ship that costs it in `largest_cost_of`), the hours of the planet's own production, the hours of the empire's production on the central planet, and the minimum level. `set_by` names the rule that asks for the most.
- `BEST_CONFIG_TABLES` — registry of the four tables.

**Investment.** The search maximizes monthly output and ignores what the buildings cost, so the result is limited only by the fields, the energy, the warehouses and the maximum levels of Section 11.1. The summary puts a price on it, in Spice-equivalent at Merchant parity:

- `buildings_cost_spice_eq` — every building level that is still missing on your colonies and on the generic planets (`levels_to_build`), warehouses included;
- `fleet_cost_spice_eq` — the ships missing from the minimum fleets;
- `research_cost_spice_eq` — the research levels between your current ones and the planned ones;
- `investment_in_months_of_output` — the sum of the three, divided by the value of the ships that the finished configuration builds in a month.

When that last number is large, lower `OPTIMIZED_BUILDING_MAX_LEVELS` to see the best configuration within reach.

In [94]:
def building_upgrade_cost_spice_eq(key: str, from_level: int, to_level: int) -> float:
    """Spice-equivalent cost of raising building `key` from `from_level` to `to_level`; 0 when it is not higher."""
    return float(
        sum(
            game_data.spice_equivalent(game_data.building_cost(int(key[1:]), level))
            for level in range(from_level + 1, to_level + 1)
        )
    )


def research_upgrade_cost_spice_eq(key: str, from_level: int, to_level: int) -> float:
    """Spice-equivalent cost of raising research `key` from `from_level` to `to_level`; 0 when it is not higher."""
    levels = game_data.RESEARCH_BY_ID[key].get("levels") or {}
    return float(
        sum(
            game_data.spice_equivalent(full_cost((levels.get(level) or {}).get("cost")))
            for level in range(from_level + 1, to_level + 1)
        )
    )


def warehouse_requirement_rows(
    plan: dict[str, Any],
    planned_levels: dict[str, int],
    net_per_hour: np.ndarray,
    *,
    warehouse_hours: float,
    ship_built: str | None,
    central_reserve: np.ndarray,
) -> list[dict[str, Any]]:
    """Why each warehouse of a planet has its level: one row per resource, with the level that every rule asks for.

    `planned_levels` holds the planet's level of every building and `net_per_hour` its net output of each
    resource. `ship_built` is the key of the target ship when the planet's shipyard builds it, and
    `central_reserve` the capacity per resource that the planet keeps for the empire: zeros, unless it is the
    central planet.
    """
    largest_cost, largest_cost_of = largest_costs(
        [
            *(
                (plan["capacity_needed_for"][resource], {resource: plan["capacity_needed"][resource]})
                for resource in RESOURCE_KEYS
            ),
            *(
                cost
                for key in (*plan["min_levels"], *WAREHOUSE_KEYS)
                for cost in building_upgrade_costs(key, plan["built_levels"][key], planned_levels[key])
            ),
            *(
                (ship, game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost") or {})
                for ship in ([ship_built] if ship_built else [])
            ),
        ]
    )
    rows: list[dict[str, Any]] = []
    for index, (resource, key) in enumerate(zip(RESOURCE_KEYS, WAREHOUSE_KEYS)):
        # The level that each rule asks for, with its reason; between equal levels the first reason is named.
        asked = [
            (warehouse_level_for(largest_cost[resource]), f"cost of {largest_cost_of[resource]}"),
            (warehouse_level_for(warehouse_hours * max(0.0, float(net_per_hour[index]))), "own production"),
            (warehouse_level_for(float(central_reserve[index])), "empire production"),
            (plan["warehouse_min_levels"][key], plan["warehouse_min_set_by"][key]),
        ]
        level, set_by = max(asked, key=lambda rule: rule[0])
        rows.append(
            {
                "slot_index": plan["slot_index"],
                "coords": plan["coords"],
                "resource": resource,
                "key": key,
                "built_level": plan["built_levels"][key],
                "level": planned_levels[key],
                "capacity": float(warehouse_capacity(planned_levels[key])),
                "largest_cost": largest_cost[resource],
                "largest_cost_of": largest_cost_of[resource],
                "level_for_largest_cost": asked[0][0],
                "level_for_own_production": asked[1][0],
                "level_for_empire_production": asked[2][0],
                "minimum_level": asked[3][0],
                "set_by": set_by if level > 0 else "nothing to store",
            }
        )
    return rows


def build_best_configuration_tables(
    model: EmpireProductionModel,
    planet_plans: list[dict[str, Any]],
    levels: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Describe one configuration: building levels per planet, each planet's output, the empire's limits, and
    the reason for the level of every warehouse."""
    outcome = {name: values[0] for name, values in model.simulate(levels[None]).items()}
    limit_totals = outcome["planet_limits"].sum(axis=0)
    ships = float(limit_totals.min())
    shipyard_column = model.limit_names.index("shipyards")
    building_rows: list[dict[str, Any]] = []
    planet_rows: list[dict[str, Any]] = []
    warehouse_rows: list[dict[str, Any]] = []
    for index, plan in enumerate(planet_plans):
        identity = {column: plan[column] for column in ("slot_index", "planet_id", "coords", "planet_status")}
        warehouses = dict(zip(WAREHOUSE_KEYS, (int(level) for level in outcome["planet_warehouse_levels"][index])))
        planned = {
            **plan["fixed_levels"],
            **warehouses,
            **dict(zip(model.optimized_keys, (int(level) for level in levels[index]))),
        }
        fields_used = sum(planned.values())
        building_rows.append(
            {
                **identity,
                "fields_max": plan["fields_max"],
                "fields_used": fields_used,
                "fields_free": plan["fields_max"] - fields_used,
                **{f"b{building_id}": planned[f"b{building_id}"] for building_id in sorted(game_data.BUILDING_BY_ID)},
            }
        )
        shipyard_ships = float(outcome["planet_limits"][index, shipyard_column])
        shipyard_total = float(limit_totals[shipyard_column])
        planet_rows.append(
            {
                **identity,
                "energy_produced": round(float(outcome["planet_energy_produced"][index]), 1),
                "energy_used": round(float(outcome["planet_energy_used"][index]), 1),
                "power_ratio": round(float(outcome["planet_power_ratio"][index]), 4),
                "smelting_ratio": round(float(outcome["planet_smelting_ratio"][index]), 4),
                **{
                    f"{resource}_per_hour": round(float(outcome["planet_net_per_hour"][index, column]), 1)
                    for column, resource in enumerate(RESOURCE_KEYS)
                },
                "shipyard_ships_per_month": round(shipyard_ships, 2),
                # The shipyards share the ships that the resources pay for in proportion to their capacity.
                "ships_built_per_month": (
                    round(shipyard_ships * ships / shipyard_total, 2) if shipyard_total > 0.0 else 0.0
                ),
                "warehouse_fields": sum(warehouses.values()),
                "levels_to_build": sum(max(0, level - plan["built_levels"][key]) for key, level in planned.items()),
                "upgrade_cost_spice_eq": round(
                    sum(
                        building_upgrade_cost_spice_eq(key, plan["built_levels"][key], level)
                        for key, level in planned.items()
                    ),
                    2,
                ),
                "fleet_cost_spice_eq": round(plan["fleet_cost_spice_eq"], 2),
            }
        )
        warehouse_rows.extend(
            warehouse_requirement_rows(
                plan,
                planned,
                outcome["planet_net_per_hour"][index],
                warehouse_hours=model.warehouse_hours,
                ship_built=model.target_ship_key if shipyard_ships > 0.0 else None,
                central_reserve=(
                    model.central_reserve if index == model.central_planet else np.zeros(len(RESOURCE_KEYS))
                ),
            )
        )

    # Per ship: the units of a resource that one ship costs; a shipyard finishes whole ships, so it "costs" 1.
    limits = pd.DataFrame(
        {
            "limit": list(model.limit_names),
            "needed_per_ship": [*model.ship_cost.tolist(), 1.0],
            "ships_per_month": np.round(limit_totals, 2),
        }
    )
    limits["available_per_month"] = np.round(limit_totals * limits["needed_per_ship"].to_numpy(), 1)
    limits["used_percent"] = [round(100.0 * ships / total, 2) if total > 0.0 else 0.0 for total in limit_totals]
    limits["is_bottleneck"] = limit_totals == limit_totals.min()
    return pd.DataFrame(building_rows), pd.DataFrame(planet_rows), limits, pd.DataFrame(warehouse_rows)


def summarize_best_configuration(
    model: EmpireProductionModel,
    levels: np.ndarray,
    buildings: pd.DataFrame,
    planets: pd.DataFrame,
    limits: pd.DataFrame,
    own_research: dict[str, int],
    planned_research: dict[str, int],
) -> dict[str, Any]:
    """Headline numbers of a configuration: ships per month, the bottleneck, and what building it would cost."""
    ship = game_data.SHIPS[SHIP_SLUG_BY_KEY[model.target_ship_key]]
    ship_cost_spice_eq = float(game_data.spice_equivalent(full_cost(ship.get("base_cost"))))
    ships = float(model.ships_per_month(levels[None])[0])
    costs = {
        "buildings_cost_spice_eq": float(planets["upgrade_cost_spice_eq"].sum()),
        "fleet_cost_spice_eq": float(planets["fleet_cost_spice_eq"].sum()),
        "research_cost_spice_eq": float(
            sum(
                research_upgrade_cost_spice_eq(key, max(0, int(own_research.get(key, 0) or 0)), level)
                for key, level in planned_research.items()
            )
        ),
    }
    investment = sum(costs.values())
    monthly_output = ships * ship_cost_spice_eq
    return {
        "target_ship": str(ship["name"]),
        "ships_per_month": round(ships, 2),
        "whole_ships_per_month": int(ships),
        "bottleneck": ", ".join(limits.loc[limits["is_bottleneck"], "limit"]),
        "hours_per_month": model.hours_per_month,
        "planets": int(len(buildings)),
        "central_planet": model.planet_names[model.central_planet] if model.central_planet >= 0 else "",
        "fields_used": int(buildings["fields_used"].sum()),
        "fields_free": int(buildings["fields_free"].sum()),
        "warehouse_fields": int(planets["warehouse_fields"].sum()),
        "pig_iron_left_over_per_month": round(float(planets["pig_iron_per_hour"].sum()) * model.hours_per_month, 1),
        "levels_to_build": int(planets["levels_to_build"].sum()),
        **{name: round(cost, 2) for name, cost in costs.items()},
        "investment_spice_eq": round(investment, 2),
        "ship_cost_spice_eq": round(ship_cost_spice_eq, 2),
        "monthly_output_spice_eq": round(monthly_output, 2),
        "investment_in_months_of_output": round(investment / monthly_output, 2) if monthly_output > 0.0 else None,
    }


(
    best_config_buildings_df,
    best_config_planets_df,
    best_config_limits_df,
    best_config_warehouses_df,
) = build_best_configuration_tables(BEST_CONFIG_MODEL, PLANET_PLANS, best_levels_arr)
BEST_CONFIG_SUMMARY: dict[str, Any] = summarize_best_configuration(
    BEST_CONFIG_MODEL,
    best_levels_arr,
    best_config_buildings_df,
    best_config_planets_df,
    best_config_limits_df,
    colony_researches,
    PLANNED_RESEARCH_LEVELS,
)
BEST_CONFIG_TABLES: dict[str, pd.DataFrame] = {
    "best_config_limits": best_config_limits_df,
    "best_config_buildings": best_config_buildings_df,
    "best_config_planets": best_config_planets_df,
    "best_config_warehouses": best_config_warehouses_df,
}

print_section_data(
    f"Section 11.5 data: best configuration, {BEST_CONFIG_SUMMARY['ships_per_month']:,.2f} "
    f"{BEST_CONFIG_SUMMARY['target_ship']}s per month (bottleneck: {BEST_CONFIG_SUMMARY['bottleneck']})",
    [
        ("BEST_CONFIG_SUMMARY", BEST_CONFIG_SUMMARY),
        ("best_config_limits_df", best_config_limits_df),
        ("best_config_buildings_df", best_config_buildings_df),
        ("best_config_planets_df", best_config_planets_df),
        ("best_config_warehouses_df", best_config_warehouses_df),
        ("BEST_CONFIG_TABLES", BEST_CONFIG_TABLES),
    ],
)

Section 11.5 data: best configuration, 30,038.98 Imperial starbases per month (bottleneck: metal)
Printed: 2026-11-17 09:59:33 -0300 | 5 item(s)
[BEST_CONFIG_SUMMARY] dict, len=17
{
  "target_ship": "Imperial starbase",
  "ships_per_month": 30038.98,
  "whole_ships_per_month": 30038,
  "bottleneck": "metal",
  "hours_per_month": 720.0,
  "planets": 11,
  "fields_used": 2499,
  "fields_free": 731,
  "pig_iron_left_over_per_month": 23462208000.0,
  "levels_to_build": 2398,
  "buildings_cost_spice_eq": 1203069033228.1,
  "fleet_cost_spice_eq": 1294400.0,
  "research_cost_spice_eq": 1083513464053.33,
  "investment_spice_eq": 2286583791681.43,
  "ship_cost_spice_eq": 2366666.67,
  "monthly_output_spice_eq": 71092243200.0,
  "investment_in_months_of_output": 32.16
}
----------------------------------------------------------------------------------------------------
[best_config_limits_df] pandas.DataFrame, shape=(4, 6)
     limit needed_per_ship ships_per_month available_per_month used_perce

### 11.6 Configuration of Each Planet

What to build on each planet to reach the result of Section 11.5. `PLANET_CONFIGURATIONS` holds one table per planet, built by `planet_configuration_table` and printed one after the other. The label of a table names the roster slot, the coordinates, the status of the planet and the fields that the configuration uses. A table lists every building that the planet has today or gets in the configuration:

- `built_level` — the level that stands today (0 everywhere on a generic planet);
- `target_level` — the level in the best configuration;
- `levels_to_build`, `upgrade_cost_spice_eq` — the levels still missing, and their cost in Spice-equivalent at Merchant parity;
- `role` — `optimized` when the genetic algorithm chose the level, `fixed` when the meta parameters set it, and `warehouse` when the model set it to what the planet needs (`best_config_warehouses_df` of Section 11.5 gives the reason).

In [95]:
def planet_configuration_table(plan: dict[str, Any], planned_levels: dict[str, int]) -> pd.DataFrame:
    """One planet's configuration: every building it has or gets, the level to reach, and what is still missing."""
    rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        key = f"b{building_id}"
        built_level, target_level = plan["built_levels"][key], planned_levels[key]
        if built_level == 0 and target_level == 0:
            continue
        if key in plan["min_levels"]:
            role = "optimized"
        else:
            role = "warehouse" if key in WAREHOUSE_KEYS else "fixed"
        rows.append(
            {
                "key": key,
                "building": str(game_data.BUILDING_BY_ID[building_id]["name"]),
                "role": role,
                "built_level": built_level,
                "target_level": target_level,
                "levels_to_build": max(0, target_level - built_level),
                "upgrade_cost_spice_eq": round(building_upgrade_cost_spice_eq(key, built_level, target_level), 2),
            }
        )
    return pd.DataFrame(rows)


# One table per planet, in roster order. The label says which planet it is and how many fields it uses.
PLANET_CONFIGURATIONS: dict[str, pd.DataFrame] = {
    f"slot {plan['slot_index']}, {plan['coords']} ({plan['planet_status']}, "
    f"{int(levels['fields_used'])} of {plan['fields_max']} fields)": planet_configuration_table(
        plan, {f"b{building_id}": int(levels[f"b{building_id}"]) for building_id in sorted(game_data.BUILDING_BY_ID)}
    )
    for plan, (_, levels) in zip(PLANET_PLANS, best_config_buildings_df.iterrows())
}

print_section_data(
    f"Section 11.6 data: configuration of each planet for {BEST_CONFIG_SUMMARY['ships_per_month']:,.2f} "
    f"{BEST_CONFIG_SUMMARY['target_ship']}s per month",
    [(f"PLANET_CONFIGURATIONS: {label}", table) for label, table in PLANET_CONFIGURATIONS.items()],
)

Section 11.6 data: configuration of each planet for 30,038.98 Imperial starbases per month
Printed: 2026-11-17 09:59:33 -0300 | 11 item(s)
[PLANET_CONFIGURATIONS: slot 1, 02:380:12 (colonized, 201 of 230 fields)] pandas.DataFrame, shape=(13, 7)
key             building      role  built_level  target_level  levels_to_build upgrade_cost_spice_eq
 b1        Pig iron mine optimized           21            47               26        10797199098.00
 b2        Blast furnace optimized           20            50               30        36304809587.33
 b3      Kryptonite mine optimized           21             0                0                  0.00
 b4           Spice mine optimized           15             0                0                  0.00
 b5         Fusion plant optimized           23            44               21         3673780313.83
 b7   Development center     fixed            9            12                3            1935360.00
 b8         Research lab     fixed            3 